# PaperDos: MS

In [ ]:
%matplotlib inline
import re
import sys
from pathlib import Path

import hyperspy.api as hs
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib import gridspec, patches, ticker
from mpl_toolkits.axes_grid1 import ImageGrid
from matplotlib.colorbar import Colorbar
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, to_hex, to_rgba
from matplotlib.lines import Line2D
from matplotlib.transforms import Bbox
from mpl_toolkits.mplot3d import Axes3D
from scipy import ndimage
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

In [ ]:
# # Ensure custom module Path is set before import
# sys.path.append(r"E:\Desktop\PaperDos\Plottings")
# from colors import main  # type: ignore

# main()

In [ ]:
# Ensure custom module Path is set before import
sys.path.append(r"E:\Desktop\PaperDos\Plottings")
from colors import tol_cmap, tol_cset  # type: ignore

# 画图的初始设置
plt.style.use(r"E:\Desktop\PaperDos\Plottings\liuchzzyy.mplstyle")
# print(plt.style.available)

# xarray setting
xr.set_options(
    cmap_sequential="viridis",
    cmap_divergent="viridis",
    display_width=150,
)  # viridis, gray

# 颜色设定
colors = tol_cset("vibrant")
if colors is not None:
    colors = list(colors)
    colors_opt = ["#b0a3d1", "#8bd0d5", "#a8e0ee", "#c5e1a3", "#ffe48b", "#f5a37d", "#e88db1"]
    colors_opt2 = list(tol_cset("bright"))
else:
    # Fallback colors in case tol_cset returns None
    colors = ["#0077BB", "#33BBEE", "#009988", "#EE7733", "#CC3311", "#EE3377", "#BBBBBB"]
if r"sunset" not in plt.colormaps():
    cmap = tol_cmap("sunset")
    if isinstance(cmap, LinearSegmentedColormap):
        plt.colormaps.register(cmap)
if r"rainbow_PuRd" not in plt.colormaps():
    cmap = tol_cmap("rainbow_PuRd")
    if isinstance(cmap, LinearSegmentedColormap):
        plt.colormaps.register(cmap)  # 备用 plasma

# Set math font
mpl.rcParams["mathtext.fontset"] = "custom"
mpl.rcParams["mathtext.rm"] = "Arial"
mpl.rcParams["mathtext.it"] = "Arial:italic"
mpl.rcParams["mathtext.bf"] = "Arial:bold"
mpl.rcParams["mathtext.sf"] = "Arial"
mpl.rcParams["mathtext.tt"] = "Arial"
mpl.rcParams["mathtext.cal"] = "Arial"
mpl.rcParams["mathtext.default"] = "regular"

# 数据/输出的文件夹
path_data_root = Path(r"D:\PaperDos.20260304\Data")
path_out = Path(r"E:\Desktop\Figures")
path_out.mkdir(parents=True, exist_ok=True)

## FigureMS 02 -- 原位 Zn-Mn 的演化过程 -V3

In [ ]:
# 读取数据

# Operando XRD, EMD, from No pH Buffer, 1 M + 0.2 M, 30 uA

# 1) 读取电化学数据
path_filelist = list(Path.joinpath(path_data_root, r"XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Echem\B").glob(r"**\*.txt"))
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = pd.to_datetime(df["time/s"], format="mixed", errors="coerce")
    df["cycle number"] = pd.to_numeric(df["cycle number"], errors="coerce").astype("Int64")
    df["Voltage/V"] = df["Ewe/V"] - df["Ece/V"]
    echem_all.append(df)

echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 2) 读取谱线时间戳（按当前固定格式：Wave / Date / filename）
path_xrd = Path.joinpath(path_data_root, r"XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Data")
path_time_spectrum = path_xrd.joinpath(r"Time_index_spectrum.xlsx")

spectrum_time_all = pd.read_excel(path_time_spectrum)
required_cols = {"Wave", "Date", "filename"}
missing_cols = required_cols.difference(set(spectrum_time_all.columns))

spectrum_time_all["spec_time"] = pd.to_datetime(
    spectrum_time_all["Date"],
    format=r"%Y-%m-%d_%H:%M:%S",
    errors="coerce",
)

spectrum_time_all["spectrum_col"] = spectrum_time_all["filename"].astype(str).str.strip().str.replace(r"\.xye$", "", regex=True)
spectrum_time_all = spectrum_time_all.dropna(subset=["spec_time", "spectrum_col"]).sort_values("spec_time").reset_index(drop=True)

# 3) 读取 XRD 全谱数据
path_dspacing = path_xrd.joinpath(r"spectrum_all_d_spacing.csv")

spectrum_all = pd.read_csv(path_dspacing, index_col=0, header=0)
spectrum_all.index = pd.to_numeric(spectrum_all.index, errors="coerce")
spectrum_all = spectrum_all.loc[~spectrum_all.index.isna(), :]
spectrum_all.columns = [str(c).strip().replace(".xye", "") for c in spectrum_all.columns]

In [ ]:
# 过滤 17_Da_* 谱线并与时间索引对齐（后续单元直接使用）

# 1) 时间表仅保留 17_Da_fxxxxx
spectrum_time_all["spectrum_col"] = spectrum_time_all["spectrum_col"].astype(str).str.strip().str.replace(r"\.xye$", "", regex=True)
mask_da = spectrum_time_all["spectrum_col"].str.match(r"^17_Da_f\d+$", na=False)
spectrum_time_all = spectrum_time_all.loc[mask_da].copy().sort_values("spec_time").reset_index(drop=True)

# 2) 谱图矩阵仅保留 17_Da_fxxxxx 列
spec_cols_da = [c for c in spectrum_all.columns if c.startswith("17_Da_f")]
spectrum_all = spectrum_all.loc[:, spec_cols_da].copy()

# 3) 取交集并按时间顺序重排，确保 time/spectrum 完全对齐
common_cols = [c for c in spectrum_time_all["spectrum_col"].tolist() if c in spectrum_all.columns]
spectrum_time_all = spectrum_time_all[spectrum_time_all["spectrum_col"].isin(common_cols)].copy()
spectrum_time_all = spectrum_time_all.drop_duplicates(subset=["spectrum_col"], keep="first").sort_values("spec_time").reset_index(drop=True)
spectrum_all = spectrum_all.reindex(columns=spectrum_time_all["spectrum_col"].tolist())

# 4) 清洗电化学数据并建立 charge_time
echem_all.drop(echem_all.index[:6500], inplace=True)
selected_echem = echem_all[echem_all["cycle number"].isin([1, 2])]
selected_echem = selected_echem[selected_echem["Voltage/V"] >= 0.8].copy()
selected_echem = selected_echem.dropna(subset=["time/s", "Voltage/V", "<I>/mA"]).sort_values(by="time/s").reset_index(drop=True)
selected_echem["charge_time"] = (selected_echem["time/s"] - selected_echem["time/s"].iloc[0]).dt.total_seconds() / 3600

# 5) 最近邻时间匹配：谱线 -> 电化学
selected_spectrum_time = (
    pd
    .merge_asof(
        spectrum_time_all[["spectrum_col", "spec_time"]].sort_values(by="spec_time"),
        selected_echem[["time/s", "Voltage/V", "<I>/mA", "charge_time"]].sort_values(by="time/s"),
        left_on="spec_time",
        right_on="time/s",
        direction="nearest",
        tolerance=pd.Timedelta("5s"),
    )
    .dropna(subset=["spectrum_col", "spec_time", "time/s", "Voltage/V", "charge_time"], inplace=False)
    .sort_values(by="spec_time")
    .reset_index(drop=True)
)

# 6) 衍生时间与索引列（新版）
time_zero = selected_echem["time/s"].iloc[0]
selected_spectrum_time["spec_charge_time"] = (selected_spectrum_time["spec_time"] - time_zero).dt.total_seconds() / 3600
selected_spectrum_time["spec_time_h"] = selected_spectrum_time["spec_charge_time"]
selected_spectrum_time["map_dt_s"] = (selected_spectrum_time["time/s"] - selected_spectrum_time["spec_time"]).dt.total_seconds().abs()
selected_spectrum_time["range_idx"] = selected_spectrum_time["spectrum_col"].str.extract(r"(\d+)$", expand=False).astype("Int64")

# 7) 选择谱线区间并建立绘图矩阵
d_spacing_range = (0.5, 15.0)
spectrum_all = spectrum_all.loc[(spectrum_all.index >= d_spacing_range[0]) & (spectrum_all.index <= d_spacing_range[1])].copy()

selected_spectrum_time = selected_spectrum_time.dropna(subset=["range_idx"]).copy()
selected_spectrum_time = selected_spectrum_time.drop_duplicates(subset=["spectrum_col"], keep="first").sort_values(by="spec_time").reset_index(drop=True)

scan_cols_plot = selected_spectrum_time["spectrum_col"].astype(str).tolist()
selected_spectrum_plot = spectrum_all.reindex(columns=scan_cols_plot).copy()

# 统一时间显示窗口：覆盖电化学与谱线时间，并上下留 15 min
spec_time_arr = selected_spectrum_time["spec_time_h"].to_numpy(dtype=float)
pad_h = 0.25
time_min = min(float(selected_echem["charge_time"].min()), float(spec_time_arr.min()))
time_max = max(float(selected_echem["charge_time"].max()), float(spec_time_arr.max()))
y_min_plot = time_min - pad_h
y_max_plot = time_max + pad_h

In [ ]:
# 读取数据, Mn

path_cell2_mn = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
path_file_mn = Path.joinpath(path_cell2_mn, r"FC1st-FD2nd\Oct2022_cell2_P2b\XANES\Mn\PCA_Larch_Mn")
xas_Mn_path_1 = Path.joinpath(path_file_mn, r"lcf_alpha_MnO2_electrode_pristine_residual_1.nor")
xas_Mn_path_2 = Path.joinpath(path_file_mn, r"lcf_alpha_MnO2_electrode_pristine_residual_2.nor")

# 电化学时间序列
path_filelist_mn = list(Path.joinpath(path_cell2_mn, r"Echem").glob(r"**\*.txt"))
echem_all_mn = []
for path_file_echem_mn in path_filelist_mn:
    with open(path_file_echem_mn, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip_mn = int(line.split(":")[1].strip())
                break

    df_mn = pd.read_csv(path_file_echem_mn, sep="\t", comment="#", skiprows=line_skip_mn - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df_mn[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df_mn[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df_mn["time/s"] = df_mn["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df_mn["cycle number"] = df_mn["cycle number"].astype(float).astype(np.int16)
    echem_all_mn.append(df_mn)

echem_all_mn = pd.concat(echem_all_mn, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 当前对齐后的 Mn operando 数据
xas_Mn_1 = pd.read_csv(
    xas_Mn_path_1,
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Mn_2 = pd.read_csv(
    xas_Mn_path_2,
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)

# 两段 Mn residual .nor 的 energy 网格略有差异
energy_Mn = xas_Mn_1.iloc[:, 0].to_numpy(dtype=float)
energy_Mn_2 = xas_Mn_2.iloc[:, 0].to_numpy(dtype=float)
energy_mask_Mn = (energy_Mn >= float(np.nanmin(energy_Mn_2))) & (energy_Mn <= float(np.nanmax(energy_Mn_2)))
xas_Mn = xas_Mn_1.loc[energy_mask_Mn].reset_index(drop=True)
energy_Mn_common = xas_Mn.iloc[:, 0].to_numpy(dtype=float)
for col in range(1, xas_Mn_2.shape[1]):
    xas_Mn[xas_Mn.shape[1]] = np.interp(
        energy_Mn_common,
        energy_Mn_2,
        xas_Mn_2.iloc[:, col].to_numpy(dtype=float),
    )

pdf_Mn = xas_Mn.iloc[:, [0, 1, 2]]  # noqa: N816
pdf_Mn.columns = [r"Energy", r"0.2M_MnSO4(aq.)", r"alpha-MnO2"]
opData_Mn = xas_Mn.iloc[:, [0, *list(range(3, xas_Mn.shape[1]))]]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))


In [ ]:
# 聚焦当前对齐后 Mn 数据
opData_Mn = opData_Mn[opData_Mn.iloc[:, 0].between(6530.0, 6610.0)].copy()

# 保留原来标记为异常的谱线，并移除最后一条 Mn 谱线
bad_spec_cols_mn = [47]
bad_spec_cols_mn = [idx for idx in bad_spec_cols_mn if 1 <= idx < opData_Mn.shape[1]]
if bad_spec_cols_mn:
    opData_Mn.iloc[:, bad_spec_cols_mn] = np.nan



In [ ]:
n_spec_mn = opData_Mn.shape[1] - 1
special_idx_mn = np.array([0, 4, 9, 12, 14, 17, 24, 30, 34, 38], dtype=int)
special_idx_mn = special_idx_mn[(special_idx_mn >= 0) & (special_idx_mn < n_spec_mn)]

energy_mn = opData_Mn.iloc[:, 0].to_numpy(dtype=float)
mu_mat_mn = opData_Mn.iloc[:, 1:].to_numpy(dtype=float).T  # (n_spec_mn, n_energy_mn)

# 电化学窗口：首圈放电谷值到次圈充电峰值
echem_index_low_mn = echem_all_mn[echem_all_mn["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high_mn = echem_all_mn[echem_all_mn["cycle number"] == 1]["Ewe/V"].idxmax()
echem_seg_mn = echem_all_mn.iloc[int(echem_index_low_mn) : int(echem_index_high_mn) + 1].copy()
echem_seg_mn = echem_seg_mn.dropna(subset=["time/s", "Ewe/V", "<I>/mA"]).sort_values("time/s").reset_index(drop=True)

# 直接从当前 LCF .nor 表头读取列名，用于谱线到采谱时间的映射
def _read_nor_cols_mn(path_nor):
    cols = {}
    with open(path_nor, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.startswith("# Column."):
                if line.startswith("#------------------------"):
                    break
                continue
            left, right = line.split(":", 1)
            cols[int(left.split(".")[1])] = right.strip().lstrip("-").strip()
    return cols

cols_Mn_1 = _read_nor_cols_mn(xas_Mn_path_1)
cols_Mn_2 = _read_nor_cols_mn(xas_Mn_path_2)
op_labels_mn = [cols_Mn_1[i] for i in range(4, xas_Mn_1.shape[1] + 1) if i in cols_Mn_1 and re.search(r"P2b_(\d+)", cols_Mn_1[i])]
op_labels_mn.extend([cols_Mn_2[i] for i in range(2, xas_Mn_2.shape[1] + 1) if i in cols_Mn_2 and re.search(r"P2b_(\d+)", cols_Mn_2[i])])

if len(op_labels_mn) != n_spec_mn:
    raise ValueError(f"FigureSI 12A: op_labels_mn={len(op_labels_mn)} != n_spec_mn={n_spec_mn}")

map_df_mn = pd.read_csv(Path.joinpath(path_cell2_mn, r"Overview", r"Time_Index_Spectrum.csv"))
map_df_mn = map_df_mn[
    (map_df_mn["Folder"].astype(str).str.contains("cell2_P2b", case=False, na=False))
    & (map_df_mn["Element"].astype(str).str.upper() == "MN")
    & map_df_mn["NorName"].notna()
].copy()
map_df_mn["Time"] = pd.to_datetime(map_df_mn["Time"], errors="coerce")
map_df_mn["NorName"] = map_df_mn["NorName"].astype(str).str.strip().str.lstrip("-")
map_df_mn = map_df_mn.dropna(subset=["Time"]).sort_values("Time")
name_to_time_mn = map_df_mn.drop_duplicates(subset=["NorName"], keep="first").set_index("NorName")["Time"]

label_s_mn = pd.Series(op_labels_mn, dtype="string").str.strip().str.lstrip("-")
spec_time_dt_mn = label_s_mn.map(name_to_time_mn)
missing_labels_mn = label_s_mn[spec_time_dt_mn.isna()].tolist()
if missing_labels_mn:
    raise ValueError(f"FigureSI 12A: missing NorName->Time mapping for labels: {missing_labels_mn[:6]}")

# 只保留落在电化学窗口内的谱线，避免 np.interp 对窗口外谱线使用边界值
echem_start_mn = echem_seg_mn["time/s"].iloc[0]
echem_end_mn = echem_seg_mn["time/s"].iloc[-1]
in_echem_window_mn = spec_time_dt_mn.between(echem_start_mn, echem_end_mn).to_numpy(dtype=bool)
if not np.all(in_echem_window_mn):
    op_labels_mn = [label for label, keep in zip(op_labels_mn, in_echem_window_mn) if keep]
    spec_time_dt_mn = spec_time_dt_mn[in_echem_window_mn].reset_index(drop=True)
    mu_mat_mn = mu_mat_mn[in_echem_window_mn, :]
    n_spec_mn = mu_mat_mn.shape[0]
special_idx_mn = special_idx_mn[(special_idx_mn >= 0) & (special_idx_mn < n_spec_mn)]

# 统一时间零点，并把采谱点映射到电化学轨迹上
time_zero_mn = min(echem_seg_mn["time/s"].iloc[0], spec_time_dt_mn.min())
echem_time_mn = (echem_seg_mn["time/s"] - time_zero_mn).dt.total_seconds().to_numpy(dtype=float) / 3600.0
echem_voltage_mn = echem_seg_mn["Ewe/V"].to_numpy(dtype=float)
echem_current_ua_mn = echem_seg_mn["<I>/mA"].to_numpy(dtype=float) * 1000.0

spec_time_raw_mn = (spec_time_dt_mn - time_zero_mn).dt.total_seconds().to_numpy(dtype=float) / 3600.0
spec_time_mn = spec_time_raw_mn.copy()
spec_volt_mn = np.interp(spec_time_mn, echem_time_mn, echem_voltage_mn)
spec_curr_ua_mn = np.interp(spec_time_mn, echem_time_mn, echem_current_ua_mn)


In [ ]:
# 读取数据

path_cell2_zn = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
path_file_zn = Path.joinpath(path_cell2_zn, r"FC1st-FD2nd\Oct2022_cell2_P2b\XANES\Zn\PCA_Larch_Zn")
xas_Zn_path = Path.joinpath(path_file_zn, r"lcf_ZHS_residual.nor")

# 电化学时间序列
path_filelist_zn = list(Path.joinpath(path_cell2_zn, r"Echem").glob(r"**\*.txt"))
echem_all_zn = []
for path_file_echem_zn in path_filelist_zn:
    with open(path_file_echem_zn, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip_zn = int(line.split(":")[1].strip())
                break

    df_zn = pd.read_csv(path_file_echem_zn, sep="\t", comment="#", skiprows=line_skip_zn - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df_zn[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df_zn[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df_zn["time/s"] = df_zn["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df_zn["cycle number"] = df_zn["cycle number"].astype(float).astype(np.int16)
    echem_all_zn.append(df_zn)

echem_all_zn = pd.concat(echem_all_zn, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 当前对齐后的 Zn operando 数据
xas_Zn = pd.read_csv(
    xas_Zn_path,
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)

charge_index_zn = [0, 1, 2, *list(range(12, xas_Zn.shape[1], 1))]
xas_Zn = xas_Zn.iloc[:, charge_index_zn]  # noqa: N816

pdf_Zn = xas_Zn.iloc[:, [0, 1, 2]]  # noqa: N816
pdf_Zn.columns = [r"Energy", r"0.5M_ZnSO4(aq.)", r"ZHS"]
opData_Zn = xas_Zn.iloc[:, [0, *list(range(3, xas_Zn.shape[1]))]]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))


In [ ]:
# 聚焦当前对齐后 Zn 数据
opData_Zn = opData_Zn[opData_Zn.iloc[:, 0].between(9640.0, 9740.0)].copy()

# 保留原来标记为异常的两条谱线
bad_spec_cols_zn = [11, 36, 37, 38]
bad_spec_cols_zn = [idx for idx in bad_spec_cols_zn if idx < opData_Zn.shape[1]]
if bad_spec_cols_zn:
    opData_Zn.iloc[:, bad_spec_cols_zn] = np.nan


In [ ]:
n_spec_zn = opData_Zn.shape[1] - 1
special_idx_zn = np.array([0, 4, 9, 12, 14, 17, 24, 30, 34, 38], dtype=int)
special_idx_zn = special_idx_zn[(special_idx_zn >= 0) & (special_idx_zn < n_spec_zn)]

energy_zn = opData_Zn.iloc[:, 0].to_numpy(dtype=float)
mu_mat_zn = opData_Zn.iloc[:, 1:].to_numpy(dtype=float).T  # (n_spec_zn, n_energy_zn)

# 电化学窗口：首圈放电谷值到次圈充电峰值
echem_index_low_zn = echem_all_zn[echem_all_zn["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high_zn = echem_all_zn[echem_all_zn["cycle number"] == 1]["Ewe/V"].idxmax()
echem_seg_zn = echem_all_zn.iloc[int(echem_index_low_zn) : int(echem_index_high_zn) + 1].copy()
echem_seg_zn = echem_seg_zn.dropna(subset=["time/s", "Ewe/V", "<I>/mA"]).sort_values("time/s").reset_index(drop=True)

# 直接从当前 LCF .nor 表头读取列名，用于谱线到采谱时间的映射
def _read_nor_cols_zn(path_nor):
    cols = {}
    with open(path_nor, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.startswith("# Column."):
                if line.startswith("#------------------------"):
                    break
                continue
            left, right = line.split(":", 1)
            cols[int(left.split(".")[1])] = right.strip().lstrip("-").strip()
    return cols

cols_zn = _read_nor_cols_zn(xas_Zn_path)
op_source_cols_zn = charge_index_zn[3:]
op_labels_zn = [cols_zn[i + 1] for i in op_source_cols_zn if i + 1 in cols_zn and re.search(r"P2b_(\d+)", cols_zn[i + 1])]

if len(op_labels_zn) != n_spec_zn:
    raise ValueError(f"FigureSI 12A: op_labels_zn={len(op_labels_zn)} != n_spec_zn={n_spec_zn}")

map_df_zn = pd.read_csv(Path.joinpath(path_cell2_zn, r"Overview", r"Time_Index_Spectrum.csv"))
map_df_zn = map_df_zn[
    (map_df_zn["Folder"].astype(str).str.contains("cell2_P2b", case=False, na=False))
    & (map_df_zn["Element"].astype(str).str.upper() == "ZN")
    & map_df_zn["NorName"].notna()
].copy()
map_df_zn["Time"] = pd.to_datetime(map_df_zn["Time"], errors="coerce")
map_df_zn["NorName"] = map_df_zn["NorName"].astype(str).str.strip().str.lstrip("-")
map_df_zn = map_df_zn.dropna(subset=["Time"]).sort_values("Time")
name_to_time_zn = map_df_zn.drop_duplicates(subset=["NorName"], keep="first").set_index("NorName")["Time"]

label_s_zn = pd.Series(op_labels_zn, dtype="string").str.strip().str.lstrip("-")
spec_time_dt_zn = label_s_zn.map(name_to_time_zn)
missing_labels_zn = label_s_zn[spec_time_dt_zn.isna()].tolist()
if missing_labels_zn:
    raise ValueError(f"FigureSI 12A: missing NorName->Time mapping for labels: {missing_labels_zn[:6]}")

# 只保留落在电化学窗口内的谱线，避免 np.interp 对窗口外谱线使用边界值
echem_start_zn = echem_seg_zn["time/s"].iloc[0]
echem_end_zn = echem_seg_zn["time/s"].iloc[-1]
in_echem_window_zn = spec_time_dt_zn.between(echem_start_zn, echem_end_zn).to_numpy(dtype=bool)
if not np.all(in_echem_window_zn):
    op_labels_zn = [label for label, keep in zip(op_labels_zn, in_echem_window_zn) if keep]
    spec_time_dt_zn = spec_time_dt_zn[in_echem_window_zn].reset_index(drop=True)
    mu_mat_zn = mu_mat_zn[in_echem_window_zn, :]
    old_special_idx_zn = special_idx_zn.copy()
    kept_old_idx_zn = np.flatnonzero(in_echem_window_zn)
    special_idx_zn = np.array([np.where(kept_old_idx_zn == idx)[0][0] for idx in old_special_idx_zn if idx in kept_old_idx_zn], dtype=int)
    n_spec_zn = mu_mat_zn.shape[0]
special_idx_zn = np.unique(np.r_[special_idx_zn, n_spec_zn - 1]).astype(int)

# 统一时间零点，并把采谱点映射到电化学轨迹上
time_zero_zn = min(echem_seg_zn["time/s"].iloc[0], spec_time_dt_zn.min())
time_shift_zn_to_mn = (time_zero_zn - time_zero_mn).total_seconds() / 3600.0
echem_time_zn = (echem_seg_zn["time/s"] - time_zero_zn).dt.total_seconds().to_numpy(dtype=float) / 3600.0
echem_time_zn = echem_time_zn + time_shift_zn_to_mn
echem_voltage_zn = echem_seg_zn["Ewe/V"].to_numpy(dtype=float)
echem_current_ua_zn = echem_seg_zn["<I>/mA"].to_numpy(dtype=float) * 1000.0

spec_time_raw_zn = (spec_time_dt_zn - time_zero_zn).dt.total_seconds().to_numpy(dtype=float) / 3600.0
spec_time_zn = spec_time_raw_zn.copy() + time_shift_zn_to_mn
spec_volt_zn = np.interp(spec_time_zn, echem_time_zn, echem_voltage_zn)
spec_curr_ua_zn = np.interp(spec_time_zn, echem_time_zn, echem_current_ua_zn)


In [ ]:
# 读取 Mn/Zn 对齐数据
path_align_root = Path.joinpath(
    path_data_root,
    r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell2_P2b\XANES\20260515\画图需要的",
)

path_align_echem = Path.joinpath(path_align_root, r"Echem", r"cell2_03C_0.7mg.txt")
path_align_mn = Path.joinpath(path_align_root, r"concentrationsMn.csv")
path_align_zn = Path.joinpath(path_align_root, r"concentrationsZn.csv")
path_align_time = Path.joinpath(path_align_root, r"Time_Index_Spectrum.csv")

def _normalize_align_name(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    return text.removeprefix("merged_").removesuffix(".dat").replace(" ", "_")

def _extract_align_sample_no(value):
    if value is None or pd.isna(value):
        return None
    match = re.search(r"cell2_P2b_(\d+)", str(value))
    return int(match.group(1)) if match else None

echem_align = pd.read_csv(path_align_echem, sep="\t", skiprows=87, encoding="latin1")
echem_align = echem_align.drop(columns=[c for c in echem_align.columns if str(c).startswith("Unnamed")])
echem_align.columns = [str(c).strip() for c in echem_align.columns]
echem_align["dt"] = pd.to_datetime(echem_align["time/s"], format="%m/%d/%Y %H:%M:%S.%f", errors="coerce")
echem_align = echem_align.dropna(subset=["dt", "Ewe/V", "<I>/mA"]).sort_values("dt").reset_index(drop=True)
echem_align["elapsed_h"] = (echem_align["dt"] - echem_align["dt"].iloc[0]).dt.total_seconds() / 3600.0
echem_align["current_state"] = np.select(
    [echem_align["<I>/mA"] < -1e-3, echem_align["<I>/mA"] > 1e-3],
    ["discharge", "charge"],
    default="rest",
)
echem_align["state_block"] = echem_align["current_state"].ne(echem_align["current_state"].shift()).cumsum()

conc_mn = pd.read_csv(path_align_mn)
conc_mn["key"] = conc_mn["sample_name"].map(_normalize_align_name)
conc_mn["sample_no"] = conc_mn["key"].map(_extract_align_sample_no)
conc_mn = conc_mn.rename(columns={
    "coef_0.2M_MnSO4": "mn_coef_0.2M_MnSO4",
    "coef_alpha_MnO2": "mn_coef_alpha_MnO2",
    "coef_FC1st_candidate_3": "mn_coef_FC1st_candidate_3",
    "coef_Phase4_free": "mn_coef_Phase4_free",
    "frac_0.2M_MnSO4": "mn_frac_0.2M_MnSO4",
    "frac_alpha_MnO2": "mn_frac_alpha_MnO2",
    "frac_FC1st_candidate_3": "mn_frac_FC1st_candidate_3",
    "frac_Phase4_free": "mn_frac_Phase4_free",
})
conc_mn = conc_mn[[
    "sample_no", "sample_index", "sample_name", "key",
    "mn_coef_0.2M_MnSO4", "mn_coef_alpha_MnO2", "mn_coef_FC1st_candidate_3", "mn_coef_Phase4_free",
    "mn_frac_0.2M_MnSO4", "mn_frac_alpha_MnO2", "mn_frac_FC1st_candidate_3", "mn_frac_Phase4_free",
]].rename(columns={
    "sample_index": "sample_index_mn",
    "sample_name": "sample_name_mn",
    "key": "key_mn",
})

conc_zn = pd.read_csv(path_align_zn)
conc_zn["key"] = conc_zn["sample_name"].map(_normalize_align_name)
conc_zn["sample_no"] = conc_zn["key"].map(_extract_align_sample_no)
conc_zn = conc_zn.rename(columns={
    "coef_P1b_phase1": "zn_coef_P1b_phase1",
    "frac_P1b_phase1": "zn_frac_P1b_phase1",
    "coef_P1b_phase2": "zn_coef_P1b_phase2",
    "frac_P1b_phase2": "zn_frac_P1b_phase2",
    "coef_ZHS_seed": "zn_coef_ZHS_seed",
    "frac_ZHS_seed": "zn_frac_ZHS_seed",
})
conc_zn = conc_zn[[
    "sample_no", "sample_index", "sample_name", "key",
    "zn_coef_P1b_phase1", "zn_frac_P1b_phase1", "zn_coef_P1b_phase2", "zn_frac_P1b_phase2", "zn_coef_ZHS_seed", "zn_frac_ZHS_seed",
]].rename(columns={
    "sample_index": "sample_index_zn",
    "sample_name": "sample_name_zn",
    "key": "key_zn",
})

time_index_align = pd.read_csv(path_align_time)
time_index_align = time_index_align.loc[time_index_align["Folder"].eq("cell2_P2b_ca")].copy()
time_index_align["key"] = time_index_align["NorName"].map(_normalize_align_name)
time_index_align["sample_no"] = time_index_align["key"].map(_extract_align_sample_no)
time_index_align = time_index_align.loc[time_index_align["sample_no"].notna()].copy()
time_index_align["dt"] = pd.to_datetime(time_index_align["Time"], format="%Y/%m/%d %H:%M", errors="coerce")
time_index_align = time_index_align.dropna(subset=["dt"]).copy()

time_index_mn = (
    time_index_align.loc[time_index_align["Element"].eq("Mn"), ["sample_no", "dt", "key"]]
    .drop_duplicates(subset=["sample_no"], keep="first")
    .rename(columns={"dt": "mn_dt", "key": "mn_key"})
)
time_index_zn = (
    time_index_align.loc[time_index_align["Element"].eq("Zn"), ["sample_no", "dt", "key"]]
    .drop_duplicates(subset=["sample_no"], keep="first")
    .rename(columns={"dt": "zn_dt", "key": "zn_key"})
)
time_index_pair = pd.merge(time_index_mn, time_index_zn, on="sample_no", how="outer")
time_index_pair["sample_dt"] = time_index_pair[["mn_dt", "zn_dt"]].mean(axis=1)

aligned_mnzn = pd.merge(conc_zn, conc_mn, on="sample_no", how="outer")
aligned_mnzn = pd.merge(aligned_mnzn, time_index_pair, on="sample_no", how="outer")
aligned_mnzn = aligned_mnzn.sort_values("sample_no").reset_index(drop=True)
aligned_mnzn["display_name"] = aligned_mnzn["sample_name_mn"].combine_first(aligned_mnzn["sample_name_zn"])
aligned_mnzn["sample_key"] = aligned_mnzn["key_mn"].combine_first(aligned_mnzn["key_zn"]).combine_first(aligned_mnzn["mn_key"]).combine_first(aligned_mnzn["zn_key"])
aligned_mnzn = aligned_mnzn.dropna(subset=["sample_dt"]).sort_values("sample_dt").reset_index(drop=True)

aligned_mnzn = pd.merge_asof(
    aligned_mnzn,
    echem_align[["dt", "elapsed_h", "Ewe/V", "<I>/mA", "cycle number"]].sort_values("dt"),
    left_on="sample_dt",
    right_on="dt",
    direction="nearest",
    tolerance=pd.Timedelta("5min"),
).sort_values("sample_no").reset_index(drop=True)
aligned_mnzn = aligned_mnzn.rename(columns={"Ewe/V": "ewe_v", "<I>/mA": "current_mA", "cycle number": "cycle_number"})
aligned_mnzn = aligned_mnzn.dropna(subset=["elapsed_h", "ewe_v"]).reset_index(drop=True)

mn_phase_specs = [
    ("mn_frac_0.2M_MnSO4", r"MnSO$_4$(aq.)", colors[0]),
    ("mn_frac_alpha_MnO2", r"$\alpha$-MnO$_2$", colors[1]),
    ("mn_frac_FC1st_candidate_3", "Phase 3", colors[4]),
    ("mn_frac_Phase4_free", "Phase 4", colors[5]),
]
zn_phase_specs = [
    ("zn_frac_P1b_phase2", r"Phase 1", colors[2]),
    ("zn_frac_ZHS_seed", "ZHS", colors[3]),
    ("zn_frac_P1b_phase1", "Phase 2", colors[4]),
]

In [ ]:
# 画图
plt.close("all")

fig = plt.figure(figsize=(7.0, 5.0))
layout = [["A", "B", "C", "GHI"], ["D", "E", "F", "GHI"], ["D", "E", "F", "GHI"]]
axs = fig.subplot_mosaic(
    layout,
    gridspec_kw={"width_ratios": [0.3, 1.0, 1.0, 1.0], "height_ratios": [1.0, 1.0, 1.0], "wspace": 0.0, "hspace": 0.0},
)

# 图 A
ax = axs["A"]

ax.plot(selected_echem["Voltage/V"], selected_echem["charge_time"], ls="-", lw=1.0, c=colors[0], label=r"Voltage")

index_labels = [15, 25, 37, 46, 50, 61, 70, 75, 87, 99, 109, 115, 126, 137]
range_idx_int = pd.to_numeric(selected_spectrum_time["range_idx"], errors="coerce").astype("Int64")
special_mask = range_idx_int.isin(index_labels)

plot_points = selected_spectrum_time.copy()
plot_points["is_special"] = special_mask.to_numpy()
special_points = plot_points[plot_points["is_special"]].copy()

for _, row in plot_points.iterrows():
    if bool(row["is_special"]):
        ax.scatter(
            row["Voltage/V"],
            row["spec_time_h"],
            c=colors[0],
            edgecolors="face",
            marker="*",
            s=50,
            zorder=5,
        )
    else:
        ax.scatter(
            row["Voltage/V"],
            row["spec_time_h"],
            c="lightgrey",
            edgecolors="face",
            s=30,
            zorder=1,
        )

ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_xlim(0.8, 2.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.6, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.3, offset=0))

ax.set_ylabel(r"Duration Time (h)", fontsize=9)
ax.set_ylim(y_min_plot, y_max_plot)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))
ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, left=True, labelbottom=True, labelleft=True, labelsize=8)

ax2 = ax.twiny()
ax2.plot(selected_echem["<I>/mA"] * 1000, selected_echem["charge_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")
ax2.set_xlabel(r"$\mathrm{Current  \ (\mu A)}$", fontsize=9)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.set_ylim(y_min_plot, y_max_plot)
ax2.tick_params(axis="both", which="both", labelcolor="k", top=True, labeltop=True, labelsize=8)

ax.text(-0.7, 1.0, "A", transform=ax.transAxes, fontsize=10, fontweight="bold", va="center", ha="right")

# 图 B
spec_time_arr = selected_spectrum_time["spec_time_h"].to_numpy(dtype=float)
if selected_spectrum_plot.shape[1] != len(spec_time_arr):
    raise ValueError(f"spectrum/time length mismatch: {selected_spectrum_plot.shape[1]} vs {len(spec_time_arr)}")

d_axis = selected_spectrum_plot.index.to_numpy(dtype=float)
d_grid, t_grid = np.meshgrid(d_axis, spec_time_arr)

special_y_time = (
    selected_spectrum_time[selected_spectrum_time["range_idx"].isin(index_labels)]["spec_time_h"]
    .dropna()
    .astype(float)
    .tolist()
)

# 图 B
ax = axs["B"]
pos = ax.get_position()
ax.set_position([pos.x0+0.005, pos.y0, pos.width, pos.height])
ax.set_box_aspect(1.0)

pc_b = ax.pcolormesh(d_grid, t_grid, selected_spectrum_plot.T.to_numpy(dtype=float), cmap="coolwarm", shading="gouraud", vmin=6000, vmax=42000)
ax.set_yticks([])
ax.set_ylim(float(spec_time_arr.min()), float(spec_time_arr.max()))
ax.set_xlim(2.0, 8.0)
ax.set_xlabel(r"d Spacing ($\AA$)", fontsize=9)
ax.tick_params(axis="both", which="both", labelsize=8)

cax_b = ax.inset_axes((1.06, 0.1, 0.06, 0.8))
cb_b = fig.colorbar(pc_b, cax=cax_b)
cb_b.set_ticks([])
ax.text(1.1, 0.98, "High", transform=ax.transAxes, fontsize=8, va="top", ha="center")
ax.text(1.1, 0.02, "Low", transform=ax.transAxes, fontsize=8, va="bottom", ha="center")

ax.text(-0.03, 1.0, "B", transform=ax.transAxes, fontsize=10, fontweight="bold", va="center", ha="right")

# 图 C
base_c = axs["C"]
base_c.set_axis_off()
opxrd_ref_d_spacing = [1.414916, 2.44884, 5.967425, 8.042391, 2.71, 3.23, 5.47, 10.94]
peak_widths = [
    (0.05, 0.05),
    (0.05, 0.05),
    (0.4, 0.4),
    (0.4, 0.4),
    (0.1, 0.1),
    (0.1, 0.1),
    (0.1, 0.1),
    (0.4, 0.4),
]

# ========== 改动开始 ==========
# 固定每个子图的宽度和高度（相对于 figure 的比例）
fixed_width = 0.04      # 每个子图宽度占 figure 宽度的比例
fixed_height = 0.33      # 每个子图高度占 figure 高度的比例（可根据原图效果调整）
wspace = 0.005           # 子图之间的水平间距（比例）

n = len(opxrd_ref_d_spacing)
total_width = n * fixed_width + (n - 1) * wspace
pos_base = base_c.get_position()
# 调整 base_c 的宽度以容纳所有子图（左边界不变）
base_c.set_position([pos_base.x0, pos_base.y0, total_width, pos_base.height])

c_shift = 0.05       # 可选的额外右移量，保留原代码中的偏移
# ========== 改动结束 ==========

for i, d0 in enumerate(opxrd_ref_d_spacing):
    # ========== 改动开始：手动计算子图位置并创建 ==========
    x0 = pos_base.x0 + i * (fixed_width + wspace) + c_shift
    y0 = pos_base.y0    # 与 base_c 同高，也可微调
    axc = fig.add_axes([x0, y0, fixed_width, fixed_height])
    # ========== 改动结束 ==========
    
    temp = selected_spectrum_plot.loc[(selected_spectrum_plot.index >= d0 - peak_widths[i][0]) & 
                                      (selected_spectrum_plot.index <= d0 + peak_widths[i][1])]
    if temp.empty:
        axc.set_axis_off()
        continue
    td, tt = np.meshgrid(temp.index.to_numpy(dtype=float), spec_time_arr)
    axc.pcolormesh(td, tt, temp.T.to_numpy(dtype=float), cmap="coolwarm", shading="gouraud")
    axc.axvline(d0, ls="--", lw=0.8, color="grey", alpha=0.5)
    axc.yaxis.set_ticks([])
    axc.set_ylim(float(spec_time_arr.min()), float(spec_time_arr.max()))
    axc.set_xlim(float(temp.index.min()), float(temp.index.max()))
    axc.xaxis.set_major_locator(ticker.FixedLocator([d0]))
    axc.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    axc.spines[["top", "right", "left"]].set_visible(False)
    axc.tick_params(axis="both", which="both", labelsize=8)
    for y_sp in special_y_time:
        axc.axhline(y=y_sp, lw=0.6, ls="--", color=colors[1], alpha=0.45)

base_c.text(0.12, 1.0, "C", transform=base_c.transAxes, fontsize=10, fontweight="bold", va="center", ha="right")

# 图 D
ax_echem =axs["D"]
pos = ax_echem.get_position()
ax_echem.set_position((pos.x0, pos.y0-0.2, pos.width, pos.height))
# ax_echem.set_box_aspect(3.0)

y_min = min(float(np.nanmin(echem_time_mn)), float(np.nanmin(spec_time_mn)), float(np.nanmin(spec_time_zn)))
y_max = max(float(np.nanmax(echem_time_mn)), float(np.nanmax(spec_time_mn)), float(np.nanmax(spec_time_zn)))

ax_echem.plot(echem_voltage_mn, echem_time_mn, c=colors[0], ls="-", lw=1.0, zorder=1)

all_idx_mn = np.arange(len(spec_time_mn), dtype=int)
special_mask_mn = np.zeros(len(spec_time_mn), dtype=bool)
if special_idx_mn.size > 0:
    special_mask_mn[special_idx_mn[special_idx_mn < len(spec_time_mn)]] = True
normal_idx_mn = all_idx_mn[(~special_mask_mn) & (all_idx_mn != len(spec_time_mn) - 1)]

if normal_idx_mn.size > 0:
    ax_echem.scatter(spec_volt_mn[normal_idx_mn], spec_time_mn[normal_idx_mn], c="lightgrey", edgecolors="face", s=18, zorder=2)
if special_idx_mn.size > 0:
    sp_mn = special_idx_mn[special_idx_mn < len(spec_time_mn)]
    if sp_mn.size > 0:
        ax_echem.scatter(spec_volt_mn[sp_mn], spec_time_mn[sp_mn], c=colors[0], edgecolors="w", marker="*", s=42, linewidths=0.35, zorder=4)

ax_echem.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax_echem.set_xlim(0.8, 2.0)
ax_echem.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=0.0))
ax_echem.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=0.0))
ax_echem.set_ylabel("Duration time (h)", fontsize=9)
ax_echem.set_ylim(y_min - 0.25, y_max + 1)
ax_echem.yaxis.set_major_locator(ticker.MultipleLocator(base=3.0, offset=0.0))
ax_echem.yaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=0.0))
ax_echem.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)

ax_echem.text(-0.7, 1.0, r"D", transform=ax_echem.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

ax_echem_top = ax_echem.twiny()
pos = ax_echem_top.get_position()
ax_echem_top.set_position((pos.x0, pos.y0-0.2, pos.width, pos.height))
# ax_echem_top.set_box_aspect(3.0)

ax_echem_top.plot(echem_current_ua_mn, echem_time_mn, c=colors[3], ls="--", lw=1.0, alpha=0.9, zorder=1)
ax_echem_top.set_xlim(-60, 60)
ax_echem_top.xaxis.set_major_locator(ticker.MultipleLocator(base=60.0, offset=0.0))
ax_echem_top.xaxis.set_minor_locator(ticker.MultipleLocator(base=30.0, offset=0.0))
ax_echem_top.set_xlabel(r"Current ($\mu$A)", fontsize=9)
ax_echem_top.set_ylim(y_min - 0.25, y_max + 1)
ax_echem_top.tick_params(axis="x", which="both", labelsize=8)

# 图 E
ax_spec = axs["E"]
ax_spec.shared_y_axes = ax_echem
pos = ax_spec.get_position()
ax_spec.set_position((pos.x0+0.045, pos.y0-0.2, pos.width*1.02, pos.height))
ax_spec.set_box_aspect(1.55)

xas_colors = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(0.0, 1.0, n_spec_mn)),
    name="mn_colors",
)

time_sorted_mn = np.sort(np.unique(np.asarray(spec_time_mn, dtype=float)))
time_diff_mn = np.diff(time_sorted_mn)
time_diff_mn = time_diff_mn[time_diff_mn > 0]
time_step_mn = float(np.nanmedian(time_diff_mn)) if time_diff_mn.size > 0 else 0.45
trace_half_height_mn = max(0.12, 0.34 * time_step_mn)

for m in range(n_spec_mn):
    spectrum = mu_mat_mn[m, :]
    if np.all(np.isnan(spectrum)):
        continue

    spec_min = float(np.nanmin(spectrum))
    spec_max = float(np.nanmax(spectrum))
    y_plot = spec_time_mn[m] + (spectrum - spec_min) * 2.0
    ax_spec.plot(energy_mn, y_plot, c=xas_colors.colors[m], ls="-", lw=1.0, zorder=2, clip_on=False)

# 特殊索引谱线再覆盖画一遍黑线
for m in special_idx_mn:
    spectrum = mu_mat_mn[m, :]
    if np.all(np.isnan(spectrum)):
        continue

    spec_min = float(np.nanmin(spectrum))
    spec_max = float(np.nanmax(spectrum))
    y_plot = spec_time_mn[m] + (spectrum - spec_min) * 2.0
    ax_spec.plot(energy_mn, y_plot, c="k", ls="-", lw=1.0, zorder=4, clip_on=False)

ax_spec.set_xlim(6530.0, 6610.0)
ax_spec.spines["top"].set_visible(False)
ax_spec.set_xlabel(r"Energy (eV)", fontsize=9)
ax_spec.xaxis.set_major_locator(ticker.MultipleLocator(base=20.0, offset=-10.0))
ax_spec.xaxis.set_minor_locator(ticker.MultipleLocator(base=10.0, offset=-10.0))
ax_spec.set_ylim(y_min - 0.25, y_max + 1)
ax_spec.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_spec.tick_params(axis="y", which="both", left=False, labelleft=False)
ax_spec.set_ylabel("")
ax_spec.text(-0.04, 1.0, r"E", transform=ax_spec.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 F
ax = axs["F"]
ax.shared_y_axes = ax_echem
pos = ax.get_position()
ax.set_position((pos.x0+0.095, pos.y0-0.2, pos.width*1.02, pos.height))
ax.set_box_aspect(1.55)

xas_colors_zn = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(0.0, 1.0, n_spec_zn)),
    name="zn_colors",
)

for m in range(n_spec_zn):
    spectrum = mu_mat_zn[m, :]
    if np.all(np.isnan(spectrum)):
        continue

    spec_min = float(np.nanmin(spectrum))
    spec_max = float(np.nanmax(spectrum))
    y_plot = spec_time_zn[m] + (spectrum - spec_min) * 2.0
    ax.plot(energy_zn, y_plot, c=xas_colors_zn.colors[m], ls="-", lw=1.0, zorder=2, clip_on=False)

for m in special_idx_zn:
    spectrum = mu_mat_zn[m, :]
    if np.all(np.isnan(spectrum)):
        continue

    spec_min = float(np.nanmin(spectrum))
    spec_max = float(np.nanmax(spectrum))
    y_plot = spec_time_zn[m] + (spectrum - spec_min) * 2.0
    ax.plot(energy_zn, y_plot, c="k", ls="-", lw=1.0, zorder=4, clip_on=False)

ax.set_xlim(9640.0, 9740.0)
ax.spines["top"].set_visible(False)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20.0, offset=0.0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10.0, offset=0.0))
ax.set_ylim(y_min - 0.25, y_max + 1)
ax.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax.tick_params(axis="y", which="both", left=False, labelleft=False)
ax.set_ylabel("")
ax.text(-0.04, 1.0, r"F", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 GHI

base_ghi = axs["GHI"]
pos_ghi = base_ghi.get_position()
base_ghi.set_axis_off()

ghi_x0 = pos_ghi.x0 + 0.18
ghi_y0 = pos_ghi.y0 - 0.2
ghi_w = pos_ghi.width*1.2
ghi_h = pos_ghi.height*1.2
ghi_gap = 0.1
ghi_panel_h = (ghi_h - 2 * ghi_gap) / 3.0

ax_g = fig.add_axes([ghi_x0, ghi_y0 + 2 * (ghi_panel_h + ghi_gap), ghi_w, ghi_panel_h])
ax_h = fig.add_axes([ghi_x0, ghi_y0 + ghi_panel_h + ghi_gap, ghi_w, ghi_panel_h], sharex=ax_g)
ax_i = fig.add_axes([ghi_x0, ghi_y0, ghi_w, ghi_panel_h], sharex=ax_g)

ax_g.set_box_aspect(0.8)
ax_h.set_box_aspect(0.8)
ax_i.set_box_aspect(0.8)

ax_g.plot(echem_time_mn, echem_voltage_mn, c=colors[0], ls="-", lw=1.0, zorder=1)
if normal_idx_mn.size > 0:
    ax_g.scatter(spec_time_mn[normal_idx_mn], spec_volt_mn[normal_idx_mn], c="lightgrey", edgecolors="face", s=18, zorder=2)
if special_idx_mn.size > 0:
    sp_mn = special_idx_mn[special_idx_mn < len(spec_time_mn)]
    if sp_mn.size > 0:
        ax_g.scatter(spec_time_mn[sp_mn], spec_volt_mn[sp_mn], c=colors[0], edgecolors="w", marker="*", s=42, linewidths=0.35, zorder=4)
ax_g.set_xlim(y_min - 0.25, y_max + 1)
ax_g.xaxis.set_major_locator(ticker.MultipleLocator(base=3.0, offset=0.0))
ax_g.xaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=0.0))
ax_g.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=8)
ax_g.set_ylim(0.8, 2.0)
ax_g.yaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=0.0))
ax_g.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=0.0))
ax_g.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_g.tick_params(axis="x", which="both", labelbottom=False)
ax_g.text(-0.23, 1.0, r"G", transform=ax_g.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

ax_g2 = ax_g.twinx()
ax_g2.plot(echem_time_mn, echem_current_ua_mn, c=colors[3], ls="--", lw=1.0, alpha=0.9, zorder=1)
ax_g2.set_xlim(y_min - 0.25, y_max + 1)
ax_g2.set_ylim(-60, 60)
ax_g2.set_ylabel(r"Current (μA)", fontsize=8)
ax_g2.yaxis.set_major_locator(ticker.MultipleLocator(base=60.0, offset=0.0))
ax_g2.yaxis.set_minor_locator(ticker.MultipleLocator(base=30.0, offset=0.0))
ax_g2.tick_params(axis="y", which="both", labelsize=8)

aligned_mnzn["time_h_mn"] = (pd.to_datetime(aligned_mnzn["mn_dt"], errors="coerce") - time_zero_mn).dt.total_seconds() / 3600.0
aligned_mnzn["time_h_zn"] = (pd.to_datetime(aligned_mnzn["zn_dt"], errors="coerce") - time_zero_mn).dt.total_seconds() / 3600.0
sample_no_ghi = pd.Series(op_labels_mn, dtype="string").str.extract(r"P2b_(\d+)", expand=False).dropna().astype(int)
sample_no_max_ghi = int(sample_no_ghi.iloc[-2])
time_low = y_min - 0.25
time_high = y_max + 1
mask_h = aligned_mnzn["time_h_mn"].between(time_low, time_high, inclusive="both") & aligned_mnzn["sample_no"].between(9, sample_no_max_ghi, inclusive="both")
mask_i = aligned_mnzn["time_h_zn"].between(time_low, time_high, inclusive="both") & aligned_mnzn["sample_no"].between(9, sample_no_max_ghi, inclusive="both")
special_sample_no_ghi = pd.Series(op_labels_mn, dtype="string").iloc[np.asarray(special_idx_mn[special_idx_mn < len(op_labels_mn)], dtype=int)].str.extract(r"P2b_(\d+)", expand=False).dropna().astype(int).tolist()
special_mask_h = mask_h & aligned_mnzn["sample_no"].isin(special_sample_no_ghi)
special_mask_i = mask_i & aligned_mnzn["sample_no"].isin(special_sample_no_ghi)

for col_name, label, color in mn_phase_specs:
    ax_h.plot(aligned_mnzn.loc[mask_h, "time_h_mn"], aligned_mnzn.loc[mask_h, col_name], c=color, ls="-", lw=1.0, marker="o", ms=3.0, label=label)
    ax_h.scatter(aligned_mnzn.loc[special_mask_h, "time_h_mn"], aligned_mnzn.loc[special_mask_h, col_name], c=color, edgecolors="w", marker="*", s=42, linewidths=0.35, zorder=4)
ax_h.set_ylabel("Mn Fraction (arb.u.)", fontsize=8)
ax_h.set_ylim(-0.03, 1.03)
ax_h.set_xlim(time_low, time_high)
ax_h.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_h.tick_params(axis="x", which="both", labelbottom=False)
ax_h.xaxis.set_major_locator(ticker.MultipleLocator(base=3.0, offset=0.0))
ax_h.xaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=0.0))
ax_h.legend(frameon=False, fontsize=7, loc="upper left", ncol=2, handlelength=1.6, columnspacing=0.8)
ax_h.text(-0.23, 1.0, r"H", transform=ax_h.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

for col_name, label, color in zn_phase_specs:
    ax_i.plot(aligned_mnzn.loc[mask_i, "time_h_zn"], aligned_mnzn.loc[mask_i, col_name], c=color, ls="-", lw=1.0, marker="o", ms=3.0, label=label)
    ax_i.scatter(aligned_mnzn.loc[special_mask_i, "time_h_zn"], aligned_mnzn.loc[special_mask_i, col_name], c=color, edgecolors="w", marker="*", s=42, linewidths=0.35, zorder=4)
ax_i.set_ylabel("Zn Fraction (arb.u.)", fontsize=8)
ax_i.set_xlabel("Duration time (h)", fontsize=8)
ax_i.set_ylim(-0.03, 1.23)
ax_i.set_xlim(time_low, time_high)
ax_i.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_i.xaxis.set_major_locator(ticker.MultipleLocator(base=3.0, offset=0.0))
ax_i.xaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=0.0))
ax_i.legend(frameon=False, fontsize=7, loc="upper left", ncol=2, handlelength=1.6, columnspacing=0.8)
ax_i.text(-0.23, 1.0, r"I", transform=ax_i.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

ax_g.spines["right"].set_visible(False)
ax_g2.spines["top"].set_visible(False)
ax_h.spines["top"].set_visible(False)
ax_h.spines["right"].set_visible(False)
ax_i.spines["top"].set_visible(False)
ax_i.spines["right"].set_visible(False)

# 保存图片
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_02_V2_300.tif"), dpi=300, bbox_inches="tight", pad_inches=0.05, pil_kwargs={"compression": "tiff_lzw"})
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_02_V2_600.tif"), dpi=600, bbox_inches="tight", pad_inches=0.05, pil_kwargs={"compression": "tiff_lzw"})
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_02_V2_600.png"), dpi=600, bbox_inches="tight", pad_inches=0.05)
# plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_02_V2_900.svg"), dpi=900, bbox_inches="tight", pad_inches=0.05)

plt.gcf().set_facecolor("white")
plt.show()


## FigureMS 03 -- 非原位 Zn-Mn 定量分析 -V1

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from scipy import ndimage as ndi


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 8},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[0].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[0].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 8},
        )
        ax.add_artist(asb)
    return asb


# 清洗数据
def remove_small_islands(arr: np.ndarray, min_component_pixels: int = 3) -> tuple[np.ndarray, int]:
    """Set tiny connected finite components to NaN."""
    out = np.array(arr, dtype=float, copy=True)
    valid = np.isfinite(out)
    if not np.any(valid):
        return out, 0
    labels, nlab = ndi.label(valid, structure=np.ones((3, 3), dtype=int))
    if nlab == 0:
        return out, 0
    comp_sizes = np.bincount(labels.ravel())
    small_labels = np.where(comp_sizes < int(max(1, min_component_pixels)))[0]
    small_labels = small_labels[small_labels != 0]
    if small_labels.size == 0:
        return out, 0
    drop_mask = np.isin(labels, small_labels)
    removed = int(np.count_nonzero(drop_mask))
    out[drop_mask] = np.nan
    return out, removed

In [ ]:
# 读取 TXM 的数据
def read_data(file_path, name, low_cut=0.20, high_cut=0.551):
    df = pd.read_csv(file_path, sep=",", header=None, comment="#")
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
    if name in [r"Pristine", r"MnOOH"]:
        return df
    else:
        return df.where((df >= low_cut) & (df <= high_cut), 0)


# TXM 的数据路径
path_data = Path.joinpath(path_data_root, r"STXM\ExSitu\Andrea")
scale = 0.013000000268220901  # 像素比例尺 (um/px)
cutoff = (0.20, 0.551)

# 定义所有数据集的信息
datasets = [
    (r"Pristine", r"Pristine/E1 Pristine/Pristine_Im_ratio2.txt"),
    (r"MnOOH", r"Pristine/F8 MnOOH/MnOOH_Imratio2.txt"),
    (r"1stDischarge", r"Charge/B6 1st0.9V/B6_Imratio2.txt"),
    (r"1stHalfCharge#1", r"Charge/F2 1st1.53V/F2_Im_ratio2.txt"),
    (r"1stHalfCharge#2", r"Charge/E3 1st1.63V/E3_Im_ratio2.txt"),
    (r"1stFullCharge", r"Charge/B2 1st1.80V/B2_Im_ratio2.txt"),
    (r"2ndHalfDischarge", r"Charge/F4 2nd1.30V/F4_Im_ratio2.txt"),
    (r"2ndFullDischarge", r"Charge/G1 2nd0.9V/G1_Im_ratio2.txt"),
]

# 加载和处理数据
data_dict = {name: read_data(path_data.joinpath(path), name, *cutoff) for name, path in datasets}

In [ ]:
# TXMs 的数据分析
averages = {}
averages_nonzero = {}
for key, df in data_dict.items():
    values = df.values.astype(float).ravel()
    averages[key] = float(values.mean())
    nonzero_values = values[values != 0]
    averages_nonzero[key] = float(nonzero_values.mean()) if nonzero_values.size > 0 else 0.0

# 对比分析：包含0值 vs 排除0值的平均值
print("\n" + "=" * 80)
print("对比分析：包含0值 vs 排除0值的平均值差异")
print("=" * 80)
print(f"{'数据集':<20} {'包含0值':<12} {'排除0值':<12} {'差异倍数':<10} {'零值比例'}")
print("-" * 80)

comparison_data = []
for key in data_dict.keys():
    avg_with_zero = averages[key]
    avg_without_zero = averages_nonzero[key]

    if avg_with_zero > 0:
        ratio = avg_without_zero / avg_with_zero
    else:
        ratio = float("inf") if avg_without_zero > 0 else 1.0

    # 计算零值比例
    total_points = data_dict[key].size
    zero_points = total_points - len(data_dict[key][data_dict[key] != 0])
    zero_percentage = (zero_points / total_points) * 100

    comparison_data.append((key, avg_with_zero, avg_without_zero, ratio, zero_percentage))

    print(f"{key:<20} {avg_with_zero:<12.6f} {avg_without_zero:<12.6f} {ratio:<10.1f}x {zero_percentage:<6.1f}%")

# 按排除0值后的平均值排序
sorted_nonzero = sorted(averages_nonzero.items(), key=lambda x: x[1], reverse=True)
print(r"按排除0值后的平均值从高到低排序:")
for i, (key, value) in enumerate(sorted_nonzero, 1):
    print(f"{i:2d}. {key:<20}: {value:.6f}")


In [ ]:
def create_subplot(subfig_pos, ax_pos, data_key, title, cax_pos, caption):
    """创建标准化子图"""
    subfig = fig.add_subfigure(subfig_pos)
    ax = subfig.add_axes(ax_pos)
    ax.set_box_aspect(1)

    data = data_dict[data_key]
    im = ax.imshow(data, vmin=data.min().min(), vmax=data.max().max(), cmap="hot", interpolation="bilinear")

    # 添加比例尺
    add_sizebar(ax, 2, scale, "w", r"$\mu m$").set_bbox_to_anchor(Bbox.from_bounds(0, 0.03, 0, 0), ax.transAxes)

    # 添加颜色条
    if cax_pos:
        cax = subfig.add_axes(cax_pos)
        cbar = subfig.colorbar(im, cax=cax, ticks=np.linspace(0, 1, 21), format="%.2f")
        cbar.ax.set_ylim(cutoff[0], cutoff[1])

    # 添加标题
    ax.set_title(title, loc="left", fontdict={"fontsize": 8, "fontfamily": "Arial"}, transform=ax.transAxes, y=1.0, color=colors[1])
    ax.axis("off")

    # labels
    if caption:
        ax.text(
            -0.05,
            1.05,
            f"{caption}",
            transform=ax.transAxes,
            fontsize=8,
            va="center",
            ha="right",
            fontfamily="Arial",
            fontweight="bold",
        )
    return ax

In [ ]:
# 读取 EELS L2/L3 的数据

PLOT_INDEX_CSV = path_data_root / r"TEM\ExSitu\αMnO2\Mn_L3_L2_Mapping\Ploting_Index.csv"
plot_idx = pd.read_csv(PLOT_INDEX_CSV, encoding="utf-8-sig")

signals = []
for _, row in plot_idx.iterrows():
    p = Path(str(row["resolved_output_xlsx"]))
    if not p.exists():
        continue

    df = pd.read_excel(p, sheet_name="L3L2_map_maskFalse")
    arr = df.drop(columns=["y"], errors="ignore").to_numpy(dtype=float)

    # 先做阈值清洗
    arr[(arr < 1.5) | (arr > 3.0)] = np.nan
    # 再过滤孤岛散点
    arr, removed_px = remove_small_islands(arr, min_component_pixels=3)

    s = hs.signals.Signal2D(arr)
    s.axes_manager[0].name = "y"
    s.axes_manager[1].name = "x"
    s.axes_manager[0].units = "nm"
    s.axes_manager[1].units = "nm"
    s.axes_manager[0].scale = float(row.get("scale_y", row["scale_x"]))
    s.axes_manager[1].scale = float(row["scale_x"])
    s.metadata.General.title = f"{row['name']} | removed={removed_px}"

    signals.append(s)

In [ ]:
# TEM-EDS 的比例
path_fc_ratio = path_data_root / r"TEM\ExSitu\αMnO2\TEM-EDS-EELS-all\TEM-EDX_Bulk_Surface.xlsx"
fc_ratio = pd.read_excel(path_fc_ratio, sheet_name=1, header=0, index_col=None).dropna(axis=1, how="all")
fc_ratio["Samples"] = fc_ratio["Samples"].astype(str).str.strip()
fc_ratio["Position"] = fc_ratio["Position"].astype(str).str.strip()
fc_ratio["Mn/O"] = pd.to_numeric(fc_ratio["Mn/O"], errors="coerce")

# TEM-EELS 的比例
path_fc_ratio_eels = path_data_root / r"TEM\ExSitu\αMnO2\TEM-EDS-EELS-all\TEM-EELS_Bulk_Surface_NewPhase.xlsx"
fc_ratio_eels = pd.read_excel(path_fc_ratio_eels, sheet_name=1, header=0, index_col=None).dropna(axis=1, how="all")
fc_ratio_eels["Samples"] = fc_ratio_eels["Samples"].astype(str).str.strip()
fc_ratio_eels["Position"] = fc_ratio_eels["Position"].astype(str).str.strip()

# 统一 EDX/EELS 样品命名，确保图 M 中各序列一一对应
# EDX 原始名: FD#1, HC#1, HC#2, FC#1, HD#1, FD#2
# EELS 原始名: FD1st, HC#A, HC#B, FC#C, HD#D, FD#E
sample_name_map = {
    "FD#1": "FD#A",
    "HC#1": "HC#B",
    "HC#2": "HC#C",
    "FC#1": "FC#D",
    "HD#1": "HD#E",
    "FD#2": "FD#F",
    "FD1st": "FD#A",
    "HC#A": "HC#B",
    "HC#B": "HC#C",
    "FC#C": "FC#D",
    "HD#D": "HD#E",
    "FD#E": "FD#F",
}
eels_sample_order = ["Pristine", "FD#A", "HC#B", "HC#C", "FC#D", "HD#E", "FD#F"]
eels_position_order = ["Bulk", "Surface", "NewPhase"]

fc_ratio["Zn/Mn"] = fc_ratio["Zn/O"] / fc_ratio["Mn/O"]
fc_ratio["Sample"] = fc_ratio["Samples"].replace(sample_name_map)
fc_ratio = fc_ratio[fc_ratio["Sample"].isin(eels_sample_order) & fc_ratio["Position"].isin(["Bulk", "Surface"])].copy()

fc_ratio_eels["Sample"] = fc_ratio_eels["Samples"].replace(sample_name_map)
fc_ratio_eels = fc_ratio_eels[
    fc_ratio_eels["Sample"].isin(eels_sample_order) & fc_ratio_eels["Position"].isin(eels_position_order)
].copy()
fc_ratio_eels["Sample"] = pd.Categorical(fc_ratio_eels["Sample"], categories=eels_sample_order, ordered=True)
fc_ratio_eels["Position"] = pd.Categorical(fc_ratio_eels["Position"], categories=eels_position_order, ordered=True)


fc_ratio["Zn/O"] = pd.to_numeric(fc_ratio["Zn/O"], errors="coerce")
fc_ratio["Zn/Mn"] = pd.to_numeric(fc_ratio["Zn/Mn"], errors="coerce")
fc_ratio_eels["Zn/O"] = pd.to_numeric(fc_ratio_eels["Zn/O"], errors="coerce")
fc_ratio_eels["Zn/Mn"] = pd.to_numeric(fc_ratio_eels["Zn/Mn"], errors="coerce")

In [ ]:
# 画图
plt.close("all")

fig = plt.figure(figsize=(7.0, 5.0), dpi=300)
layout = [
    ["A", "A", "B", "B"],
    ["C", "D", "E", "E"],
]
axs = fig.subplot_mosaic(
    layout,
    gridspec_kw={
        "width_ratios": [1.0, 1.0, 1.0, 1.0],
        "height_ratios": [1.0, 1.0],
        "wspace": 0.0,
        "hspace": 0.0,
    },
)

# 图 A
ax_a = axs["A"]
pos = ax_a.get_position()
ax_a.set_position([pos.x0, pos.y0, pos.width*1.165, pos.height*1.165])
txm_rect = ax_a.get_position().bounds
ax_a.set_axis_off()

txm_grid = ImageGrid(
    fig,
    txm_rect,
    nrows_ncols=(2, 3),
    axes_pad=0.15,
    share_all=False,
    label_mode="L",
    cbar_mode="single",
    cbar_location="right",
)

txm_panels = [
    ("1stDischarge", r"FD#A"),
    ("1stHalfCharge#1", r"HC#B"),
    ("1stHalfCharge#2", r"HC#C"),
    ("1stFullCharge", r"FC#D"),
    ("2ndHalfDischarge", r"HD#E"),
    ("2ndFullDischarge", r"FD#F"),
]

txm_norm = mpl.colors.Normalize(vmin=0.2, vmax=0.6)
for i, (data_key, title) in enumerate(txm_panels):
    ax = txm_grid[i]
    last_txm_im = ax.imshow(data_dict[data_key], cmap="hot", norm=txm_norm, interpolation="bilinear")
    if i == 0:
        add_sizebar(ax, 2, scale, "w", r"$\mu m$").set_bbox_to_anchor(Bbox.from_bounds(0, 0.01, 0, 0), ax.transAxes)   
        ax.text(-0.05, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

    ax.set_title(title, loc="left", fontsize=8, fontfamily="Arial", color=colors[1], pad=2)
    ax.axis("off")

if last_txm_im is not None:
    cbar_txm = txm_grid.cbar_axes[0].colorbar(last_txm_im)
    cbar_txm.set_label("Shape Index", fontsize=8, labelpad=4.0)
    cbar_txm.set_ticks(np.arange(0.2, 0.61, 0.1))
    cbar_txm.ax.tick_params(labelsize=8)
    cax = txm_grid.cbar_axes[0]
    pos = cax.get_position()
    cax.set_axes_locator(None)
    cax.set_position([pos.x0+0.55, pos.y0 + pos.height * 0.2, 0.018, pos.height * 0.70])


# 图 B
ax_b = axs["B"]
b_spec = ax_b.get_subplotspec()
ax_b.set_axis_off()

gs_b_outer = b_spec.subgridspec(2, 1, height_ratios=[1.0, 1.0], hspace=0.0, wspace=0.0)

norm_l3 = mpl.colors.Normalize(vmin=1.6, vmax=2.8)
cmap_l3 = mpl.colormaps["hot"].copy()
cmap_l3.set_bad("black")

l3_items = [
    (0, "Pristine"),
    (1, "FD#A"),
    (2, "HC#B"),
    (3, "HC#C"),
    (4, "FC#D"),
    (6, "HD#E"),
    (7, "FD#F"),
]


def _l3_arr(sig_idx):
    arr = signals[sig_idx].data
    if sig_idx in (0, 1, 7):
        arr = arr.T
    if sig_idx == 4:
        arr = arr[:78, :]
    if sig_idx == 6:
        arr = arr[:58, :76]
    return arr

arr_row1 = [_l3_arr(sig) for sig, _ in l3_items[:4]]
arr_row2 = [_l3_arr(sig) for sig, _ in l3_items[4:]]

wr_row1 = [max(arr.shape[1], 1) / max(arr.shape[0], 1) for arr in arr_row1]
wr_row2 = [max(arr.shape[1], 1) / max(arr.shape[0], 1) for arr in arr_row2]

gs_b_row1 = gs_b_outer[0].subgridspec(1, 4, width_ratios=wr_row1, wspace=0.03)
gs_b_row2 = gs_b_outer[1].subgridspec(1, 3, width_ratios=wr_row2, wspace=0.03)

b_axes = [
    fig.add_subplot(gs_b_row1[0, 0]),
    fig.add_subplot(gs_b_row1[0, 1]),
    fig.add_subplot(gs_b_row1[0, 2]),
    fig.add_subplot(gs_b_row1[0, 3]),
    fig.add_subplot(gs_b_row2[0, 0]),
    fig.add_subplot(gs_b_row2[0, 1]),
    fig.add_subplot(gs_b_row2[0, 2]),
]

shift_x=[0.17, 0.17, 0.17, 0.17, 0.17, 0.17, 0.17]
shift_y=[0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05]
shift_title=[0.22, 0.3, 0.12, 0.2, 0.1, 0.1, 0.25]
last_l3_im = None
for i, ((sig_idx, title), ax) in enumerate(zip(l3_items, b_axes)):
    arr = _l3_arr(sig_idx)
    pos = ax.get_position()
    ax.set_position([pos.x0+shift_x[i], pos.y0+shift_y[i], pos.width, pos.height])
    last_l3_im = ax.imshow(arr, cmap=cmap_l3, norm=norm_l3, interpolation="bilinear", aspect=1.0)
    add_sizebar(ax, size=plot_idx[r"recommended_scalebar_len"][sig_idx], bardata=signals[sig_idx], color="w").set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.01, 0, 0), ax.transAxes)
    ax.axis("off")
    ax.text(shift_title[i], 1.0, title, transform=ax.transAxes, fontsize=8, va="bottom", ha="center", fontfamily="Arial")
    if i == 0:
        ax.text(-0.15, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

if last_l3_im is not None:
    cax = ax_b.inset_axes([1.35, 0.25, 0.04, 0.8])
    cbar_eels = fig.colorbar(last_l3_im, cax=cax)
    cbar_eels.set_label("L3/L2 ratio", fontsize=8, labelpad=4.0)
    cbar_eels.set_ticks(np.arange(1.6, 2.81, 0.3))
    cbar_eels.ax.tick_params(labelsize=8)

# 图 C
ax_c = axs["C"]
pos = ax_c.get_position()
ax_c.set_position([pos.x0+0.02, pos.y0, pos.width, pos.height])
ax_c.set_box_aspect(1.5)

line_paras = [
    (0.29, 0.1, 0.95, "k"),
    (0.74, 0.1, 0.25, "k"),
    (0.32, 0.4, 0.5, "w"),
    (0.39, 0.45, 0.55, "w"),
    (0.45, 0.6, 0.7, "w"),
    (0.44, 0.68, 0.78, "k"),
    (0.42, 0.75, 0.85, "w"),
    (0.35, 0.88, 0.95, "w"),
]
hist_keys = [r"Pristine", r"MnOOH", r"1stDischarge", r"1stHalfCharge#1", r"1stHalfCharge#2", r"1stFullCharge", r"2ndHalfDischarge", r"2ndFullDischarge"]
hist_names = [r"Pristine", r"MnOOH", r"FD#A", r"HC#B", r"HC#C", r"FC#D", r"HD#E", r"FD#F"]

for i, key in enumerate(hist_keys):
    value = data_dict[key]
    color_i = colors[i % len(colors)]
    label_i = hist_names[i]
    if key in [r"Pristine", r"MnOOH"]:
        ax_c.hist(value.values.flatten(), bins=100, density=True, color=color_i, align="mid", range=(0.2, 1.0), zorder=0, label=label_i, bottom=0, alpha=1.0, stacked=True)
        ax_c.axvline(x=line_paras[i][0], ymax=line_paras[i][1], ymin=line_paras[i][2], color=line_paras[i][3], linestyle="--", linewidth=1.0, zorder=10)
    else:
        ax_c.hist(value.values.flatten(), bins=100, density=True, color=color_i, align="mid", range=cutoff, zorder=len(hist_keys) - i + 1, label=label_i, bottom=2.0 + 3.0 * i, alpha=1.0)
        ax_c.axvline(x=line_paras[i][0], ymax=line_paras[i][1], ymin=line_paras[i][2], color=line_paras[i][3], linestyle="--", linewidth=1.0, zorder=10)

ax_c.set_ylabel(r"Normalized Frequency", fontsize=9)
ax_c.set_xlabel(r"Shape Index", fontsize=9)
ax_c.set_xlim(0.2, 0.9)
ax_c.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax_c.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax_c.tick_params(axis="x", labelsize=8)
ax_c.tick_params(axis="y", which="both", left=False, labelleft=False)
ax_c.legend(loc="upper right", ncols=1, frameon=False, fontsize=8, handletextpad=0.18, labelspacing=0.08, borderpad=0.03, columnspacing=0.8)
ax_c.text(-0.12, 1.0, r"C", transform=ax_c.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

de_width_scale = 1.2
de_box_aspect = 1.5 / de_width_scale

# 图 D
ax_d = axs["D"]
pos = ax_d.get_position()
ax_d.set_position([pos.x0+0.13, pos.y0, pos.width * de_width_scale, pos.height])
ax_d.set_box_aspect(de_box_aspect)

x = np.arange(len(eels_sample_order), dtype=float)
x_map = {s: i for i, s in enumerate(eels_sample_order)}

bulk_edx_zo = fc_ratio[fc_ratio["Position"] == "Bulk"].groupby("Sample", observed=True)["Zn/O"].mean().reindex(eels_sample_order)
surface_edx_zo = fc_ratio[fc_ratio["Position"] == "Surface"].groupby("Sample", observed=True)["Zn/O"].mean().reindex(eels_sample_order)

(line1,) = ax_d.plot(x, bulk_edx_zo.to_numpy(dtype=float), c=colors[0], lw=1.0, ls="-", marker="o", ms=4.5, mec=colors[0], mfc=colors[0], label="Bulk")
(line2,) = ax_d.plot(x, surface_edx_zo.to_numpy(dtype=float), c=colors[0], lw=1.0, ls="-", marker="D", ms=4.5, mec=colors[0], mfc=colors[0], alpha=0.9, label="Surface")

ax_d.set_ylabel(r"EDX Zn/O Relative Ratio (arb. u.)", fontsize=9, color=colors[0])
ax_d.set_ylim(-0.05, 1.2)
ax_d.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2))
ax_d.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1))
ax_d.tick_params(axis="y", labelsize=8, colors=colors[0])
ax_d.set_xticks(x)
ax_d.set_xticklabels(eels_sample_order, rotation=55, ha="right", rotation_mode="anchor", fontfamily="Arial", fontsize=8)

ax2 = ax_d.twinx()
pos = ax2.get_position()
ax2.set_position([pos.x0+0.13, pos.y0, pos.width * de_width_scale, pos.height])
ax2.set_box_aspect(de_box_aspect)

bulk_eels_zo = fc_ratio_eels[fc_ratio_eels["Position"] == "Bulk"].groupby("Sample", observed=True)["Zn/O"].mean().reindex(eels_sample_order)
surface_eels_zo = fc_ratio_eels[fc_ratio_eels["Position"] == "Surface"].groupby("Sample", observed=True)["Zn/O"].mean().reindex(eels_sample_order)

(line3,) = ax2.plot(x, bulk_eels_zo.to_numpy(dtype=float), c=colors[1], lw=1.0, ls="--", marker="o", ms=4.5, mfc="none", mec=colors[1], label="Bulk")
(line4,) = ax2.plot(x, surface_eels_zo.to_numpy(dtype=float), c=colors[1], lw=1.0, ls="--", marker="D", ms=4.5, mfc="none", mec=colors[1], alpha=0.9, label="Surface")

sub_np_zo = fc_ratio_eels[fc_ratio_eels["Position"] == "NewPhase"].copy()
sub_np_zo["x"] = sub_np_zo["Sample"].astype(str).map(x_map)
sub_np_zo = sub_np_zo.dropna(subset=["x", "Zn/O"])
star1 = None
if not sub_np_zo.empty:
    star1 = ax2.scatter(sub_np_zo["x"].to_numpy(dtype=float), sub_np_zo["Zn/O"].to_numpy(dtype=float), marker="*", s=40, c=colors[1], linewidths=0.4, edgecolors="w", zorder=6, label="NewPhase")

ax2.set_ylabel(r"EELS Zn/O Relative Ratio (arb. u.)", fontsize=9, color=colors[1])
ax2.set_ylim(-0.05, 0.4)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.1))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.05))
ax2.tick_params(axis="y", labelsize=8, colors=colors[1])

legend_handles = [line1, line2, line3, line4]
if star1 is not None:
    legend_handles.append(star1)
ax_d.legend(handles=legend_handles, loc="upper left", fontsize=8, frameon=False, ncol=2, bbox_to_anchor=(0.02, 1.0), columnspacing=0.6, handlelength=1.5)
ax_d.text(-0.25, 1.0, r"D", transform=ax_d.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 E
ax_e = axs["E"]
pos = ax_e.get_position()
ax_e.set_position([pos.x0+0.23, pos.y0, pos.width, pos.height])
ax_e.set_box_aspect(de_box_aspect)

bulk_edx = fc_ratio[fc_ratio["Position"] == "Bulk"].groupby("Sample", observed=True)["Zn/Mn"].mean().reindex(eels_sample_order)
surface_edx = fc_ratio[fc_ratio["Position"] == "Surface"].groupby("Sample", observed=True)["Zn/Mn"].mean().reindex(eels_sample_order)
surface_edx.iloc[:2] = np.nan

(line1,) = ax_e.plot(x, bulk_edx.to_numpy(dtype=float), c=colors[0], lw=1.0, ls="-", marker="o", ms=4.5, mec=colors[0], mfc=colors[0], label="Bulk")
(line2,) = ax_e.plot(x, surface_edx.to_numpy(dtype=float), c=colors[0], lw=1.0, ls="-", marker="D", ms=4.5, mec=colors[0], mfc=colors[0], alpha=0.9, label="Surface")

ax_e.set_ylabel(r"EDX Zn/Mn Relative Ratio (arb. u.)", fontsize=9, color=colors[0])
ax_e.set_ylim(-0.05, 3.0)
ax_e.yaxis.set_major_locator(ticker.MultipleLocator(base=0.6))
ax_e.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.3))
ax_e.tick_params(axis="y", labelsize=8, colors=colors[0])
ax_e.set_xticks(x)
ax_e.set_xticklabels(eels_sample_order, rotation=55, ha="right", rotation_mode="anchor", fontfamily="Arial", fontsize=8)

ax2 = ax_e.twinx()
pos = ax2.get_position()
ax2.set_position([pos.x0+0.23, pos.y0, pos.width, pos.height])
ax2.set_box_aspect(de_box_aspect)

bulk_eels = fc_ratio_eels[fc_ratio_eels["Position"] == "Bulk"].groupby("Sample", observed=True)["Zn/Mn"].mean().reindex(eels_sample_order)
surface_eels = fc_ratio_eels[fc_ratio_eels["Position"] == "Surface"].groupby("Sample", observed=True)["Zn/Mn"].mean().reindex(eels_sample_order)
surface_eels.iloc[-1] = np.nan

(line3,) = ax2.plot(x, bulk_eels.to_numpy(dtype=float), c=colors[1], lw=1.0, ls="--", marker="o", ms=4.5, mfc="none", mec=colors[1], label="Bulk")
(line4,) = ax2.plot(x, surface_eels.to_numpy(dtype=float), c=colors[1], lw=1.0, ls="--", marker="D", ms=4.5, mfc="none", mec=colors[1], alpha=0.9, label="Surface")

sub_np = fc_ratio_eels[fc_ratio_eels["Position"] == "NewPhase"].copy()
sub_np["x"] = sub_np["Sample"].astype(str).map(x_map)
sub_np = sub_np.dropna(subset=["x", "Zn/Mn"])
star2 = None
if not sub_np.empty:
    star2 = ax2.scatter(sub_np["x"].to_numpy(dtype=float), sub_np["Zn/Mn"].to_numpy(dtype=float), marker="*", s=40, c=colors[1], linewidths=0.4, edgecolors="w", zorder=6, label="NewPhase")

ax2.set_ylabel(r"EELS Zn/Mn Relative Ratio (arb. u.)", fontsize=9, color=colors[1])
ax2.set_ylim(-0.05, 4)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=1.0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax2.tick_params(axis="y", labelsize=8, colors=colors[1], right=True, labelright=True)

legend_handles = [line1, line2, line3, line4]
if star2 is not None:
    legend_handles.append(star2)
ax_e.legend(handles=legend_handles, loc="upper left", fontsize=8, frameon=False, ncol=2, bbox_to_anchor=(0.02, 1.0), columnspacing=0.6, handlelength=1.5)
ax_e.text(-0.25, 1.0, r"E", transform=ax_e.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# # 保存图片
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_03_V3_300.tif"), dpi=300, bbox_inches="tight", pad_inches=0.05, pil_kwargs={"compression": "tiff_lzw"})
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_03_V3_600.tif"), dpi=600, bbox_inches="tight", pad_inches=0.05, pil_kwargs={"compression": "tiff_lzw"})
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_03_V3_600.png"), dpi=600, bbox_inches="tight", pad_inches=0.05)
# plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_03_V3_900.svg"), dpi=900, bbox_inches="tight", pad_inches=0.05)

plt.gcf().set_facecolor("white")
plt.show()


## FigureMS 02 -- 原位 Zn-Mn 的演化过程 -V2

In [ ]:
# 读取数据

# Operando XRD, EMD, from No pH Buffer, 1 M + 0.2 M, 30 uA

# 1) 读取电化学数据
path_filelist = list(Path.joinpath(path_data_root, r"XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Echem\B").glob(r"**\*.txt"))
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = pd.to_datetime(df["time/s"], format="mixed", errors="coerce")
    df["cycle number"] = pd.to_numeric(df["cycle number"], errors="coerce").astype("Int64")
    df["Voltage/V"] = df["Ewe/V"] - df["Ece/V"]
    echem_all.append(df)

echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 2) 读取谱线时间戳（按当前固定格式：Wave / Date / filename）
path_xrd = Path.joinpath(path_data_root, r"XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Data")
path_time_spectrum = path_xrd.joinpath(r"Time_index_spectrum.xlsx")

spectrum_time_all = pd.read_excel(path_time_spectrum)
required_cols = {"Wave", "Date", "filename"}
missing_cols = required_cols.difference(set(spectrum_time_all.columns))

spectrum_time_all["spec_time"] = pd.to_datetime(
    spectrum_time_all["Date"],
    format=r"%Y-%m-%d_%H:%M:%S",
    errors="coerce",
)

spectrum_time_all["spectrum_col"] = spectrum_time_all["filename"].astype(str).str.strip().str.replace(r"\.xye$", "", regex=True)
spectrum_time_all = spectrum_time_all.dropna(subset=["spec_time", "spectrum_col"]).sort_values("spec_time").reset_index(drop=True)

# 3) 读取 XRD 全谱数据
path_dspacing = path_xrd.joinpath(r"spectrum_all_d_spacing.csv")

spectrum_all = pd.read_csv(path_dspacing, index_col=0, header=0)
spectrum_all.index = pd.to_numeric(spectrum_all.index, errors="coerce")
spectrum_all = spectrum_all.loc[~spectrum_all.index.isna(), :]
spectrum_all.columns = [str(c).strip().replace(".xye", "") for c in spectrum_all.columns]

In [ ]:
# 过滤 17_Da_* 谱线并与时间索引对齐（后续单元直接使用）

# 1) 时间表仅保留 17_Da_fxxxxx
spectrum_time_all["spectrum_col"] = spectrum_time_all["spectrum_col"].astype(str).str.strip().str.replace(r"\.xye$", "", regex=True)
mask_da = spectrum_time_all["spectrum_col"].str.match(r"^17_Da_f\d+$", na=False)
spectrum_time_all = spectrum_time_all.loc[mask_da].copy().sort_values("spec_time").reset_index(drop=True)

# 2) 谱图矩阵仅保留 17_Da_fxxxxx 列
spec_cols_da = [c for c in spectrum_all.columns if c.startswith("17_Da_f")]
spectrum_all = spectrum_all.loc[:, spec_cols_da].copy()

# 3) 取交集并按时间顺序重排，确保 time/spectrum 完全对齐
common_cols = [c for c in spectrum_time_all["spectrum_col"].tolist() if c in spectrum_all.columns]
spectrum_time_all = spectrum_time_all[spectrum_time_all["spectrum_col"].isin(common_cols)].copy()
spectrum_time_all = spectrum_time_all.drop_duplicates(subset=["spectrum_col"], keep="first").sort_values("spec_time").reset_index(drop=True)
spectrum_all = spectrum_all.reindex(columns=spectrum_time_all["spectrum_col"].tolist())

# 4) 清洗电化学数据并建立 charge_time
echem_all.drop(echem_all.index[:6500], inplace=True)
selected_echem = echem_all[echem_all["cycle number"].isin([1, 2])]
selected_echem = selected_echem[selected_echem["Voltage/V"] >= 0.8].copy()
selected_echem = selected_echem.dropna(subset=["time/s", "Voltage/V", "<I>/mA"]).sort_values(by="time/s").reset_index(drop=True)
selected_echem["charge_time"] = (selected_echem["time/s"] - selected_echem["time/s"].iloc[0]).dt.total_seconds() / 3600

# 5) 最近邻时间匹配：谱线 -> 电化学
selected_spectrum_time = (
    pd
    .merge_asof(
        spectrum_time_all[["spectrum_col", "spec_time"]].sort_values(by="spec_time"),
        selected_echem[["time/s", "Voltage/V", "<I>/mA", "charge_time"]].sort_values(by="time/s"),
        left_on="spec_time",
        right_on="time/s",
        direction="nearest",
        tolerance=pd.Timedelta("5s"),
    )
    .dropna(subset=["spectrum_col", "spec_time", "time/s", "Voltage/V", "charge_time"], inplace=False)
    .sort_values(by="spec_time")
    .reset_index(drop=True)
)

# 6) 衍生时间与索引列（新版）
time_zero = selected_echem["time/s"].iloc[0]
selected_spectrum_time["spec_charge_time"] = (selected_spectrum_time["spec_time"] - time_zero).dt.total_seconds() / 3600
selected_spectrum_time["spec_time_h"] = selected_spectrum_time["spec_charge_time"]
selected_spectrum_time["map_dt_s"] = (selected_spectrum_time["time/s"] - selected_spectrum_time["spec_time"]).dt.total_seconds().abs()
selected_spectrum_time["range_idx"] = selected_spectrum_time["spectrum_col"].str.extract(r"(\d+)$", expand=False).astype("Int64")

# 7) 选择谱线区间并建立绘图矩阵
d_spacing_range = (0.5, 15.0)
spectrum_all = spectrum_all.loc[(spectrum_all.index >= d_spacing_range[0]) & (spectrum_all.index <= d_spacing_range[1])].copy()

selected_spectrum_time = selected_spectrum_time.dropna(subset=["range_idx"]).copy()
selected_spectrum_time = selected_spectrum_time.drop_duplicates(subset=["spectrum_col"], keep="first").sort_values(by="spec_time").reset_index(drop=True)

scan_cols_plot = selected_spectrum_time["spectrum_col"].astype(str).tolist()
selected_spectrum_plot = spectrum_all.reindex(columns=scan_cols_plot).copy()

# 统一时间显示窗口：覆盖电化学与谱线时间，并上下留 15 min
spec_time_arr = selected_spectrum_time["spec_time_h"].to_numpy(dtype=float)
pad_h = 0.25
time_min = min(float(selected_echem["charge_time"].min()), float(spec_time_arr.min()))
time_max = max(float(selected_echem["charge_time"].max()), float(spec_time_arr.max()))
y_min_plot = time_min - pad_h
y_max_plot = time_max + pad_h

In [ ]:
# 读取数据, Mn

path_cell2_mn = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
path_file_mn = Path.joinpath(path_cell2_mn, r"FC1st-FD2nd\Oct2022_cell2_P2b\XANES\Mn\PCA_Larch_Mn")
xas_Mn_path_1 = Path.joinpath(path_file_mn, r"lcf_alpha_MnO2_electrode_pristine_residual_1.nor")
xas_Mn_path_2 = Path.joinpath(path_file_mn, r"lcf_alpha_MnO2_electrode_pristine_residual_2.nor")

# 电化学时间序列
path_filelist_mn = list(Path.joinpath(path_cell2_mn, r"Echem").glob(r"**\*.txt"))
echem_all_mn = []
for path_file_echem_mn in path_filelist_mn:
    with open(path_file_echem_mn, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip_mn = int(line.split(":")[1].strip())
                break

    df_mn = pd.read_csv(path_file_echem_mn, sep="\t", comment="#", skiprows=line_skip_mn - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df_mn[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df_mn[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df_mn["time/s"] = df_mn["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df_mn["cycle number"] = df_mn["cycle number"].astype(float).astype(np.int16)
    echem_all_mn.append(df_mn)

echem_all_mn = pd.concat(echem_all_mn, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 当前对齐后的 Mn operando 数据
xas_Mn_1 = pd.read_csv(
    xas_Mn_path_1,
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Mn_2 = pd.read_csv(
    xas_Mn_path_2,
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)

# 两段 Mn residual .nor 的 energy 网格略有差异
energy_Mn = xas_Mn_1.iloc[:, 0].to_numpy(dtype=float)
energy_Mn_2 = xas_Mn_2.iloc[:, 0].to_numpy(dtype=float)
energy_mask_Mn = (energy_Mn >= float(np.nanmin(energy_Mn_2))) & (energy_Mn <= float(np.nanmax(energy_Mn_2)))
xas_Mn = xas_Mn_1.loc[energy_mask_Mn].reset_index(drop=True)
energy_Mn_common = xas_Mn.iloc[:, 0].to_numpy(dtype=float)
for col in range(1, xas_Mn_2.shape[1]):
    xas_Mn[xas_Mn.shape[1]] = np.interp(
        energy_Mn_common,
        energy_Mn_2,
        xas_Mn_2.iloc[:, col].to_numpy(dtype=float),
    )

pdf_Mn = xas_Mn.iloc[:, [0, 1, 2]]  # noqa: N816
pdf_Mn.columns = [r"Energy", r"0.2M_MnSO4(aq.)", r"alpha-MnO2"]
opData_Mn = xas_Mn.iloc[:, [0, *list(range(3, xas_Mn.shape[1]))]]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))


In [ ]:
# 聚焦当前对齐后 Mn 数据
opData_Mn = opData_Mn[opData_Mn.iloc[:, 0].between(6530.0, 6610.0)].copy()

# 保留原来标记为异常的谱线，并移除最后一条 Mn 谱线
bad_spec_cols_mn = [47]
bad_spec_cols_mn = [idx for idx in bad_spec_cols_mn if 1 <= idx < opData_Mn.shape[1]]
if bad_spec_cols_mn:
    opData_Mn.iloc[:, bad_spec_cols_mn] = np.nan



In [ ]:
n_spec_mn = opData_Mn.shape[1] - 1
special_idx_mn = np.array([0, 4, 9, 12, 14, 17, 24, 30, 34, 38], dtype=int)
special_idx_mn = special_idx_mn[(special_idx_mn >= 0) & (special_idx_mn < n_spec_mn)]

energy_mn = opData_Mn.iloc[:, 0].to_numpy(dtype=float)
mu_mat_mn = opData_Mn.iloc[:, 1:].to_numpy(dtype=float).T  # (n_spec_mn, n_energy_mn)

# 电化学窗口：首圈放电谷值到次圈充电峰值
echem_index_low_mn = echem_all_mn[echem_all_mn["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high_mn = echem_all_mn[echem_all_mn["cycle number"] == 1]["Ewe/V"].idxmax()
echem_seg_mn = echem_all_mn.iloc[int(echem_index_low_mn) : int(echem_index_high_mn) + 1].copy()
echem_seg_mn = echem_seg_mn.dropna(subset=["time/s", "Ewe/V", "<I>/mA"]).sort_values("time/s").reset_index(drop=True)

# 直接从当前 LCF .nor 表头读取列名，用于谱线到采谱时间的映射
def _read_nor_cols_mn(path_nor):
    cols = {}
    with open(path_nor, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.startswith("# Column."):
                if line.startswith("#------------------------"):
                    break
                continue
            left, right = line.split(":", 1)
            cols[int(left.split(".")[1])] = right.strip().lstrip("-").strip()
    return cols

cols_Mn_1 = _read_nor_cols_mn(xas_Mn_path_1)
cols_Mn_2 = _read_nor_cols_mn(xas_Mn_path_2)
op_labels_mn = [cols_Mn_1[i] for i in range(4, xas_Mn_1.shape[1] + 1) if i in cols_Mn_1 and re.search(r"P2b_(\d+)", cols_Mn_1[i])]
op_labels_mn.extend([cols_Mn_2[i] for i in range(2, xas_Mn_2.shape[1] + 1) if i in cols_Mn_2 and re.search(r"P2b_(\d+)", cols_Mn_2[i])])

if len(op_labels_mn) != n_spec_mn:
    raise ValueError(f"FigureSI 12A: op_labels_mn={len(op_labels_mn)} != n_spec_mn={n_spec_mn}")

map_df_mn = pd.read_csv(Path.joinpath(path_cell2_mn, r"Overview", r"Time_Index_Spectrum.csv"))
map_df_mn = map_df_mn[
    (map_df_mn["Folder"].astype(str).str.contains("cell2_P2b", case=False, na=False))
    & (map_df_mn["Element"].astype(str).str.upper() == "MN")
    & map_df_mn["NorName"].notna()
].copy()
map_df_mn["Time"] = pd.to_datetime(map_df_mn["Time"], errors="coerce")
map_df_mn["NorName"] = map_df_mn["NorName"].astype(str).str.strip().str.lstrip("-")
map_df_mn = map_df_mn.dropna(subset=["Time"]).sort_values("Time")
name_to_time_mn = map_df_mn.drop_duplicates(subset=["NorName"], keep="first").set_index("NorName")["Time"]

label_s_mn = pd.Series(op_labels_mn, dtype="string").str.strip().str.lstrip("-")
spec_time_dt_mn = label_s_mn.map(name_to_time_mn)
missing_labels_mn = label_s_mn[spec_time_dt_mn.isna()].tolist()
if missing_labels_mn:
    raise ValueError(f"FigureSI 12A: missing NorName->Time mapping for labels: {missing_labels_mn[:6]}")

# 只保留落在电化学窗口内的谱线，避免 np.interp 对窗口外谱线使用边界值
echem_start_mn = echem_seg_mn["time/s"].iloc[0]
echem_end_mn = echem_seg_mn["time/s"].iloc[-1]
in_echem_window_mn = spec_time_dt_mn.between(echem_start_mn, echem_end_mn).to_numpy(dtype=bool)
if not np.all(in_echem_window_mn):
    op_labels_mn = [label for label, keep in zip(op_labels_mn, in_echem_window_mn) if keep]
    spec_time_dt_mn = spec_time_dt_mn[in_echem_window_mn].reset_index(drop=True)
    mu_mat_mn = mu_mat_mn[in_echem_window_mn, :]
    n_spec_mn = mu_mat_mn.shape[0]
special_idx_mn = special_idx_mn[(special_idx_mn >= 0) & (special_idx_mn < n_spec_mn)]

# 统一时间零点，并把采谱点映射到电化学轨迹上
time_zero_mn = min(echem_seg_mn["time/s"].iloc[0], spec_time_dt_mn.min())
echem_time_mn = (echem_seg_mn["time/s"] - time_zero_mn).dt.total_seconds().to_numpy(dtype=float) / 3600.0
echem_voltage_mn = echem_seg_mn["Ewe/V"].to_numpy(dtype=float)
echem_current_ua_mn = echem_seg_mn["<I>/mA"].to_numpy(dtype=float) * 1000.0

spec_time_raw_mn = (spec_time_dt_mn - time_zero_mn).dt.total_seconds().to_numpy(dtype=float) / 3600.0
spec_time_mn = spec_time_raw_mn.copy()
spec_volt_mn = np.interp(spec_time_mn, echem_time_mn, echem_voltage_mn)
spec_curr_ua_mn = np.interp(spec_time_mn, echem_time_mn, echem_current_ua_mn)


In [ ]:
# 读取数据

path_cell2_zn = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
path_file_zn = Path.joinpath(path_cell2_zn, r"FC1st-FD2nd\Oct2022_cell2_P2b\XANES\Zn\PCA_Larch_Zn")
xas_Zn_path = Path.joinpath(path_file_zn, r"lcf_ZHS_residual.nor")

# 电化学时间序列
path_filelist_zn = list(Path.joinpath(path_cell2_zn, r"Echem").glob(r"**\*.txt"))
echem_all_zn = []
for path_file_echem_zn in path_filelist_zn:
    with open(path_file_echem_zn, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip_zn = int(line.split(":")[1].strip())
                break

    df_zn = pd.read_csv(path_file_echem_zn, sep="\t", comment="#", skiprows=line_skip_zn - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df_zn[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df_zn[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df_zn["time/s"] = df_zn["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df_zn["cycle number"] = df_zn["cycle number"].astype(float).astype(np.int16)
    echem_all_zn.append(df_zn)

echem_all_zn = pd.concat(echem_all_zn, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 当前对齐后的 Zn operando 数据
xas_Zn = pd.read_csv(
    xas_Zn_path,
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)

charge_index_zn = [0, 1, 2, *list(range(12, xas_Zn.shape[1], 1))]
xas_Zn = xas_Zn.iloc[:, charge_index_zn]  # noqa: N816

pdf_Zn = xas_Zn.iloc[:, [0, 1, 2]]  # noqa: N816
pdf_Zn.columns = [r"Energy", r"0.5M_ZnSO4(aq.)", r"ZHS"]
opData_Zn = xas_Zn.iloc[:, [0, *list(range(3, xas_Zn.shape[1]))]]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))


In [ ]:
# 聚焦当前对齐后 Zn 数据
opData_Zn = opData_Zn[opData_Zn.iloc[:, 0].between(9640.0, 9740.0)].copy()

# 保留原来标记为异常的两条谱线
bad_spec_cols_zn = [11, 36, 37, 38]
bad_spec_cols_zn = [idx for idx in bad_spec_cols_zn if idx < opData_Zn.shape[1]]
if bad_spec_cols_zn:
    opData_Zn.iloc[:, bad_spec_cols_zn] = np.nan


In [ ]:
n_spec_zn = opData_Zn.shape[1] - 1
special_idx_zn = np.array([0, 4, 9, 12, 14, 17, 24, 30, 34, 38], dtype=int)
special_idx_zn = special_idx_zn[(special_idx_zn >= 0) & (special_idx_zn < n_spec_zn)]

energy_zn = opData_Zn.iloc[:, 0].to_numpy(dtype=float)
mu_mat_zn = opData_Zn.iloc[:, 1:].to_numpy(dtype=float).T  # (n_spec_zn, n_energy_zn)

# 电化学窗口：首圈放电谷值到次圈充电峰值
echem_index_low_zn = echem_all_zn[echem_all_zn["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high_zn = echem_all_zn[echem_all_zn["cycle number"] == 1]["Ewe/V"].idxmax()
echem_seg_zn = echem_all_zn.iloc[int(echem_index_low_zn) : int(echem_index_high_zn) + 1].copy()
echem_seg_zn = echem_seg_zn.dropna(subset=["time/s", "Ewe/V", "<I>/mA"]).sort_values("time/s").reset_index(drop=True)

# 直接从当前 LCF .nor 表头读取列名，用于谱线到采谱时间的映射
def _read_nor_cols_zn(path_nor):
    cols = {}
    with open(path_nor, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.startswith("# Column."):
                if line.startswith("#------------------------"):
                    break
                continue
            left, right = line.split(":", 1)
            cols[int(left.split(".")[1])] = right.strip().lstrip("-").strip()
    return cols

cols_zn = _read_nor_cols_zn(xas_Zn_path)
op_source_cols_zn = charge_index_zn[3:]
op_labels_zn = [cols_zn[i + 1] for i in op_source_cols_zn if i + 1 in cols_zn and re.search(r"P2b_(\d+)", cols_zn[i + 1])]

if len(op_labels_zn) != n_spec_zn:
    raise ValueError(f"FigureSI 12A: op_labels_zn={len(op_labels_zn)} != n_spec_zn={n_spec_zn}")

map_df_zn = pd.read_csv(Path.joinpath(path_cell2_zn, r"Overview", r"Time_Index_Spectrum.csv"))
map_df_zn = map_df_zn[
    (map_df_zn["Folder"].astype(str).str.contains("cell2_P2b", case=False, na=False))
    & (map_df_zn["Element"].astype(str).str.upper() == "ZN")
    & map_df_zn["NorName"].notna()
].copy()
map_df_zn["Time"] = pd.to_datetime(map_df_zn["Time"], errors="coerce")
map_df_zn["NorName"] = map_df_zn["NorName"].astype(str).str.strip().str.lstrip("-")
map_df_zn = map_df_zn.dropna(subset=["Time"]).sort_values("Time")
name_to_time_zn = map_df_zn.drop_duplicates(subset=["NorName"], keep="first").set_index("NorName")["Time"]

label_s_zn = pd.Series(op_labels_zn, dtype="string").str.strip().str.lstrip("-")
spec_time_dt_zn = label_s_zn.map(name_to_time_zn)
missing_labels_zn = label_s_zn[spec_time_dt_zn.isna()].tolist()
if missing_labels_zn:
    raise ValueError(f"FigureSI 12A: missing NorName->Time mapping for labels: {missing_labels_zn[:6]}")

# 只保留落在电化学窗口内的谱线，避免 np.interp 对窗口外谱线使用边界值
echem_start_zn = echem_seg_zn["time/s"].iloc[0]
echem_end_zn = echem_seg_zn["time/s"].iloc[-1]
in_echem_window_zn = spec_time_dt_zn.between(echem_start_zn, echem_end_zn).to_numpy(dtype=bool)
if not np.all(in_echem_window_zn):
    op_labels_zn = [label for label, keep in zip(op_labels_zn, in_echem_window_zn) if keep]
    spec_time_dt_zn = spec_time_dt_zn[in_echem_window_zn].reset_index(drop=True)
    mu_mat_zn = mu_mat_zn[in_echem_window_zn, :]
    old_special_idx_zn = special_idx_zn.copy()
    kept_old_idx_zn = np.flatnonzero(in_echem_window_zn)
    special_idx_zn = np.array([np.where(kept_old_idx_zn == idx)[0][0] for idx in old_special_idx_zn if idx in kept_old_idx_zn], dtype=int)
    n_spec_zn = mu_mat_zn.shape[0]
special_idx_zn = np.unique(np.r_[special_idx_zn, n_spec_zn - 1]).astype(int)

# 统一时间零点，并把采谱点映射到电化学轨迹上
time_zero_zn = min(echem_seg_zn["time/s"].iloc[0], spec_time_dt_zn.min())
time_shift_zn_to_mn = (time_zero_zn - time_zero_mn).total_seconds() / 3600.0
echem_time_zn = (echem_seg_zn["time/s"] - time_zero_zn).dt.total_seconds().to_numpy(dtype=float) / 3600.0
echem_time_zn = echem_time_zn + time_shift_zn_to_mn
echem_voltage_zn = echem_seg_zn["Ewe/V"].to_numpy(dtype=float)
echem_current_ua_zn = echem_seg_zn["<I>/mA"].to_numpy(dtype=float) * 1000.0

spec_time_raw_zn = (spec_time_dt_zn - time_zero_zn).dt.total_seconds().to_numpy(dtype=float) / 3600.0
spec_time_zn = spec_time_raw_zn.copy() + time_shift_zn_to_mn
spec_volt_zn = np.interp(spec_time_zn, echem_time_zn, echem_voltage_zn)
spec_curr_ua_zn = np.interp(spec_time_zn, echem_time_zn, echem_current_ua_zn)


In [ ]:
# 画图
plt.close("all")

fig = plt.figure(figsize=(7.0, 5.0))
layout = [["A", "B", "C"], ["D", "E", "F"], ["D", "E", "F"]]
axs = fig.subplot_mosaic(
    layout,
    gridspec_kw={"width_ratios": [0.3, 1.0, 1.0], "height_ratios": [1.0, 1.0, 1.0], "wspace": 0.0, "hspace": 0.0},
)

# 图 A
ax = axs["A"]

ax.plot(selected_echem["Voltage/V"], selected_echem["charge_time"], ls="-", lw=1.0, c=colors[0], label=r"Voltage")

index_labels = [15, 25, 37, 46, 50, 61, 70, 75, 87, 99, 109, 115, 126, 137]
range_idx_int = pd.to_numeric(selected_spectrum_time["range_idx"], errors="coerce").astype("Int64")
special_mask = range_idx_int.isin(index_labels)

plot_points = selected_spectrum_time.copy()
plot_points["is_special"] = special_mask.to_numpy()
special_points = plot_points[plot_points["is_special"]].copy()

for _, row in plot_points.iterrows():
    if bool(row["is_special"]):
        ax.scatter(
            row["Voltage/V"],
            row["spec_time_h"],
            c=colors[0],
            edgecolors="face",
            marker="*",
            s=50,
            zorder=5,
        )
    else:
        ax.scatter(
            row["Voltage/V"],
            row["spec_time_h"],
            c="lightgrey",
            edgecolors="face",
            s=30,
            zorder=1,
        )

ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_xlim(0.8, 2.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.6, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.3, offset=0))

ax.set_ylabel(r"Duration Time (h)", fontsize=9)
ax.set_ylim(y_min_plot, y_max_plot)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))
ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, left=True, labelbottom=True, labelleft=True, labelsize=8)

ax2 = ax.twiny()
ax2.plot(selected_echem["<I>/mA"] * 1000, selected_echem["charge_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")
ax2.set_xlabel(r"$\mathrm{Current  \ (\mu A)}$", fontsize=9)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.set_ylim(y_min_plot, y_max_plot)
ax2.tick_params(axis="both", which="both", labelcolor="k", top=True, labeltop=True, labelsize=8)

ax.text(-0.45, 1.0, "A", transform=ax.transAxes, fontsize=10, fontweight="bold", va="center", ha="right")

# 图 B
spec_time_arr = selected_spectrum_time["spec_time_h"].to_numpy(dtype=float)
if selected_spectrum_plot.shape[1] != len(spec_time_arr):
    raise ValueError(f"spectrum/time length mismatch: {selected_spectrum_plot.shape[1]} vs {len(spec_time_arr)}")

d_axis = selected_spectrum_plot.index.to_numpy(dtype=float)
d_grid, t_grid = np.meshgrid(d_axis, spec_time_arr)

special_y_time = (
    selected_spectrum_time[selected_spectrum_time["range_idx"].isin(index_labels)]["spec_time_h"]
    .dropna()
    .astype(float)
    .tolist()
)

# 图 B
ax = axs["B"]
pos = ax.get_position()
ax.set_position([pos.x0-0.05, pos.y0, pos.width, pos.height])
ax.set_box_aspect(1.0)

pc_b = ax.pcolormesh(d_grid, t_grid, selected_spectrum_plot.T.to_numpy(dtype=float), cmap="coolwarm", shading="gouraud", vmin=6000, vmax=42000)
ax.set_yticks([])
ax.set_ylim(float(spec_time_arr.min()), float(spec_time_arr.max()))
ax.set_xlim(2.0, 8.0)
ax.set_xlabel(r"d Spacing ($\AA$)", fontsize=9)
ax.tick_params(axis="both", which="both", labelsize=8)

cax_b = ax.inset_axes((1.06, 0.1, 0.06, 0.8))
cb_b = fig.colorbar(pc_b, cax=cax_b)
cb_b.set_ticks([])
ax.text(1.1, 0.98, "High", transform=ax.transAxes, fontsize=8, va="top", ha="center")
ax.text(1.1, 0.02, "Low", transform=ax.transAxes, fontsize=8, va="bottom", ha="center")

ax.text(-0.03, 1.0, "B", transform=ax.transAxes, fontsize=10, fontweight="bold", va="center", ha="right")

# 图 C
base_c = axs["C"]
base_c.set_axis_off()
opxrd_ref_d_spacing = [1.414916, 2.44884, 5.967425, 8.042391, 2.71, 3.23, 5.47, 10.94]
peak_widths = [
    (0.05, 0.05),
    (0.05, 0.05),
    (0.4, 0.4),
    (0.4, 0.4),
    (0.1, 0.1),
    (0.1, 0.1),
    (0.1, 0.1),
    (0.4, 0.4),
]

# ========== 改动开始 ==========
# 固定每个子图的宽度和高度（相对于 figure 的比例）
fixed_width = 0.05      # 每个子图宽度占 figure 宽度的比例
fixed_height = 0.33      # 每个子图高度占 figure 高度的比例（可根据原图效果调整）
wspace = 0.005           # 子图之间的水平间距（比例）

n = len(opxrd_ref_d_spacing)
total_width = n * fixed_width + (n - 1) * wspace
pos_base = base_c.get_position()
# 调整 base_c 的宽度以容纳所有子图（左边界不变）
base_c.set_position([pos_base.x0, pos_base.y0, total_width, pos_base.height])

c_shift = -0.07       # 可选的额外右移量，保留原代码中的偏移
# ========== 改动结束 ==========

for i, d0 in enumerate(opxrd_ref_d_spacing):
    # ========== 改动开始：手动计算子图位置并创建 ==========
    x0 = pos_base.x0 + i * (fixed_width + wspace) + c_shift
    y0 = pos_base.y0    # 与 base_c 同高，也可微调
    axc = fig.add_axes([x0, y0, fixed_width, fixed_height])
    # ========== 改动结束 ==========
    
    temp = selected_spectrum_plot.loc[(selected_spectrum_plot.index >= d0 - peak_widths[i][0]) & 
                                      (selected_spectrum_plot.index <= d0 + peak_widths[i][1])]
    if temp.empty:
        axc.set_axis_off()
        continue
    td, tt = np.meshgrid(temp.index.to_numpy(dtype=float), spec_time_arr)
    axc.pcolormesh(td, tt, temp.T.to_numpy(dtype=float), cmap="coolwarm", shading="gouraud")
    axc.axvline(d0, ls="--", lw=0.8, color="grey", alpha=0.5)
    axc.yaxis.set_ticks([])
    axc.set_ylim(float(spec_time_arr.min()), float(spec_time_arr.max()))
    axc.set_xlim(float(temp.index.min()), float(temp.index.max()))
    axc.xaxis.set_major_locator(ticker.FixedLocator([d0]))
    axc.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    axc.spines[["top", "right", "left"]].set_visible(False)
    axc.tick_params(axis="both", which="both", labelsize=8)
    for y_sp in special_y_time:
        axc.axhline(y=y_sp, lw=0.6, ls="--", color=colors[1], alpha=0.45)

base_c.text(-0.17, 1.0, "C", transform=base_c.transAxes, fontsize=10, fontweight="bold", va="center", ha="right")

# 图 D
ax_echem =axs["D"]
pos = ax_echem.get_position()
ax_echem.set_position((pos.x0, pos.y0-0.2, pos.width, pos.height))
# ax_echem.set_box_aspect(3.0)

y_min = min(float(np.nanmin(echem_time_mn)), float(np.nanmin(spec_time_mn)), float(np.nanmin(spec_time_zn)))
y_max = max(float(np.nanmax(echem_time_mn)), float(np.nanmax(spec_time_mn)), float(np.nanmax(spec_time_zn)))

ax_echem.plot(echem_voltage_mn, echem_time_mn, c=colors[0], ls="-", lw=1.0, zorder=1)

all_idx_mn = np.arange(len(spec_time_mn), dtype=int)
special_mask_mn = np.zeros(len(spec_time_mn), dtype=bool)
if special_idx_mn.size > 0:
    special_mask_mn[special_idx_mn[special_idx_mn < len(spec_time_mn)]] = True
normal_idx_mn = all_idx_mn[(~special_mask_mn) & (all_idx_mn != len(spec_time_mn) - 1)]

if normal_idx_mn.size > 0:
    ax_echem.scatter(spec_volt_mn[normal_idx_mn], spec_time_mn[normal_idx_mn], c="lightgrey", edgecolors="face", s=18, zorder=2)
if special_idx_mn.size > 0:
    sp_mn = special_idx_mn[special_idx_mn < len(spec_time_mn)]
    if sp_mn.size > 0:
        ax_echem.scatter(spec_volt_mn[sp_mn], spec_time_mn[sp_mn], c=colors[0], edgecolors="w", marker="*", s=42, linewidths=0.35, zorder=4)

ax_echem.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax_echem.set_xlim(0.8, 2.0)
ax_echem.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=0.0))
ax_echem.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=0.0))
ax_echem.set_ylabel("Duration time (h)", fontsize=9)
ax_echem.set_ylim(y_min - 0.25, y_max + 1)
ax_echem.yaxis.set_major_locator(ticker.MultipleLocator(base=3.0, offset=0.0))
ax_echem.yaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=0.0))
ax_echem.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)

ax_echem.text(-0.45, 1.0, r"D", transform=ax_echem.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

ax_echem_top = ax_echem.twiny()
pos = ax_echem_top.get_position()
ax_echem_top.set_position((pos.x0, pos.y0-0.2, pos.width, pos.height))
# ax_echem_top.set_box_aspect(3.0)

ax_echem_top.plot(echem_current_ua_mn, echem_time_mn, c=colors[3], ls="--", lw=1.0, alpha=0.9, zorder=1)
ax_echem_top.set_xlim(-60, 60)
ax_echem_top.xaxis.set_major_locator(ticker.MultipleLocator(base=60.0, offset=0.0))
ax_echem_top.xaxis.set_minor_locator(ticker.MultipleLocator(base=30.0, offset=0.0))
ax_echem_top.set_xlabel(r"Current ($\mu$A)", fontsize=9)
ax_echem_top.set_ylim(y_min - 0.25, y_max + 1)
ax_echem_top.tick_params(axis="x", which="both", labelsize=8)

# 图 E
ax_spec = axs["E"]
ax_spec.shared_y_axes = ax_echem
pos = ax_spec.get_position()
ax_spec.set_position((pos.x0+0.01, pos.y0-0.2, pos.width, pos.height))
ax_spec.set_box_aspect(1.4)

xas_colors = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(0.0, 1.0, n_spec_mn)),
    name="mn_colors",
)

time_sorted_mn = np.sort(np.unique(np.asarray(spec_time_mn, dtype=float)))
time_diff_mn = np.diff(time_sorted_mn)
time_diff_mn = time_diff_mn[time_diff_mn > 0]
time_step_mn = float(np.nanmedian(time_diff_mn)) if time_diff_mn.size > 0 else 0.45
trace_half_height_mn = max(0.12, 0.34 * time_step_mn)

for m in range(n_spec_mn):
    spectrum = mu_mat_mn[m, :]
    if np.all(np.isnan(spectrum)):
        continue

    spec_min = float(np.nanmin(spectrum))
    spec_max = float(np.nanmax(spectrum))
    y_plot = spec_time_mn[m] + (spectrum - spec_min) * 2.0
    ax_spec.plot(energy_mn, y_plot, c=xas_colors.colors[m], ls="-", lw=1.0, zorder=2, clip_on=False)

# 特殊索引谱线再覆盖画一遍黑线
for m in special_idx_mn:
    spectrum = mu_mat_mn[m, :]
    if np.all(np.isnan(spectrum)):
        continue

    spec_min = float(np.nanmin(spectrum))
    spec_max = float(np.nanmax(spectrum))
    y_plot = spec_time_mn[m] + (spectrum - spec_min) * 2.0
    ax_spec.plot(energy_mn, y_plot, c="k", ls="-", lw=1.0, zorder=4, clip_on=False)

ax_spec.set_xlim(6530.0, 6610.0)
ax_spec.spines["top"].set_visible(False)
ax_spec.set_xlabel(r"Energy (eV)", fontsize=9)
ax_spec.xaxis.set_major_locator(ticker.MultipleLocator(base=20.0, offset=-10.0))
ax_spec.xaxis.set_minor_locator(ticker.MultipleLocator(base=10.0, offset=-10.0))
ax_spec.set_ylim(y_min - 0.25, y_max + 1)
ax_spec.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax_spec.tick_params(axis="y", which="both", left=False, labelleft=False)
ax_spec.set_ylabel("")
ax_spec.text(-0.04, 1.0, r"E", transform=ax_spec.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 F
ax = axs["F"]
ax.shared_y_axes = ax_echem
pos = ax.get_position()
ax.set_position((pos.x0-0.025, pos.y0-0.2, pos.width, pos.height))
ax.set_box_aspect(1.4)

xas_colors_zn = ListedColormap(
    mpl.colormaps["coolwarm"](np.linspace(0.0, 1.0, n_spec_zn)),
    name="zn_colors",
)

for m in range(n_spec_zn):
    spectrum = mu_mat_zn[m, :]
    if np.all(np.isnan(spectrum)):
        continue

    spec_min = float(np.nanmin(spectrum))
    spec_max = float(np.nanmax(spectrum))
    y_plot = spec_time_zn[m] + (spectrum - spec_min) * 2.0
    ax.plot(energy_zn, y_plot, c=xas_colors_zn.colors[m], ls="-", lw=1.0, zorder=2, clip_on=False)

for m in special_idx_zn:
    spectrum = mu_mat_zn[m, :]
    if np.all(np.isnan(spectrum)):
        continue

    spec_min = float(np.nanmin(spectrum))
    spec_max = float(np.nanmax(spectrum))
    y_plot = spec_time_zn[m] + (spectrum - spec_min) * 2.0
    ax.plot(energy_zn, y_plot, c="k", ls="-", lw=1.0, zorder=4, clip_on=False)

ax.set_xlim(9640.0, 9740.0)
ax.spines["top"].set_visible(False)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20.0, offset=0.0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10.0, offset=0.0))
ax.set_ylim(y_min - 0.25, y_max + 1)
ax.tick_params(axis="both", which="both", direction="out", top=False, right=False, labelsize=8)
ax.tick_params(axis="y", which="both", left=False, labelleft=False)
ax.set_ylabel("")
ax.text(-0.04, 1.0, r"F", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_02_V2_300.tif"), dpi=300, bbox_inches="tight", pad_inches=0.05, pil_kwargs={"compression": "tiff_lzw"})
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_02_V2_600.tif"), dpi=600, bbox_inches="tight", pad_inches=0.05, pil_kwargs={"compression": "tiff_lzw"})
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_02_V2_600.png"), dpi=600, bbox_inches="tight", pad_inches=0.05)
# plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_02_V2_900.svg"), dpi=900, bbox_inches="tight", pad_inches=0.05)

plt.gcf().set_facecolor("white")
plt.show()


## FigureMS 04 -- HRTEM + EXAFS - V2

In [ ]:
def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(f"{size:g}", barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 8},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[0].scale,  # type: ignore
            "{} {}".format(f"{size:g}", bardata.axes_manager[0].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 8},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    """转换单位：µm -> nm"""
    results = []
    for file in data:
        if len(file.axes_manager.shape) >= 2:
            if len(file.axes_manager.navigation_shape) == 2:
                if file.axes_manager.navigation_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(file.axes_manager.signal_shape) == 2:
                if file.axes_manager.signal_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
        if len(file.axes_manager.shape) == 3:
            if len(file.axes_manager.navigation_shape) == 2:
                for axis in file.axes_manager.navigation_axes:
                    axis.offset = 0
            elif len(file.axes_manager.signal_shape) == 2:
                for axis in file.axes_manager.signal_axes:
                    axis.offset = 0
        results.append(file)
    return results



In [ ]:
# 0.9V, HD#A
path_tem_hda = path_data_root / r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\TEM\Results\0019 - B8_Ceta_520_kx.emd"
HAADF_hda = hs.load(path_tem_hda)
HAADF_hda = convert_units(HAADF_hda)[0]

# 1.53V, HC#B
path_tem_hcb = path_data_root / r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\TEM\Results\01\TEM1.53V\0010-20241211_Ceta_340_kx_Ceta.dm4"
HAADF_hcb = hs.load(path_tem_hcb)
HAADF_hcb = convert_units(HAADF_hcb)[0]

# 1.63V, HC#C
path_tem_hcc = path_data_root / r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\TEM\Results\01\TEM163V\0004-20241211_Ceta_210_kx_Ceta.dm4"
HAADF_hcc = hs.load(path_tem_hcc)
HAADF_hcc = convert_units(HAADF_hcc)[0]

# 1.80V, FC#D
path_tem_fcd = path_data_root / r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\TEM\Results\01\Workspace 1\0024 - 0947 Pristine MnO2 HAADF 2.60 Mx.emd.conv_HAADF.dm4"
HAADF_fcd = hs.load(path_tem_fcd)
HAADF_fcd = convert_units(HAADF_fcd)[0]

# 1.30V, HD#E
path_tem_hde = path_data_root / r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\TEM\Results\01\TEM130V\0023-Ceta_420_kx_Camera_Ceta.dm4"
HAADF_hde = hs.load(path_tem_hde)
HAADF_hde = convert_units(HAADF_hde)[0]

# 0.9V, HD#F
path_tem_hdf = path_data_root / r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\TEM\Results\02\TEM0.9V-2\0016-Ceta_420_kx_Camera_Ceta.dm4"
HAADF_hdf = hs.load(path_tem_hdf)
HAADF_hdf = convert_units(HAADF_hdf)[0]

In [ ]:
# 读取 HRTEM d-spacing 数据
path_file = path_data_root / r"TEM\ExSitu\αMnO2\DSpacing_XRD_HRTEM\haadf_fft_dspacing\haadf_fft_d_clusters_all.csv"
d_spacings = pd.read_csv(path_file)
d_spacings = d_spacings.loc[:, [r'figure', r'cluster_id', r'd_A_mean', r'd_A_std', r'intensity_sum']]

In [ ]:
d_spacings = d_spacings.dropna(subset=["figure", "cluster_id", "d_A_mean", "d_A_std"]).copy()
# # 移除 std 为 0 或负值的数据
# d_spacings = d_spacings.loc[d_spacings["d_A_std"] > 0].copy()

name_order = ["FD#A", "HC#B", "HC#C", "FC#D", "HD#E", "FD#F"]

d_spacings["figure"] = pd.Categorical(d_spacings["figure"], categories=name_order, ordered=True)
d_spacings = d_spacings.sort_values(["figure", "cluster_id"])

In [ ]:
# 图 B FFT 统一裁剪
fft_source_items = {
    "FD#A": HAADF_hda,
    "HC#B": HAADF_hcb,
    "HC#C": HAADF_hcc,
    "FC#D": HAADF_fcd,
    "HD#E": HAADF_hde,
    "FD#F": HAADF_hdf,
}

fft_ref_name = "HC#B"
fft_ref_sig = fft_source_items[fft_ref_name]
fft_ref_fft = fft_ref_sig.fft(shift=True)
fft_ref_data = np.asarray(fft_ref_fft.data)
if fft_ref_data.ndim > 2:
    fft_ref_data = np.squeeze(fft_ref_data)
fft_ref_h, fft_ref_w = fft_ref_data.shape[-2], fft_ref_data.shape[-1]
fft_ref_crop_px = int(min(fft_ref_h, fft_ref_w))
fft_ref_q_scale = float(fft_ref_fft.axes_manager.signal_axes[0].scale)
fft_ref_q_span = fft_ref_crop_px * fft_ref_q_scale

fft_crop_plan = {}
for name, sig in fft_source_items.items():
    fft_sig = sig.fft(shift=True)
    fft_data = np.asarray(fft_sig.data)
    if fft_data.ndim > 2:
        fft_data = np.squeeze(fft_data)
    h, w = fft_data.shape[-2], fft_data.shape[-1]
    q_scale = float(fft_sig.axes_manager.signal_axes[0].scale)

    crop_size_px = int(round(fft_ref_q_span / q_scale))
    crop_size_px = max(16, min(crop_size_px, h, w))
    if crop_size_px % 2 == 1:
        crop_size_px -= 1

    y0_px = (h - crop_size_px) // 2
    x0_px = (w - crop_size_px) // 2

    fft_crop_plan[name] = {
        "crop_size_px": int(crop_size_px),
        "y0_px": int(y0_px),
        "x0_px": int(x0_px),
        "q_scale": float(q_scale),
    }

fft_nice = np.array([0.2, 0.5, 1, 2, 5, 10, 20], dtype=float)
fft_ref_target = max((fft_ref_crop_px * fft_ref_q_scale) * 0.20, fft_ref_q_scale)
fft_bar_len = float(fft_nice[np.argmin(np.abs(fft_nice - fft_ref_target))])


In [ ]:
# 读取数据

# 1.1 读取电化学数据
path_filelist = list(Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\Echem").glob(r"**\*.txt"))
# 读取电化学数据
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem_all.append(df)

# 合并所有电化学数据为一个二维表格
echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 1.2 读取 EXAFS reference + operando 数据（Mn）
path_file = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")

data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_1.chir2_mag"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_2.chir2_mag"),
    sep=r"\s+",
    index_col=0,
    header=None,
    comment="#",
)
exafs_Mn = pd.concat([data1, data2.iloc[:, 4:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, 3, *range(13, exafs_Mn.shape[1])]
exafs_Mn = exafs_Mn.iloc[:, charge_index]  # noqa: N816

opData_Mn = exafs_Mn.iloc[:, 4:].copy()  # noqa: N816
opData_Mn.columns = list(range(opData_Mn.shape[1]))


In [ ]:
# 清洗数据

# 2.1 电化学时间轴与窗口
echem_all["Time"] = (echem_all["time/s"] - echem_all["time/s"].iloc[0]).dt.total_seconds() / 3600
echem_index_low = echem_all[echem_all["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all[echem_all["cycle number"] == 1]["Ewe/V"].idxmax()

# 2.2 清洗 EXAFS operando 谱线
opData_Mn = opData_Mn.drop(columns=opData_Mn.columns[1])  # noqa: N816
opData_Mn.columns = list(range(opData_Mn.shape[1]))  # noqa: N816
cutoff_idx = np.where(opData_Mn.index < 6.0)[0]
if cutoff_idx.size == 0:
    raise ValueError("No R-space points below 6.0 Å found.")
opData_Mn = opData_Mn.iloc[: cutoff_idx[-1] + 1, :]  # noqa: N816

# 2.3 读取对应的 NorName 列表
def _read_chir_cols(path_chir):
    cols = {}
    with open(path_chir, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.startswith("# Column."):
                if line.startswith("#------------------------"):
                    break
                continue
            left, right = line.split(":", 1)
            cols[int(left.split(".")[1])] = right.strip().lstrip("-").strip()
    return cols

chir1 = Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_1.chir2_mag")
chir2 = Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_2.chir2_mag")
cols_1 = _read_chir_cols(chir1)
cols_2 = _read_chir_cols(chir2)

labels_all = [cols_1[i] for i in sorted(cols_1) if i >= 2] + [cols_2[i] for i in sorted(cols_2) if i >= 6]
keep_idx = [0, 1, 2, 3, *range(13, len(labels_all))]
op_labels = [labels_all[i] for i in keep_idx][4:]
if len(op_labels) > 1:
    op_labels.pop(1)

# 2.4 NorName -> Time 映射，并得到谱线真实时间
map_df = pd.read_csv(Path.joinpath(path_file, r"Overview", r"Time_Index_Spectrum.csv"),)
map_df = map_df[(map_df["Folder"] == "cell2_P2b_ca") & (map_df["Element"] == "Mn") & map_df["NorName"].notna()].copy()
map_df["Time"] = pd.to_datetime(map_df["Time"], errors="coerce")
map_df["NorName"] = map_df["NorName"].astype(str).str.strip().str.lstrip("-")
map_df = map_df.dropna(subset=["Time"]).sort_values("Time")
name_to_time = map_df.drop_duplicates(subset=["NorName"], keep="first").set_index("NorName")["Time"]

label_s = pd.Series(op_labels, dtype="string").str.strip().str.lstrip("-")
spec_time_dt = label_s.map(name_to_time)
missing_labels = label_s[spec_time_dt.isna()].tolist()
if missing_labels:
    raise ValueError(f"FigureSI 27: missing NorName->Time mapping for labels: {missing_labels[:6]}")

spec_time = ((spec_time_dt - echem_all["time/s"].iloc[echem_index_low]).dt.total_seconds() / 3600.0).to_numpy(dtype=float)

# 2.5 预计算画图复用变量
echem_time = (echem_all["Time"].iloc[echem_index_low:echem_index_high] - echem_all["Time"].iloc[echem_index_low]).to_numpy(dtype=float)
echem_voltage = echem_all["Ewe/V"].iloc[echem_index_low:echem_index_high].to_numpy(dtype=float)
echem_current_ua = (echem_all["<I>/mA"].iloc[echem_index_low:echem_index_high] * 1000).to_numpy(dtype=float)
spec_voltage = np.interp(spec_time, echem_time, echem_voltage)

special_idx = np.array([4, 7, 11, 13, 16, 20, 23, 29, 34, 37], dtype=int)
special_idx = special_idx[(special_idx >= 0) & (special_idx < len(spec_time))]

if len(spec_time) != opData_Mn.shape[1]:
    raise ValueError(f"FigureSI 27: len(spec_time)={len(spec_time)} != opData_Mn.shape[1]={opData_Mn.shape[1]}")


In [ ]:
# XRD-XAS 相关性数据
path_corr_csv = Path.joinpath(path_data_root, r"XRD_XAS_相关性数据_2.csv")
corr_raw = pd.read_csv(path_corr_csv, encoding="utf-8-sig")

In [ ]:
# 数值化并选择用于相关性的字段
corr_df = corr_raw.apply(pd.to_numeric, errors="coerce").copy()
env_cols = [
    "XAS_SOC",
    "XAS_R_peak1_A",
    "XAS_R_peak1_intensity",
    "XAS_R_peak2_A",
    "XAS_R_peak2_intensity",
    "XRD_SOC",
    "XRD_d_peak1_A",
    "XRD_d_peak1_intensity",
    "XRD_d_peak2_A",
    "XRD_d_peak2_intensity",
]
env_cols = [c for c in env_cols if c in corr_df.columns]

if len(env_cols) < 2:
    raise ValueError(f"Figure E: not enough columns for correlation, got {env_cols}")

corr_matrix = corr_df[env_cols].corr(method="pearson", min_periods=8)


# 标签映射

def _var_label(var: str) -> str:
    mapping = {
        "XAS_SOC": "EXAFS SOC",
        "XAS_voltage_V": "XAS-Voltage",
        "XAS_R_peak1_A": "EXAFS Mn-O (d)",
        "XAS_R_peak1_intensity": "EXAFS Mn-O (i)",
        "XAS_R_peak2_A": "EXAFS Mn-Mn (d)",
        "XAS_R_peak2_intensity": "EXAFS Mn-Mn (i)",
        "XRD_SOC": "XRD SOC",
        "XRD_voltage_V": "XRD-Voltage",
        "XRD_d_peak1_A": "XRD ZSH (d)",
        "XRD_d_peak2_A": "XRD MnO2 (d)",
        "XRD_d_peak1_intensity": "XRD ZSH (i)",
        "XRD_d_peak2_intensity": "XRD MnO2 (i)",
    }
    return mapping.get(var, var)


In [ ]:
# 画图
plt.close("all")

fig = plt.figure(figsize=(7.0, 5.0), dpi=300)
axs = fig.subplot_mosaic(
    [["A", "B", "C", "C"], ["D", "D", "E", "F"]],
    gridspec_kw={"height_ratios": [1, 1], "width_ratios": [1, 1, 1, 1], "wspace": 0.0, "hspace": 0.0},
)


# 图 A
ax_d_slot = axs["A"]
pos_d_slot = ax_d_slot.get_position()
ax_d_slot.remove()

fig.text(pos_d_slot.x0-0.11, pos_d_slot.y1, "A", fontsize=10, fontweight="bold", va="center", ha="left")

ax_d1 = fig.add_axes([
    pos_d_slot.x0-0.1,
    pos_d_slot.y0,
    pos_d_slot.width,
    pos_d_slot.height,
])
ax_d1.set_box_aspect(3.0)

ax_d1.plot(echem_voltage, echem_time, c=colors[0], ls="-", lw=1.0)
ax_d1.scatter(spec_voltage, spec_time, c="lightgrey", edgecolors="none", s=12, alpha=0.75, zorder=2)
ax_d1.scatter(
    spec_voltage[special_idx],
    spec_time[special_idx],
    marker="*",
    c=colors[0],
    edgecolors="k",
    linewidths=0.25,
    s=55,
    zorder=6,
)
ax_d1.set_ylabel(r"Duration Hours (h)", fontsize=9)
ax_d1.set_ylim(-0.3, 19)
ax_d1.yaxis.set_major_locator(ticker.MultipleLocator(base=4))
ax_d1.yaxis.set_minor_locator(ticker.MultipleLocator(base=2))
ax_d1.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}}\!)$", fontsize=9)
ax_d1.set_xlim(0.7, 1.9)
ax_d1.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=-0.1))
ax_d1.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=-0.1))
ax_d1.tick_params(axis="both", which="both", bottom=True, left=True, labelbottom=True, labelleft=True, labelsize=8)
# ax_d1.text(0.03, 0.15, r"C/10", ha="left", va="top", rotation=90, transform=ax_d1.transAxes, fontsize=8, c="k")

ax_d1_top = ax_d1.twiny()
ax_d1_top.set_position(ax_d1.get_position())
ax_d1_top.set_box_aspect(3.0)

ax_d1_top.plot(echem_current_ua, echem_time, c=colors[1], ls="--", lw=1.0)
ax_d1_top.set_xlabel(r"$\mathrm{Current \ (\mu A)}$", fontsize=9)
ax_d1_top.set_xlim(-80, 80)
ax_d1_top.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax_d1_top.xaxis.set_minor_locator(ticker.MultipleLocator(base=20, offset=0))
ax_d1_top.tick_params(axis="both", which="both", top=True, labeltop=True)

# 图 B
ax_e_slot = axs["B"]
pos_e_slot = ax_e_slot.get_position()
ax_e_slot.remove()
fig.text(pos_e_slot.x0-0.14, pos_e_slot.y1, "B", fontsize=10, fontweight="bold", va="center", ha="left")

ax_d2 = fig.add_axes([
    pos_e_slot.x0 - 0.12,
    pos_e_slot.y0,
    pos_e_slot.width,
    pos_e_slot.height,
], sharey=ax_d1)
# ax_d2.set_box_aspect(1.0)

r_axis = opData_Mn.index.to_numpy(dtype=float)  # noqa: N816
spec_time_arr = np.asarray(spec_time, dtype=float)
if spec_time_arr.size != opData_Mn.shape[1]:
    raise ValueError(f"FigureMS 04: len(spec_time)={spec_time_arr.size} != opData_Mn.shape[1]={opData_Mn.shape[1]}")
if np.any(np.diff(spec_time_arr) < 0):
    raise ValueError("FigureMS 04: spec_time must be non-decreasing for time-axis mapping")

if spec_time_arr.size == 1:
    time_edges = np.array([spec_time_arr[0] - 0.5, spec_time_arr[0] + 0.5], dtype=float)
else:
    time_edges = np.empty(spec_time_arr.size + 1, dtype=float)
    time_edges[1:-1] = 0.5 * (spec_time_arr[:-1] + spec_time_arr[1:])
    time_edges[0] = spec_time_arr[0] - 0.5 * (spec_time_arr[1] - spec_time_arr[0])
    time_edges[-1] = spec_time_arr[-1] + 0.5 * (spec_time_arr[-1] - spec_time_arr[-2])

r_grid, t_grid = np.meshgrid(r_axis, spec_time_arr)
ax_d2.pcolormesh(r_grid, t_grid, opData_Mn.T, cmap="coolwarm", shading="gouraud", zorder=1)
ax_d2.set_xlim(0.0, 6.0)
ax_d2.set_ylim(-0.3, 19)
ax_d2.set_xlabel(r"$\mathrm{R \ space \ (\AA)}$", fontsize=9)
ax_d2.xaxis.set_major_locator(ticker.MultipleLocator(base=1, offset=0))
ax_d2.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.5, offset=0))
ax_d2.tick_params(axis="both", which="both", bottom=True, left=False, labelbottom=True, labelleft=False, labelsize=8)

overlay_indices = [0, 13]
time_span = float(time_edges[-1] - time_edges[0])
overlay_amp = max(time_span * 0.08, 0.8)
for i_overlay in overlay_indices:
    if 0 <= i_overlay < opData_Mn.shape[1]:
        trace = opData_Mn.iloc[:, i_overlay].to_numpy(dtype=float)
        trace_norm = (trace - np.nanmin(trace)) / (np.nanmax(trace) - np.nanmin(trace) + 1e-12)
        y_curve = float(spec_time_arr[i_overlay]) + trace_norm * overlay_amp
        ax_d2.plot(r_axis, y_curve, ls="-", lw=0.9, color="k", alpha=0.95, zorder=6)

for idx in np.unique(np.concatenate([special_idx, np.array(overlay_indices, dtype=int)])):
    if 0 <= idx < spec_time_arr.size:
        ax_d2.axhline(y=float(spec_time_arr[idx]), lw=0.6, ls="--", color=colors[0], alpha=1.0, zorder=3)

colorbar = Colorbar(
    ax=ax_d2.inset_axes((1.05, 0.1, 0.07, 0.7)),
    location="right",
    orientation="vertical",
    cmap="coolwarm",
    ticklocation="right",
    drawedges=False,
)
colorbar.set_ticks([])
colorbar.ax.text(1.04, 0.86, r"High", transform=ax_d2.transAxes, fontsize=8, va="top", ha="left", fontfamily="Arial")
colorbar.ax.text(1.04, 0.08, r"Low", transform=ax_d2.transAxes, fontsize=8, va="top", ha="left", fontfamily="Arial")

# 图 C
ax = axs["C"]
pos = ax.get_position()
ax.remove()
fig.text(pos.x0-0.045, pos.y1, "C", fontsize=10, fontweight="bold", va="center", ha="left")

gridA = ImageGrid(
    fig,
    (pos.x0-0.025, pos.y0-0.022, pos.width*1.09, pos.height*1.09),
    nrows_ncols=(2, 3),
    axes_pad=0.1,
    share_all=True,
    label_mode="L",
)

image_items = [
    ("FD#A", HAADF_hda),
    ("HC#B", HAADF_hcb),
    ("HC#C", HAADF_hcc),
    ("FC#D", HAADF_fcd),
    ("HD#E", HAADF_hde),
    ("FD#F", HAADF_hdf),
]

for ax, (name, sig) in zip(gridA, image_items):
    arr = np.asarray(sig.data)
    if arr.ndim > 2:
        arr = np.squeeze(arr)
    vmin, vmax = np.percentile(arr, [2, 98])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = float(np.min(arr)), float(np.max(arr))
    ax.imshow(arr, cmap="gray", vmin=vmin, vmax=vmax)
    ax.set_axis_off()
    ax.text(0.02, 0.97, name, transform=ax.transAxes, va="top", ha="left", fontsize=8, color="w")

    px_scale = float(sig.axes_manager.signal_axes[0].scale)
    target = max((arr.shape[1] * px_scale) * 0.20, px_scale)
    nice = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500], dtype=float)
    bar_len = float(nice[np.argmin(np.abs(nice - target))])
    add_sizebar(ax, bar_len, sig, "w").set_bbox_to_anchor(Bbox.from_bounds(0.00, 0.01, 0, 0), ax.transAxes)

# 图 D
ax_f_slot = axs["D"]
pos_f = ax_f_slot.get_position()
ax_f_slot.remove()

axe = fig.add_axes([
    pos_f.x0-0.015,
    pos_f.y0-0.08,
    pos_f.width * 0.90,
    pos_f.height * 0.90,
])
axe.set_box_aspect(1.0)

labels = [_var_label(c) for c in corr_matrix.columns]
mat = corr_matrix.to_numpy(dtype=float)

im = axe.imshow(
    mat,
    cmap="RdYlBu_r",
    vmin=-1,
    vmax=1,
    interpolation="nearest",
    origin="upper",
    aspect="equal",
)

n = len(labels)
axe.set_xticks(np.arange(n))
axe.set_yticks(np.arange(n))
axe.set_xticklabels(labels, rotation=90, fontsize=8)
axe.set_yticklabels(labels, rotation=0, fontsize=8)
# axe.xaxis.tick_top()
# axe.tick_params(axis="both", which="major", top=False, bottom=True, labeltop=False, labelbottom=True, length=3)
axe.xaxis.tick_bottom()
axe.tick_params(axis="both", which="major", top=False, bottom=True, labeltop=False, labelbottom=True, length=3)
# axe.yaxis.tick_right()
# axe.tick_params(axis="y", which="major", right=True, left=False, labelright=True, labelleft=False, length=3)
axe.yaxis.tick_left()
axe.tick_params(axis="y", which="major", right=False, left=True, labelright=False, labelleft=True, length=3)

axe.set_xticks(np.arange(-0.5, n, 1), minor=True)
axe.set_yticks(np.arange(-0.5, n, 1), minor=True)
axe.grid(which="minor", color="white", linestyle="-", linewidth=0.3)
axe.tick_params(axis="both", which="minor", top=False, bottom=False, left=False, right=False)

cax = axe.inset_axes((1.04, 0.1, 0.06, 0.8))
cbar = fig.colorbar(im, cax=cax, orientation="vertical", pad=0.04)
cbar.ax.tick_params(labelsize=8)

axe.text(
    -0.46,
    0.97,
    r"D",
    transform=axe.transAxes,
    fontsize=10,
    va="bottom",
    ha="right",
    fontfamily="Arial",
    fontweight="bold",
    clip_on=False,
)

# 图 E
ax = axs["E"]
pos = ax.get_position()
ax.remove()

gsB = fig.add_gridspec(
    nrows=3, ncols=2,
    left=pos.x0-0.03, right=pos.x1-0.03,
    bottom=pos.y0-0.08, top=pos.y1-0.08,
    wspace=0.02, hspace=0.06
)

fig.text(pos.x0-0.045, pos.y1-0.08, "E", fontsize=10, fontweight="bold", va="center", ha="left")

# 遍历子图并填充内容
for (row, col), (name, sig) in zip(
        [(r, c) for r in range(3) for c in range(2)],
        image_items
):
    ax_fft = fig.add_subplot(gsB[row, col])

    fft_sig = sig.fft(shift=True)
    fft_arr = np.asarray(fft_sig.data)
    if fft_arr.ndim > 2:
        fft_arr = np.squeeze(fft_arr)

    plan = fft_crop_plan.get(name)
    if plan is not None:
        n = int(plan["crop_size_px"])
        y0 = int(plan["y0_px"])
        x0 = int(plan["x0_px"])
        fft_arr = fft_arr[y0:y0+n, x0:x0+n]

    fft_arr = np.log1p(np.abs(fft_arr))

    vmin, vmax = np.percentile(fft_arr, [5, 99.5])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = float(np.min(fft_arr)), float(np.max(fft_arr))

    ax_fft.imshow(fft_arr, cmap="gray", vmin=vmin, vmax=vmax)
    ax_fft.set_axis_off()
    ax_fft.text(0.02, 0.97, name, transform=ax_fft.transAxes,
                va="top", ha="left", fontsize=8, color="w")

    # 所有 FFT 子图使用 HC#B 参考比例尺
    add_sizebar(ax_fft, fft_bar_len, fft_sig, "w").set_bbox_to_anchor(
        Bbox.from_bounds(0.00, 0.01, 0, 0), ax_fft.transAxes
    )

# 图 F
ax = axs["F"]
pos = ax.get_position()
ax.set_position([pos.x0+0.025, pos.y0-0.08, pos.width, pos.height])
ax.set_box_aspect(1.5)
ax.text(-0.26, 1.0, "F", transform=ax.transAxes, fontsize=10, fontweight="bold", va="center", ha="left")

name_order = ["FD#A", "HC#B", "HC#C", "FC#D", "HD#E", "FD#F"]
cluster_min = float(d_spacings["cluster_id"].min())
cluster_max = float(d_spacings["cluster_id"].max())
offset_step = max(cluster_max - cluster_min + 3.0, 8.0)

xticks = []
for i, fig_name in enumerate(name_order):
    sub = d_spacings.loc[d_spacings["figure"] == fig_name]
    if sub.empty:
        xticks.append(i * offset_step)
        continue
    x = sub["cluster_id"].to_numpy(dtype=float) + i * offset_step
    y = sub["d_A_mean"].to_numpy(dtype=float)
    yerr = sub["d_A_std"].to_numpy(dtype=float)
    color = colors[i % len(colors)] if len(colors) else None
    ax.scatter(x, y, s=20, color=color, edgecolors="none", label=fig_name, zorder=3)
    ax.errorbar(x, y, yerr=yerr, fmt="none", ecolor=color, elinewidth=1.0, capsize=2, zorder=2)
    xticks.append(float(np.mean(x)))

ax.set_xticks(xticks)
ax.set_xticklabels(name_order, rotation=60, ha="center")
ax.tick_params(axis="both", which="both", left=True, bottom=True, labelleft=True, labelbottom=True, labelsize=8)
ax.set_ylabel(r"d-spacing ($\AA$)", fontsize=9)
ax.set_ylim(0.0, 12.0)
ax.grid(alpha=0.20, linestyle="-", linewidth=0.5)
# ax.legend(loc="upper left", fontsize=8, frameon=False, ncol=1)


# 保存图片
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_04_V2_300.tif"), dpi=300, bbox_inches="tight", pad_inches=0.05, pil_kwargs={"compression": "tiff_lzw"})
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_04_V2_600.tif"), dpi=600, bbox_inches="tight", pad_inches=0.05, pil_kwargs={"compression": "tiff_lzw"})
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_04_V2_600.png"), dpi=600, bbox_inches="tight", pad_inches=0.05)
# plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_04_V2_900.svg"), dpi=900, bbox_inches="tight", pad_inches=0.05)

plt.gcf().set_facecolor("white")
plt.show()


## FigureMS 03 -- 非原位 Zn-Mn 定量分析

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from scipy import ndimage as ndi


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 8},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[0].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[0].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 8},
        )
        ax.add_artist(asb)
    return asb


# 清洗数据
def remove_small_islands(arr: np.ndarray, min_component_pixels: int = 3) -> tuple[np.ndarray, int]:
    """Set tiny connected finite components to NaN."""
    out = np.array(arr, dtype=float, copy=True)
    valid = np.isfinite(out)
    if not np.any(valid):
        return out, 0
    labels, nlab = ndi.label(valid, structure=np.ones((3, 3), dtype=int))
    if nlab == 0:
        return out, 0
    comp_sizes = np.bincount(labels.ravel())
    small_labels = np.where(comp_sizes < int(max(1, min_component_pixels)))[0]
    small_labels = small_labels[small_labels != 0]
    if small_labels.size == 0:
        return out, 0
    drop_mask = np.isin(labels, small_labels)
    removed = int(np.count_nonzero(drop_mask))
    out[drop_mask] = np.nan
    return out, removed

In [ ]:
# 读取 TXM 的数据
def read_data(file_path, name, low_cut=0.20, high_cut=0.551):
    df = pd.read_csv(file_path, sep=",", header=None, comment="#")
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
    if name in [r"Pristine", r"MnOOH"]:
        return df
    else:
        return df.where((df >= low_cut) & (df <= high_cut), 0)


# TXM 的数据路径
path_data = Path.joinpath(path_data_root, r"STXM\ExSitu\Andrea")
scale = 0.013000000268220901  # 像素比例尺 (um/px)
cutoff = (0.20, 0.551)

# 定义所有数据集的信息
datasets = [
    (r"Pristine", r"Pristine/E1 Pristine/Pristine_Im_ratio2.txt"),
    (r"MnOOH", r"Pristine/F8 MnOOH/MnOOH_Imratio2.txt"),
    (r"1stDischarge", r"Charge/B6 1st0.9V/B6_Imratio2.txt"),
    (r"1stHalfCharge#1", r"Charge/F2 1st1.53V/F2_Im_ratio2.txt"),
    (r"1stHalfCharge#2", r"Charge/E3 1st1.63V/E3_Im_ratio2.txt"),
    (r"1stFullCharge", r"Charge/B2 1st1.80V/B2_Im_ratio2.txt"),
    (r"2ndHalfDischarge", r"Charge/F4 2nd1.30V/F4_Im_ratio2.txt"),
    (r"2ndFullDischarge", r"Charge/G1 2nd0.9V/G1_Im_ratio2.txt"),
]

# 加载和处理数据
data_dict = {name: read_data(path_data.joinpath(path), name, *cutoff) for name, path in datasets}

In [ ]:
# TXMs 的数据分析
averages = {}
averages_nonzero = {}
for key, df in data_dict.items():
    values = df.values.astype(float).ravel()
    averages[key] = float(values.mean())
    nonzero_values = values[values != 0]
    averages_nonzero[key] = float(nonzero_values.mean()) if nonzero_values.size > 0 else 0.0

# 对比分析：包含0值 vs 排除0值的平均值
print("\n" + "=" * 80)
print("对比分析：包含0值 vs 排除0值的平均值差异")
print("=" * 80)
print(f"{'数据集':<20} {'包含0值':<12} {'排除0值':<12} {'差异倍数':<10} {'零值比例'}")
print("-" * 80)

comparison_data = []
for key in data_dict.keys():
    avg_with_zero = averages[key]
    avg_without_zero = averages_nonzero[key]

    if avg_with_zero > 0:
        ratio = avg_without_zero / avg_with_zero
    else:
        ratio = float("inf") if avg_without_zero > 0 else 1.0

    # 计算零值比例
    total_points = data_dict[key].size
    zero_points = total_points - len(data_dict[key][data_dict[key] != 0])
    zero_percentage = (zero_points / total_points) * 100

    comparison_data.append((key, avg_with_zero, avg_without_zero, ratio, zero_percentage))

    print(f"{key:<20} {avg_with_zero:<12.6f} {avg_without_zero:<12.6f} {ratio:<10.1f}x {zero_percentage:<6.1f}%")

# 按排除0值后的平均值排序
sorted_nonzero = sorted(averages_nonzero.items(), key=lambda x: x[1], reverse=True)
print(r"按排除0值后的平均值从高到低排序:")
for i, (key, value) in enumerate(sorted_nonzero, 1):
    print(f"{i:2d}. {key:<20}: {value:.6f}")


In [ ]:
def create_subplot(subfig_pos, ax_pos, data_key, title, cax_pos, caption):
    """创建标准化子图"""
    subfig = fig.add_subfigure(subfig_pos)
    ax = subfig.add_axes(ax_pos)
    ax.set_box_aspect(1)

    data = data_dict[data_key]
    im = ax.imshow(data, vmin=data.min().min(), vmax=data.max().max(), cmap="hot", interpolation="bilinear")

    # 添加比例尺
    add_sizebar(ax, 2, scale, "w", r"$\mu m$").set_bbox_to_anchor(Bbox.from_bounds(0, 0.03, 0, 0), ax.transAxes)

    # 添加颜色条
    if cax_pos:
        cax = subfig.add_axes(cax_pos)
        cbar = subfig.colorbar(im, cax=cax, ticks=np.linspace(0, 1, 21), format="%.2f")
        cbar.ax.set_ylim(cutoff[0], cutoff[1])

    # 添加标题
    ax.set_title(title, loc="left", fontdict={"fontsize": 8, "fontfamily": "Arial"}, transform=ax.transAxes, y=1.0, color=colors[1])
    ax.axis("off")

    # labels
    if caption:
        ax.text(
            -0.05,
            1.05,
            f"{caption}",
            transform=ax.transAxes,
            fontsize=8,
            va="center",
            ha="right",
            fontfamily="Arial",
            fontweight="bold",
        )
    return ax

In [ ]:
# 读取 EELS L2/L3 的数据

PLOT_INDEX_CSV = path_data_root / r"TEM\ExSitu\αMnO2\Mn_L3_L2_Mapping\Ploting_Index.csv"
plot_idx = pd.read_csv(PLOT_INDEX_CSV, encoding="utf-8-sig")

signals = []
for _, row in plot_idx.iterrows():
    p = Path(str(row["resolved_output_xlsx"]))
    if not p.exists():
        continue

    df = pd.read_excel(p, sheet_name="L3L2_map_maskFalse")
    arr = df.drop(columns=["y"], errors="ignore").to_numpy(dtype=float)

    # 先做阈值清洗
    arr[(arr < 1.5) | (arr > 3.0)] = np.nan
    # 再过滤孤岛散点
    arr, removed_px = remove_small_islands(arr, min_component_pixels=3)

    s = hs.signals.Signal2D(arr)
    s.axes_manager[0].name = "y"
    s.axes_manager[1].name = "x"
    s.axes_manager[0].units = "nm"
    s.axes_manager[1].units = "nm"
    s.axes_manager[0].scale = float(row.get("scale_y", row["scale_x"]))
    s.axes_manager[1].scale = float(row["scale_x"])
    s.metadata.General.title = f"{row['name']} | removed={removed_px}"

    signals.append(s)

In [ ]:
# 读取一个常规的 αMnO2 的数据
echem = []
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\GCD\1M ZnSO4 + 0.2M MnSO4").glob(r"LC*.xlsx"))
# 读取电化学数据
for path_file in path_filelist[0:1]:
    df = pd.read_excel(path_file, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echem.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem)):
    echem[i] = echem[i][echem[i]["Cycle"].isin([0, 1, 2])]
    echem[i] = echem[i][echem[i]["Voltage/V"] >= 0]

# 读取 EDS 的数据
path_pristine = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Pristine\αMnO2 + CNTs\2024-EMCA\EDS\0002 - B8_HAADF_47000_x")
pristine = hs.load(path_pristine.joinpath(r"Data", r"0002 - B8_HAADF_47000_x.emd"))  # type: ignore

path_FD1 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st0.9V\1M ZnSO4 + 1M MnSO4\2024-EMCA\EDS\Data")
FD1 = hs.load(path_FD1.joinpath(r"0003 - B8_HAADF_67000_x", r"0003 - B8_HAADF_67000_x.emd"))  # type: ignore

path_HC1 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.53V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS\Data")
HC1 = hs.load(path_HC1.joinpath(r"0005-20241211_HAADF_82000_x_EDS", r"0005-20241211_HAADF_82000_x_EDS.emd"))  # type: ignore

path_HC2 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.63V\1M ZnSO4 + 0.2M MnSO4\2024-EMCA\EDS\Data")
HC2 = hs.load(path_HC2.joinpath(r"0009-20241211_HAADF_82000_x_EDS", r"0009-20241211_HAADF_82000_x_EDS.emd"))  # type: ignore

path_FC1 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EDS\Data")
FC1 = hs.load(path_FC1.joinpath(r"0022 - 1508 Pristine MnO2 HAADF 41000 x", r"0022 - 1508 Pristine MnO2 HAADF 41000 x.emd"))

path_HD1 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd1.30V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B1\EDS\Data")
HD1 = hs.load(path_HD1.joinpath(r"0013-HAADF_33000_x_EDS_SI", r"0013-HAADF_33000_x_EDS_SI.emd"))  # type: ignore

path_FD2 = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\EDS\Data")
FD2 = hs.load(path_FD2.joinpath(r"0028-HAADF_67000_x_EDS_SI", r"0028-HAADF_67000_x_EDS_SI.emd"))  # type: ignore

# TEM-EDS 的比例
path_fc_ratio = path_data_root / r"TEM\ExSitu\αMnO2\TEM-EDS-EELS-all\TEM-EDX_Bulk_Surface.xlsx"
fc_ratio = pd.read_excel(path_fc_ratio, sheet_name=1, header=0, index_col=None).dropna(axis=1, how="all")
fc_ratio["Samples"] = fc_ratio["Samples"].astype(str).str.strip()
fc_ratio["Position"] = fc_ratio["Position"].astype(str).str.strip()
fc_ratio["Mn/O"] = pd.to_numeric(fc_ratio["Mn/O"], errors="coerce")

# TEM-EELS 的比例
path_fc_ratio_eels = path_data_root / r"TEM\ExSitu\αMnO2\TEM-EDS-EELS-all\TEM-EELS_Bulk_Surface_NewPhase.xlsx"
fc_ratio_eels = pd.read_excel(path_fc_ratio_eels, sheet_name=1, header=0, index_col=None).dropna(axis=1, how="all")
fc_ratio_eels["Samples"] = fc_ratio_eels["Samples"].astype(str).str.strip()
fc_ratio_eels["Position"] = fc_ratio_eels["Position"].astype(str).str.strip()

# 统一 EDX/EELS 样品命名，确保图 M 中各序列一一对应
# EDX 原始名: FD#1, HC#1, HC#2, FC#1, HD#1, FD#2
# EELS 原始名: FD1st, HC#A, HC#B, FC#C, HD#D, FD#E
sample_name_map = {
    "FD#1": "FD#A",
    "HC#1": "HC#B",
    "HC#2": "HC#C",
    "FC#1": "FC#D",
    "HD#1": "HD#E",
    "FD#2": "FD#F",
    "FD1st": "FD#A",
    "HC#A": "HC#B",
    "HC#B": "HC#C",
    "FC#C": "FC#D",
    "HD#D": "HD#E",
    "FD#E": "FD#F",
}
eels_sample_order = ["Pristine", "FD#A", "HC#B", "HC#C", "FC#D", "HD#E", "FD#F"]
eels_position_order = ["Bulk", "Surface", "NewPhase"]

fc_ratio["Zn/Mn"] = fc_ratio["Zn/O"] / fc_ratio["Mn/O"]
fc_ratio["Sample"] = fc_ratio["Samples"].replace(sample_name_map)
fc_ratio = fc_ratio[fc_ratio["Sample"].isin(eels_sample_order) & fc_ratio["Position"].isin(["Bulk", "Surface"])].copy()

fc_ratio_eels["Sample"] = fc_ratio_eels["Samples"].replace(sample_name_map)
fc_ratio_eels = fc_ratio_eels[
    fc_ratio_eels["Sample"].isin(eels_sample_order) & fc_ratio_eels["Position"].isin(eels_position_order)
].copy()
fc_ratio_eels["Sample"] = pd.Categorical(fc_ratio_eels["Sample"], categories=eels_sample_order, ordered=True)
fc_ratio_eels["Position"] = pd.Categorical(fc_ratio_eels["Position"], categories=eels_position_order, ordered=True)


fc_ratio["Zn/O"] = pd.to_numeric(fc_ratio["Zn/O"], errors="coerce")
fc_ratio["Zn/Mn"] = pd.to_numeric(fc_ratio["Zn/Mn"], errors="coerce")
fc_ratio_eels["Zn/O"] = pd.to_numeric(fc_ratio_eels["Zn/O"], errors="coerce")
fc_ratio_eels["Zn/Mn"] = pd.to_numeric(fc_ratio_eels["Zn/Mn"], errors="coerce")

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 8},
        )
        ax.add_artist(asb)
    else:
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata.axes_manager[0].scale,  # type: ignore
            "{} {}".format(size, bardata.axes_manager[0].units),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 8},
        )
        ax.add_artist(asb)
    return asb


def convert_units(data):
    results = []
    for file in data:
        if len(file.axes_manager.shape) >= 2:
            if len(file.axes_manager.navigation_shape) == 2:
                if file.axes_manager.navigation_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
            elif len(file.axes_manager.signal_shape) == 2:  # noqa: SIM102
                if file.axes_manager.signal_axes[0].units == r"µm":
                    file.axes_manager.convert_units(axes="signal", units="nm", same_units=True, factor=1000)

        if len(file.axes_manager.shape) == 3:
            if len(file.axes_manager.navigation_shape) == 2:
                for axis in file.axes_manager.navigation_axes:
                    axis.offset = 0
            elif len(file.axes_manager.signal_shape) == 2:
                for axis in file.axes_manager.signal_axes:
                    axis.offset = 0
        results.append(file)
    return results


def element_maps(signal_list, elements=(r"K", r"Mn", r"Zn"), crop_box=None):
    element_maps = {}
    for signal in signal_list:
        title = signal.metadata["General"]["title"]
        if title in elements:
            if crop_box:
                if signal.axes_manager.navigation_shape:
                    signal_crop = signal.inav[crop_box].copy()
                elif signal.axes_manager.signal_shape:
                    signal_crop = signal.isig[crop_box].copy()
            else:
                signal_crop = signal.copy()

            element_maps[title] = signal_crop
    return element_maps


eds_colors = {
    "C": "#FF8000",
    "K": "#1A32CB",
    "Mn": "#FFFF00",
    "Zn": "#fc00fc",
    "S": "#1f3cef",
    "O": "#00ff00",
}

In [ ]:
pristine = convert_units(pristine)
FD1 = convert_units(FD1)
HC1 = convert_units(HC1)
HC2 = convert_units(HC2)
FC1 = convert_units(FC1)
HD1 = convert_units(HD1)
FD2 = convert_units(FD2)

pristine = element_maps(pristine, elements=["K", "Mn"], crop_box=(slice(0, 60), slice(None)))
FD1 = element_maps(FD1, elements=["K", "Mn", "Zn"], crop_box=(slice(90, 200), slice(None)))
HC1 = element_maps(HC1, elements=["K", "Mn", "Zn"], crop_box=(slice(None), slice(90, 200)))
HC2 = element_maps(HC2, elements=["K", "Mn", "Zn"])
FC1 = element_maps(FC1, elements=["K", "Mn", "Zn"])
HD1 = element_maps(HD1, elements=["K", "Mn", "Zn"])
FD2 = element_maps(FD2, elements=["K", "Mn", "Zn"])
FD2 = {k: v.transpose(signal_axes=(1, 0)) for k, v in FD2.items()}  # 在数据处理阶段使用 HyperSpy 转置


In [ ]:
# 读取 STEM-EELS 谱线的数据
path_eels = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\TEM-EDS-EELS-all")

TEM_EELS_Mn = pd.read_csv(path_eels.joinpath(r"TEM-EELS_Mn_selected.nor"), sep=r"\s+", header=None, index_col=None, comment=r"#")
TEM_EELS_O = pd.read_csv(path_eels.joinpath(r"TEM-EELS_O_selected.nor"), sep=r"\s+", header=None, index_col=None, comment=r"#")

TEM_EELS_Mn = {
    "energy": TEM_EELS_Mn.iloc[:, 0].copy(),
    "bulk": TEM_EELS_Mn.iloc[:, [1, 3, 7, 9, 12, 19, 21]].copy(),
    "surface": TEM_EELS_Mn.iloc[:, [2, 6, 8, 13, 18, 20]].copy(),
    "new_phase": TEM_EELS_Mn.iloc[:, [4, 5, 10, 11, 14, 15, 16, 17]].copy(),
}

In [ ]:
# 删除多余的 1stFD
TEM_EELS_Mn["bulk"].drop(columns=TEM_EELS_Mn["bulk"].columns[1], inplace=True)
TEM_EELS_Mn["surface"].drop(columns=TEM_EELS_Mn["surface"].columns[1], inplace=True)

In [ ]:
# 画图
plt.close("all")

fig = plt.figure(figsize=(7.0, 5.0), dpi=300)
layout = [
    ["A", "A", "BCFG"],
    ["D", "D", "BCFG"],
    ["E", "E", "BCFG"],
]
axs = fig.subplot_mosaic(
    layout,
    gridspec_kw={
        "width_ratios": [1.0, 1.0, 2.0],
        "height_ratios": [1.0, 1.0, 1.0],
        "wspace": 0.0,
        "hspace": 0.0,
    },
)

ax_bcfg = axs["BCFG"]
bcfg_spec = ax_bcfg.get_subplotspec()
ax_bcfg.remove()
gs_bcfg = bcfg_spec.subgridspec(
    2,
    2,
    width_ratios=[1.0, 1.0],
    height_ratios=[1.0, 1.0],
    wspace=0.0,
    hspace=0.0,
)
ax_b = fig.add_subplot(gs_bcfg[0, 0])
ax_c = fig.add_subplot(gs_bcfg[0, 1])
ax_f = fig.add_subplot(gs_bcfg[1, 0])
ax_g = fig.add_subplot(gs_bcfg[1, 1])

# 图 A
ax_a = axs["A"]
pos = ax_a.get_position()
ax_a.set_position([pos.x0, pos.y0, pos.width * 1.065, pos.height * 1.065])
txm_rect = ax_a.get_position().bounds
ax_a.set_axis_off()

txm_grid = ImageGrid(
    fig,
    txm_rect,
    nrows_ncols=(2, 3),
    axes_pad=0.15,
    share_all=False,
    label_mode="L",
    cbar_mode="single",
    cbar_location="right",
)

txm_panels = [
    ("1stDischarge", r"FD#A"),
    ("1stHalfCharge#1", r"HC#B"),
    ("1stHalfCharge#2", r"HC#C"),
    ("1stFullCharge", r"FC#D"),
    ("2ndHalfDischarge", r"HD#E"),
    ("2ndFullDischarge", r"FD#F"),
]

txm_norm = mpl.colors.Normalize(vmin=0.2, vmax=0.6)
for i, (data_key, title) in enumerate(txm_panels):
    ax = txm_grid[i]
    last_txm_im = ax.imshow(data_dict[data_key], cmap="hot", norm=txm_norm, interpolation="bilinear")
    if i == 0:
        add_sizebar(ax, 2, scale, "w", r"$\mu m$").set_bbox_to_anchor(Bbox.from_bounds(0, 0.01, 0, 0), ax.transAxes)   
        ax.text(-0.05, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

    ax.set_title(title, loc="left", fontsize=8, fontfamily="Arial", color=colors[1], pad=2)
    ax.axis("off")

if last_txm_im is not None:
    cbar_txm = txm_grid.cbar_axes[0].colorbar(last_txm_im)
    cbar_txm.set_label("Shape Index", fontsize=8, labelpad=4.0)
    cbar_txm.set_ticks(np.arange(0.2, 0.61, 0.1))
    cbar_txm.ax.tick_params(labelsize=8)
    cax = txm_grid.cbar_axes[0]
    pos = cax.get_position()
    cax.set_axes_locator(None)
    cax.set_position([pos.x0+0.47, pos.y0 + pos.height * 0.20, 0.012, pos.height * 0.70])

# 图 B
scale_ratio = 1.0
ax = ax_b
pos = ax.get_position()
ax.set_position([pos.x0+0.13, pos.y0, pos.width*scale_ratio, pos.height*scale_ratio])
ax.set_box_aspect(1.5)

line_paras = [
    (0.29, 0.1, 0.95, "k"),
    (0.74, 0.1, 0.25, "k"),
    (0.32, 0.4, 0.5, "w"),
    (0.39, 0.45, 0.55, "w"),
    (0.45, 0.6, 0.7, "w"),
    (0.44, 0.68, 0.78, "k"),
    (0.42, 0.75, 0.85, "w"),
    (0.35, 0.88, 0.95, "w"),
]
hist_keys = [r"Pristine", r"MnOOH", r"1stDischarge", r"1stHalfCharge#1", r"1stHalfCharge#2", r"1stFullCharge", r"2ndHalfDischarge", r"2ndFullDischarge"]
hist_names = [r"Pristine", r"MnOOH", r"FD#A", r"HC#B", r"HC#C", r"FC#D", r"HD#E", r"FD#F"]

for i, key in enumerate(hist_keys):
    value = data_dict[key]
    color_i = colors[i % len(colors)]
    label_i = hist_names[i]
    if key in [r"Pristine", r"MnOOH"]:
        ax.hist(value.values.flatten(), bins=100, density=True, color=color_i, align="mid", range=(0.2, 1.0), zorder=0, label=label_i, bottom=0, alpha=1.0, stacked=True)
        ax.axvline(x=line_paras[i][0], ymax=line_paras[i][1], ymin=line_paras[i][2], color=line_paras[i][3], linestyle="--", linewidth=1.0, zorder=10)
    else:
        ax.hist(value.values.flatten(), bins=100, density=True, color=color_i, align="mid", range=cutoff, zorder=len(hist_keys) - i + 1, label=label_i, bottom=2.0 + 3.0 * i, alpha=1.0)
        ax.axvline(x=line_paras[i][0], ymax=line_paras[i][1], ymin=line_paras[i][2], color=line_paras[i][3], linestyle="--", linewidth=1.0, zorder=10)

ax.set_ylabel(r"Normalized Frequency", fontsize=9)
ax.set_xlabel(r"Shape Index", fontsize=9)
ax.set_xlim(0.2, 0.9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax.tick_params(axis="x", labelsize=8)
ax.tick_params(axis="y", which="both", left=False, labelleft=False)
ax.legend(loc="upper right", ncols=1, frameon=False, fontsize=8, handletextpad=0.18, labelspacing=0.08, borderpad=0.03, columnspacing=0.8)
ax.text(-0.12, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
ax = ax_c
pos = ax.get_position()
ax.set_position([pos.x0+0.26, pos.y0, pos.width*scale_ratio, pos.height*scale_ratio])
ax.set_box_aspect(1.5)

# labels = [r"Pristine", r"FD#A", r"HC#B", r"HC#C", r"FC#D", r"HD#E", r"FD#F"]
labels = [r"Pristine", r"HC#B", r"HC#C", r"FC#D", r"HD#E", r"FD#F"]

shift_energy_hd_mn = 3.6
shift_energy_fd_mn = 2.7
for i in range(TEM_EELS_Mn["bulk"].shape[1]):
    if i < 4:
        ax.plot(TEM_EELS_Mn["energy"], TEM_EELS_Mn["bulk"].iloc[:, i] + 0.4 * i, label=labels[i], color=colors[i], lw=1.0)
    else:
        shift_i = shift_energy_hd_mn if i == 4 else shift_energy_fd_mn
        ax.plot(TEM_EELS_Mn["energy"] + shift_i, TEM_EELS_Mn["bulk"].iloc[:, i] + 0.4 * i, label=labels[i], color=colors[i], lw=1.0)

for i in range(TEM_EELS_Mn["surface"].shape[1]):
    if i < 3:
        ax.plot(TEM_EELS_Mn["energy"], TEM_EELS_Mn["surface"].iloc[:, i] + 0.4 * i + 4, linestyle="-", label=None, color=colors[i + 1], lw=1.0)
    else:
        shift_i = shift_energy_hd_mn if i == 3 else shift_energy_fd_mn
        ax.plot(TEM_EELS_Mn["energy"] + shift_i, TEM_EELS_Mn["surface"].iloc[:, i] + 0.4 * i + 4, linestyle="-", label=None, color=colors[i + 1], lw=1.0)

for i in range(TEM_EELS_Mn["new_phase"].shape[1] // 2):
    if i < 3:
        ax.plot(TEM_EELS_Mn["energy"], TEM_EELS_Mn["new_phase"].iloc[:, 2 * i] + 1.0 * i + 9, linestyle="-", label=None, color=colors[i + 2], lw=1.0)
        ax.plot(TEM_EELS_Mn["energy"], TEM_EELS_Mn["new_phase"].iloc[:, 2 * i + 1] + 1.0 * i + 9, linestyle="-", label=None, color=colors[i + 2], lw=1.0)
    else:
        ax.plot(TEM_EELS_Mn["energy"] + shift_energy_hd_mn, TEM_EELS_Mn["new_phase"].iloc[:, 2 * i] + 1.0 * i + 9, linestyle="-", label=None, color=colors[i + 2], lw=1.0)
        ax.plot(TEM_EELS_Mn["energy"] + shift_energy_hd_mn, TEM_EELS_Mn["new_phase"].iloc[:, 2 * i + 1] + 1.0 * i + 9, linestyle="-", label=None, color=colors[i + 2], lw=1.0)

ax.set_xlim(630, 660)
ax.set_xlabel(r"Energy (eV)", fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=0))
ax.set_ylim(-0.1, 20)
ax.set_ylabel(r"Relative Absorption (arb. u.)", fontsize=9)
ax.set_yticks([])
ax.tick_params(axis="both", which="both", labelsize=8, bottom=True, left=True, labelbottom=True, labelleft=True)
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 1.01), ncols=2, frameon=False, labelcolor="linecolor", fontsize=8, columnspacing=0.5)
ax.text(0.65, 0.22, r"Bulk", transform=ax.transAxes, fontsize=8, va="center", ha="left", fontfamily="Arial")
ax.text(0.65, 0.45, r"Surface", transform=ax.transAxes, fontsize=8, va="center", ha="left", fontfamily="Arial")
ax.text(0.65, 0.75, r"New Phase", transform=ax.transAxes, fontsize=8, va="center", ha="left", fontfamily="Arial")
ax.text(0.88, 0.95, r"Mn", transform=ax.transAxes, fontsize=9, va="center", ha="left", fontfamily="Arial", fontweight="bold")
ax.vlines(642.5, -0.1, 16, colors="gray", linestyles="dashed", linewidth=1.0, zorder=1)
ax.text(-0.12, 1.0, r"C", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
ax_d = axs["D"]
d_spec = ax_d.get_subplotspec()
ax_d.set_axis_off()

gs_d_outer = d_spec.subgridspec(2, 1, height_ratios=[1.0, 1.0], hspace=0.0, wspace=0.0)

norm_l3 = mpl.colors.Normalize(vmin=1.6, vmax=2.8)
cmap_l3 = mpl.colormaps["hot"].copy()
cmap_l3.set_bad("black")

l3_items = [
    (0, "Pristine"),
    (1, "FD#A"),
    (2, "HC#B"),
    (3, "HC#C"),
    (4, "FC#D"),
    (6, "HD#E"),
    (7, "FD#F"),
]


def _l3_arr(sig_idx):
    arr = signals[sig_idx].data
    if sig_idx in (0, 1, 7):
        arr = arr.T
    if sig_idx == 4:
        arr = arr[:78, :]
    if sig_idx == 6:
        arr = arr[:58, :76]
    return arr

arr_row1 = [_l3_arr(sig) for sig, _ in l3_items[:4]]
arr_row2 = [_l3_arr(sig) for sig, _ in l3_items[4:]]

wr_row1 = [max(arr.shape[1], 1) / max(arr.shape[0], 1) for arr in arr_row1]
wr_row2 = [max(arr.shape[1], 1) / max(arr.shape[0], 1) for arr in arr_row2]

gs_d_row1 = gs_d_outer[0].subgridspec(1, 4, width_ratios=wr_row1, wspace=0.03)
gs_d_row2 = gs_d_outer[1].subgridspec(1, 3, width_ratios=wr_row2, wspace=0.03)

d_axes = [
    fig.add_subplot(gs_d_row1[0, 0]),
    fig.add_subplot(gs_d_row1[0, 1]),
    fig.add_subplot(gs_d_row1[0, 2]),
    fig.add_subplot(gs_d_row1[0, 3]),
    fig.add_subplot(gs_d_row2[0, 0]),
    fig.add_subplot(gs_d_row2[0, 1]),
    fig.add_subplot(gs_d_row2[0, 2]),
]

shift_x=[0.03, 0.015, -0.008, -0.0345, 0.012, -0.02, -0.035]
shift_y=[-0.07, -0.07, -0.07, -0.07, -0.10, -0.10, -0.10]
shift_title=[0.30, 0.3, 0.12, 0.2, 0.12, 0.15, 0.35]
last_l3_im = None
for i, ((sig_idx, title), ax) in enumerate(zip(l3_items, d_axes)):
    arr = _l3_arr(sig_idx)
    pos = ax.get_position()
    ax.set_position([pos.x0+shift_x[i], pos.y0+shift_y[i], pos.width, pos.height])
    last_l3_im = ax.imshow(arr, cmap=cmap_l3, norm=norm_l3, interpolation="bilinear", aspect=1.0)
    add_sizebar(ax, size=plot_idx[r"recommended_scalebar_len"][sig_idx], bardata=signals[sig_idx], color="w").set_bbox_to_anchor(Bbox.from_bounds(0.0, 0.01, 0, 0), ax.transAxes)
    ax.axis("off")
    ax.text(shift_title[i], 1.0, title, transform=ax.transAxes, fontsize=8, va="bottom", ha="center", fontfamily="Arial")
    if i == 0:
        ax.text(-0.15, 1.0, r"D", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

if last_l3_im is not None:
    cax = ax_d.inset_axes([0.93, 0.0, 0.025, 0.7])
    cbar_eels = fig.colorbar(last_l3_im, cax=cax)
    cbar_eels.set_label("L3/L2 ratio", fontsize=8, labelpad=4.0)
    cbar_eels.set_ticks(np.arange(1.6, 2.81, 0.3))
    cbar_eels.ax.tick_params(labelsize=8)

# 图 E
def add_sizebar_um(ax, size_um, signal, color="w"):
    nm_per_px = signal.axes_manager[0].scale
    return add_sizebar(ax, size_um, nm_per_px / 1000.0, color, barunits=r"$\mu m$")

ax = axs["E"]
pos = ax.get_position()
ax.set_position([pos.x0+0.01, pos.y0-0.16, pos.width, pos.height])
ax.set_box_aspect(0.4)
ax.set_axis_off()

for i, data in enumerate(echem):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        idx_min = temp["Voltage/V"].idxmin()
        if j == 0:
            ax.plot(temp.loc[idx_min:, "SpeCap/mAh/g"] + temp.loc[idx_min - 1, "SpeCap/mAh/g"], temp.loc[idx_min:, "Voltage/V"], ls="-", lw=1.0, c=colors[j], zorder=0)
        if j == 1:
            ax.plot(temp.loc[:idx_min, "SpeCap/mAh/g"] + 690, temp.loc[:idx_min, "Voltage/V"], ls="-", lw=1.0, c=colors[j], zorder=0)

# Pristine
axins = ax.inset_axes((0.0, 0.75, 0.28, 0.30), zorder=1)
hs.plot.plot_images([pristine["K"], pristine["Mn"]], ax=axins, overlay=True, colors=[eds_colors["K"], eds_colors["Mn"]], axes_decor="off", label=None, alphas=[1.0, 1.0], vmax="100th")
add_sizebar_um(axins, size_um=1, signal=pristine["K"], color="w").set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.text(0.75, -0.10, r"Pristine", transform=axins.transAxes, fontsize=8, va="top", ha="center", color=colors[2])

# FD#A
axins = ax.inset_axes((0.03, 0.05, 0.22, 0.33), zorder=1)
hs.plot.plot_images([FD1["K"], FD1["Mn"], FD1["Zn"]], ax=axins, overlay=True, colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]], axes_decor="off", label=None, alphas=[1.0, 1.0, 1.0], vmax="100th")
add_sizebar_um(axins, size_um=0.5, signal=FD1["K"], color="w").set_bbox_to_anchor(Bbox.from_bounds(-0.04, -0.03, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.text(0.75, -0.10, r"FD#A", transform=axins.transAxes, fontsize=8, va="top", ha="center", color=colors[0])

axins.text(
    0.08,
    1.32,
    r"Mn",
    c="k",
    bbox={"ec": None, "fc": eds_colors["Mn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=axins.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)
axins.text(
    0.50,
    1.32,
    r"Zn",
    c="w",
    bbox={"ec": None, "fc": eds_colors["Zn"], "alpha": 1.0, "boxstyle": "Square, pad=0.3"},
    transform=axins.transAxes,
    fontsize=8,
    va="top",
    ha="left",
    fontfamily="Arial",
    fontweight="bold",
)

# HC#B
axins = ax.inset_axes((0.22, 0.05, 0.22, 0.33), zorder=1)
hs.plot.plot_images([HC1["K"], HC1["Mn"], HC1["Zn"]], ax=axins, overlay=True, colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]], axes_decor="off", label=None, alphas=[1.0, 1.0, 1.0], vmax="100th")
add_sizebar_um(axins, size_um=0.5, signal=HC1["K"], color="w").set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.text(0.80, -0.10, r"HC#B", transform=axins.transAxes, fontsize=8, va="top", ha="center", color=colors[0])

# HC#C
axins = ax.inset_axes((0.38, 0.05, 0.3, 0.5), zorder=1)
hs.plot.plot_images([HC2["K"], HC2["Mn"], HC2["Zn"]], ax=axins, overlay=True, colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]], axes_decor="off", label=None, alphas=[1.0, 1.0, 1.0], vmax="100th")
add_sizebar_um(axins, size_um=0.5, signal=HC2["K"], color="w").set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.text(0.8, -0.05, r"HC#C", transform=axins.transAxes, fontsize=8, va="top", ha="center", color=colors[0])

# FC#D
axins = ax.inset_axes((0.56, 0.7, 0.25, 0.35), zorder=1)
hs.plot.plot_images([FC1["K"], FC1["Mn"], FC1["Zn"]], ax=axins, overlay=True, colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]], axes_decor="off", label=None, alphas=[1.0, 1.0, 1.0], vmax="100th")
add_sizebar_um(axins, size_um=0.5, signal=FC1["K"], color="w").set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.text(0.8, -0.10, r"FC#D", transform=axins.transAxes, fontsize=8, va="top", ha="center", color=colors[0])

# HD#E
axins = ax.inset_axes((0.68, 0.55, 0.4, 0.6), zorder=1)
hs.plot.plot_images([HD1["K"], HD1["Mn"], HD1["Zn"]], ax=axins, overlay=True, colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]], axes_decor="off", label=None, alphas=[1.0, 1.0, 1.0], vmax="100th")
add_sizebar_um(axins, size_um=5, signal=HD1["K"], color="w").set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.text(0.75, -0.05, r"HD#E", transform=axins.transAxes, fontsize=8, va="top", ha="center", color=colors[0])

# FD#F
axins = ax.inset_axes((0.56, -0.10, 0.4, 0.45), zorder=1)
hs.plot.plot_images([FD2["K"], FD2["Mn"], FD2["Zn"]], ax=axins, overlay=True, colors=[eds_colors["K"], eds_colors["Mn"], eds_colors["Zn"]], axes_decor="off", label=None, alphas=[1.0, 1.0, 1.0], vmax="100th")
add_sizebar_um(axins, size_um=1, signal=FD2["K"], color="w").set_bbox_to_anchor(Bbox.from_bounds(-0.02, -0.02, 0, 0), transform=axins.transAxes)
axins.set_axis_off()
axins.text(0.85, -0.05, r"FD#F", transform=axins.transAxes, fontsize=8, va="top", ha="center", color=colors[1])

# # 路径箭头
# arrow_style = dict(arrowstyle="->", connectionstyle="angle,angleA=0,angleB=90,rad=10", ls="-", lw=1.0, color="grey")
# ax.annotate("", xy=(0.17, 0.34), xytext=(0.25, 0.50), xycoords="axes fraction", textcoords="axes fraction", arrowprops=arrow_style)
# ax.annotate("", xy=(0.32, 0.34), xytext=(0.43, 0.58), xycoords="axes fraction", textcoords="axes fraction", arrowprops=arrow_style)
# ax.annotate("", xy=(0.52, 0.70), xytext=(0.61, 0.76), xycoords="axes fraction", textcoords="axes fraction", arrowprops=arrow_style)
# ax.annotate("", xy=(0.62, 0.24), xytext=(0.62, 0.40), xycoords="axes fraction", textcoords="axes fraction", arrowprops=arrow_style)
# ax.annotate("", xy=(0.81, 0.07), xytext=(0.88, 0.20), xycoords="axes fraction", textcoords="axes fraction", arrowprops=arrow_style)

ax.text(0.025, 1.05, r"E", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 F
ax = ax_f
pos = ax.get_position()
ax.set_position([pos.x0+0.13, pos.y0 - 0.1, pos.width*scale_ratio, pos.height*scale_ratio])
ax.set_box_aspect(1.5)

x = np.arange(len(eels_sample_order), dtype=float)
x_map = {s: i for i, s in enumerate(eels_sample_order)}

bulk_edx_zo = fc_ratio[fc_ratio["Position"] == "Bulk"].groupby("Sample", observed=True)["Zn/O"].mean().reindex(eels_sample_order)
surface_edx_zo = fc_ratio[fc_ratio["Position"] == "Surface"].groupby("Sample", observed=True)["Zn/O"].mean().reindex(eels_sample_order)

(line1,) = ax.plot(x, bulk_edx_zo.to_numpy(dtype=float), c=colors[0], lw=1.0, ls="-", marker="o", ms=4.5, mec=colors[0], mfc=colors[0], label="Bulk")
(line2,) = ax.plot(x, surface_edx_zo.to_numpy(dtype=float), c=colors[0], lw=1.0, ls="-", marker="D", ms=4.5, mec=colors[0], mfc=colors[0], alpha=0.9, label="Surface")

ax.set_ylabel(r"EDX Zn/O Relative Ratio (arb. u.)", fontsize=9, color=colors[0])
ax.set_ylim(-0.05, 1.2)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1))
ax.tick_params(axis="y", labelsize=8, colors=colors[0])
ax.set_xticks(x)
ax.set_xticklabels(eels_sample_order, rotation=55, ha="right", rotation_mode="anchor", fontfamily="Arial", fontsize=8)

ax2 = ax.twinx()
pos = ax2.get_position()
ax2.set_position([pos.x0+0.13, pos.y0 - 0.1, pos.width*scale_ratio, pos.height*scale_ratio])
ax2.set_box_aspect(1.5)

bulk_eels_zo = fc_ratio_eels[fc_ratio_eels["Position"] == "Bulk"].groupby("Sample", observed=True)["Zn/O"].mean().reindex(eels_sample_order)
surface_eels_zo = fc_ratio_eels[fc_ratio_eels["Position"] == "Surface"].groupby("Sample", observed=True)["Zn/O"].mean().reindex(eels_sample_order)

(line3,) = ax2.plot(x, bulk_eels_zo.to_numpy(dtype=float), c=colors[1], lw=1.0, ls="--", marker="o", ms=4.5, mfc="none", mec=colors[1], label="Bulk")
(line4,) = ax2.plot(x, surface_eels_zo.to_numpy(dtype=float), c=colors[1], lw=1.0, ls="--", marker="D", ms=4.5, mfc="none", mec=colors[1], alpha=0.9, label="Surface")

sub_np_zo = fc_ratio_eels[fc_ratio_eels["Position"] == "NewPhase"].copy()
sub_np_zo["x"] = sub_np_zo["Sample"].astype(str).map(x_map)
sub_np_zo = sub_np_zo.dropna(subset=["x", "Zn/O"])
star1 = None
if not sub_np_zo.empty:
    star1 = ax2.scatter(sub_np_zo["x"].to_numpy(dtype=float), sub_np_zo["Zn/O"].to_numpy(dtype=float), marker="*", s=40, c=colors[1], linewidths=0.4, edgecolors="w", zorder=6, label="NewPhase")

ax2.set_ylabel(r"EELS Zn/O Relative Ratio (arb. u.)", fontsize=9, color=colors[1])
ax2.set_ylim(-0.05, 0.4)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.1))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.05))
ax2.tick_params(axis="y", labelsize=8, colors=colors[1])

legend_handles = [line1, line2, line3, line4]
if star1 is not None:
    legend_handles.append(star1)
ax.legend(handles=legend_handles, loc="upper left", fontsize=8, frameon=False, ncol=2, bbox_to_anchor=(0.02, 1.0), columnspacing=0.6, handlelength=1.5)
ax.text(-0.25, 1.0, r"F", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 G
ax = ax_g
pos = ax.get_position()
ax.set_position([pos.x0+0.26, pos.y0-0.1, pos.width*scale_ratio, pos.height*scale_ratio])
ax.set_box_aspect(1.5)

bulk_edx = fc_ratio[fc_ratio["Position"] == "Bulk"].groupby("Sample", observed=True)["Zn/Mn"].mean().reindex(eels_sample_order)
surface_edx = fc_ratio[fc_ratio["Position"] == "Surface"].groupby("Sample", observed=True)["Zn/Mn"].mean().reindex(eels_sample_order)
surface_edx.iloc[:2] = np.nan

(line1,) = ax.plot(x, bulk_edx.to_numpy(dtype=float), c=colors[0], lw=1.0, ls="-", marker="o", ms=4.5, mec=colors[0], mfc=colors[0], label="Bulk")
(line2,) = ax.plot(x, surface_edx.to_numpy(dtype=float), c=colors[0], lw=1.0, ls="-", marker="D", ms=4.5, mec=colors[0], mfc=colors[0], alpha=0.9, label="Surface")

ax.set_ylabel(r"EDX Zn/Mn Relative Ratio (arb. u.)", fontsize=9, color=colors[0])
ax.set_ylim(-0.05, 3.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.6))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.3))
ax.tick_params(axis="y", labelsize=8, colors=colors[0])
ax.set_xticks(x)
ax.set_xticklabels(eels_sample_order, rotation=55, ha="right", rotation_mode="anchor", fontfamily="Arial", fontsize=8)

ax2 = ax.twinx()
pos = ax2.get_position()
ax2.set_position([pos.x0+0.26, pos.y0-0.1, pos.width*scale_ratio, pos.height*scale_ratio])
ax2.set_box_aspect(1.5)

bulk_eels = fc_ratio_eels[fc_ratio_eels["Position"] == "Bulk"].groupby("Sample", observed=True)["Zn/Mn"].mean().reindex(eels_sample_order)
surface_eels = fc_ratio_eels[fc_ratio_eels["Position"] == "Surface"].groupby("Sample", observed=True)["Zn/Mn"].mean().reindex(eels_sample_order)
surface_eels.iloc[-1] = np.nan

(line3,) = ax2.plot(x, bulk_eels.to_numpy(dtype=float), c=colors[1], lw=1.0, ls="--", marker="o", ms=4.5, mfc="none", mec=colors[1], label="Bulk")
(line4,) = ax2.plot(x, surface_eels.to_numpy(dtype=float), c=colors[1], lw=1.0, ls="--", marker="D", ms=4.5, mfc="none", mec=colors[1], alpha=0.9, label="Surface")

sub_np = fc_ratio_eels[fc_ratio_eels["Position"] == "NewPhase"].copy()
sub_np["x"] = sub_np["Sample"].astype(str).map(x_map)
sub_np = sub_np.dropna(subset=["x", "Zn/Mn"])
star2 = None
if not sub_np.empty:
    star2 = ax2.scatter(sub_np["x"].to_numpy(dtype=float), sub_np["Zn/Mn"].to_numpy(dtype=float), marker="*", s=40, c=colors[1], linewidths=0.4, edgecolors="w", zorder=6, label="NewPhase")

ax2.set_ylabel(r"EELS Zn/Mn Relative Ratio (arb. u.)", fontsize=9, color=colors[1])
ax2.set_ylim(-0.05, 4)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=1.0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.5))
ax2.tick_params(axis="y", labelsize=8, colors=colors[1], right=True, labelright=True)

legend_handles = [line1, line2, line3, line4]
if star2 is not None:
    legend_handles.append(star2)
ax.legend(handles=legend_handles, loc="upper left", fontsize=8, frameon=False, ncol=2, bbox_to_anchor=(0.02, 1.0), columnspacing=0.6, handlelength=1.5)
ax.text(-0.25, 1.0, r"G", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# # 保存图片
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_03_V2_300.tif"), dpi=300, bbox_inches="tight", pad_inches=0.05, pil_kwargs={"compression": "tiff_lzw"})
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_03_V2_600.tif"), dpi=600, bbox_inches="tight", pad_inches=0.05, pil_kwargs={"compression": "tiff_lzw"})
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_03_V2_600.png"), dpi=600, bbox_inches="tight", pad_inches=0.05)
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_03_V2_900.svg"), dpi=900, bbox_inches="tight", pad_inches=0.05)

plt.gcf().set_facecolor("white")
plt.show()


## FigureMS 02 -- 原位 Zn-Mn 的演化过程

In [ ]:
# 读取数据

# Operando XRD, EMD, from No pH Buffer, 1 M + 0.2 M, 30 uA

# 1) 读取电化学数据
path_filelist = list(Path.joinpath(path_data_root, r"XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Echem\B").glob(r"**\*.txt"))
echem_all = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break

    df = pd.read_csv(path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "Ece/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = pd.to_datetime(df["time/s"], format="mixed", errors="coerce")
    df["cycle number"] = pd.to_numeric(df["cycle number"], errors="coerce").astype("Int64")
    df["Voltage/V"] = df["Ewe/V"] - df["Ece/V"]
    echem_all.append(df)

echem_all = pd.concat(echem_all, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 2) 读取谱线时间戳（按当前固定格式：Wave / Date / filename）
path_xrd = Path.joinpath(path_data_root, r"XRD\Operando\EMD\2025-MSPD\Results\IS17_D\Data")
path_time_spectrum = path_xrd.joinpath(r"Time_index_spectrum.xlsx")

spectrum_time_all = pd.read_excel(path_time_spectrum)
required_cols = {"Wave", "Date", "filename"}
missing_cols = required_cols.difference(set(spectrum_time_all.columns))

spectrum_time_all["spec_time"] = pd.to_datetime(
    spectrum_time_all["Date"],
    format=r"%Y-%m-%d_%H:%M:%S",
    errors="coerce",
)

spectrum_time_all["spectrum_col"] = spectrum_time_all["filename"].astype(str).str.strip().str.replace(r"\.xye$", "", regex=True)
spectrum_time_all = spectrum_time_all.dropna(subset=["spec_time", "spectrum_col"]).sort_values("spec_time").reset_index(drop=True)

# 3) 读取 XRD 全谱数据
path_dspacing = path_xrd.joinpath(r"spectrum_all_d_spacing.csv")

spectrum_all = pd.read_csv(path_dspacing, index_col=0, header=0)
spectrum_all.index = pd.to_numeric(spectrum_all.index, errors="coerce")
spectrum_all = spectrum_all.loc[~spectrum_all.index.isna(), :]
spectrum_all.columns = [str(c).strip().replace(".xye", "") for c in spectrum_all.columns]

In [ ]:
# 过滤 17_Da_* 谱线并与时间索引对齐（后续单元直接使用）

# 1) 时间表仅保留 17_Da_fxxxxx
spectrum_time_all["spectrum_col"] = spectrum_time_all["spectrum_col"].astype(str).str.strip().str.replace(r"\.xye$", "", regex=True)
mask_da = spectrum_time_all["spectrum_col"].str.match(r"^17_Da_f\d+$", na=False)
spectrum_time_all = spectrum_time_all.loc[mask_da].copy().sort_values("spec_time").reset_index(drop=True)

# 2) 谱图矩阵仅保留 17_Da_fxxxxx 列
spec_cols_da = [c for c in spectrum_all.columns if c.startswith("17_Da_f")]
spectrum_all = spectrum_all.loc[:, spec_cols_da].copy()

# 3) 取交集并按时间顺序重排，确保 time/spectrum 完全对齐
common_cols = [c for c in spectrum_time_all["spectrum_col"].tolist() if c in spectrum_all.columns]
spectrum_time_all = spectrum_time_all[spectrum_time_all["spectrum_col"].isin(common_cols)].copy()
spectrum_time_all = spectrum_time_all.drop_duplicates(subset=["spectrum_col"], keep="first").sort_values("spec_time").reset_index(drop=True)
spectrum_all = spectrum_all.reindex(columns=spectrum_time_all["spectrum_col"].tolist())

# 4) 清洗电化学数据并建立 charge_time
echem_all.drop(echem_all.index[:6500], inplace=True)
selected_echem = echem_all[echem_all["cycle number"].isin([1, 2])]
selected_echem = selected_echem[selected_echem["Voltage/V"] >= 0.8].copy()
selected_echem = selected_echem.dropna(subset=["time/s", "Voltage/V", "<I>/mA"]).sort_values(by="time/s").reset_index(drop=True)
selected_echem["charge_time"] = (selected_echem["time/s"] - selected_echem["time/s"].iloc[0]).dt.total_seconds() / 3600

# 5) 最近邻时间匹配：谱线 -> 电化学
selected_spectrum_time = (
    pd
    .merge_asof(
        spectrum_time_all[["spectrum_col", "spec_time"]].sort_values(by="spec_time"),
        selected_echem[["time/s", "Voltage/V", "<I>/mA", "charge_time"]].sort_values(by="time/s"),
        left_on="spec_time",
        right_on="time/s",
        direction="nearest",
        tolerance=pd.Timedelta("5s"),
    )
    .dropna(subset=["spectrum_col", "spec_time", "time/s", "Voltage/V", "charge_time"], inplace=False)
    .sort_values(by="spec_time")
    .reset_index(drop=True)
)

# 6) 衍生时间与索引列（新版）
time_zero = selected_echem["time/s"].iloc[0]
selected_spectrum_time["spec_charge_time"] = (selected_spectrum_time["spec_time"] - time_zero).dt.total_seconds() / 3600
selected_spectrum_time["spec_time_h"] = selected_spectrum_time["spec_charge_time"]
selected_spectrum_time["map_dt_s"] = (selected_spectrum_time["time/s"] - selected_spectrum_time["spec_time"]).dt.total_seconds().abs()
selected_spectrum_time["range_idx"] = selected_spectrum_time["spectrum_col"].str.extract(r"(\d+)$", expand=False).astype("Int64")

# 7) 选择谱线区间并建立绘图矩阵
d_spacing_range = (0.5, 15.0)
spectrum_all = spectrum_all.loc[(spectrum_all.index >= d_spacing_range[0]) & (spectrum_all.index <= d_spacing_range[1])].copy()

selected_spectrum_time = selected_spectrum_time.dropna(subset=["range_idx"]).copy()
selected_spectrum_time = selected_spectrum_time.drop_duplicates(subset=["spectrum_col"], keep="first").sort_values(by="spec_time").reset_index(drop=True)

scan_cols_plot = selected_spectrum_time["spectrum_col"].astype(str).tolist()
selected_spectrum_plot = spectrum_all.reindex(columns=scan_cols_plot).copy()

# 统一时间显示窗口：覆盖电化学与谱线时间，并上下留 15 min
spec_time_arr = selected_spectrum_time["spec_time_h"].to_numpy(dtype=float)
pad_h = 0.25
time_min = min(float(selected_echem["charge_time"].min()), float(spec_time_arr.min()))
time_max = max(float(selected_echem["charge_time"].max()), float(spec_time_arr.max()))
y_min_plot = time_min - pad_h
y_max_plot = time_max + pad_h

In [ ]:
# 读取 EXAFS 数据

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\Echem").glob(r"**\*.txt"))
# 读取电化学数据
echem_all1 = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(path_file, sep="	", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal=".").dropna(axis=1, how="all")
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(pd.to_numeric, errors="coerce")
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem_all1.append(df)

# 合并所有电化学数据为一个二维表格
echem_all1 = pd.concat(echem_all1, axis=0, ignore_index=True).sort_values(by="time/s").reset_index(drop=True)

# 读取 reference + operando 数据 # 放电数据
path_file = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
# Mn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Mn = pd.concat([data1, data2.iloc[:, 5:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, 3, 4, *list(range(14, xas_Mn.shape[1], 1))]
xas_Mn = xas_Mn.iloc[:, charge_index]  # noqa: N816
pdf_Mn = xas_Mn.iloc[:, 0:5]  # noqa: N816
pdf_Mn.columns = [
    r"Energy",
    r"0.2M_MnSO4(aq.)",
    r"alpha_MnO2_Electrode",
    r"alpha_MnO2_Powder",
    r"FC1st",
]
opData_Mn = xas_Mn.iloc[:, [0, *list(range(5, xas_Mn.shape[1]))]]  # noqa: N816
opData_Mn.columns = list(range(0, opData_Mn.shape[1], 1))

In [ ]:
# 清洗 Mn 数据

# 获取 delta mu 比较好的归一化范围
opData_Mn = opData_Mn[opData_Mn.iloc[:, 0].between(6539, 6591)].copy()

# 电化学窗口
echem_all1["Time"] = (echem_all1["time/s"] - echem_all1["time/s"].iloc[0]).dt.total_seconds() / 3600
echem_index_low = echem_all1[echem_all1["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all1[echem_all1["cycle number"] == 1]["Ewe/V"].idxmax()

echem_seg = echem_all1.iloc[int(echem_index_low) : int(echem_index_high) + 1].copy()
echem_seg = echem_seg.dropna(subset=["time/s", "Ewe/V", "<I>/mA"]).sort_values("time/s").reset_index(drop=True)


# 2) 读取 operando Mn 谱列 NorName（与 opData_Mn 的列筛选逻辑严格一致）
def _read_nor_cols(path_nor):
    cols = {}
    with open(path_nor, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.startswith("# Column."):
                if line.startswith("#------------------------"):
                    break
                continue
            left, right = line.split(":", 1)
            cols[int(left.split(".")[1])] = right.strip().lstrip("-").strip()
    return cols


nor1 = Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_1.nor")
nor2 = Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_2.nor")
cols_1 = _read_nor_cols(nor1)
cols_2 = _read_nor_cols(nor2)
cols1_vals = [cols_1[i] for i in sorted(cols_1)]
cols2_vals = [cols_2[i] for i in sorted(cols_2)]

# 与 cell5 完全一致：先拼接第二文件去掉前 5 列，再套用相同 charge_index，再去掉参考列
all_cols = [*cols1_vals, *cols2_vals[5:]]
charge_index = [0, 1, 2, 3, 4, *list(range(14, len(all_cols), 1))]
selected_cols = [all_cols[i] for i in charge_index if i < len(all_cols)]
op_labels = [str(x).strip().lstrip("-") for x in selected_cols[5:] if re.search(r"P2b_(\d+)", str(x))]

n_spec = opData_Mn.shape[1] - 1
if len(op_labels) != n_spec:
    raise ValueError(f"TEST: op_labels={len(op_labels)} != n_spec={n_spec}")

# 3) NorName -> Time 映射
map_df = pd.read_csv(Path.joinpath(path_file, r"Overview", r"Time_Index_Spectrum.csv"))
map_df = map_df[(map_df["Folder"].astype(str).str.contains("cell2_P2b", case=False, na=False)) & (map_df["Element"].astype(str).str.upper() == "MN") & map_df["NorName"].notna()].copy()
map_df["Time"] = pd.to_datetime(map_df["Time"], errors="coerce")
map_df["NorName"] = map_df["NorName"].astype(str).str.strip().str.lstrip("-")
map_df = map_df.dropna(subset=["Time"]).sort_values("Time")
name_to_time = map_df.drop_duplicates(subset=["NorName"], keep="first").set_index("NorName")["Time"]

label_s = pd.Series(op_labels, dtype="string").str.strip().str.lstrip("-")
spec_time_dt = label_s.map(name_to_time)
missing_labels = label_s[spec_time_dt.isna()].tolist()
if missing_labels:
    raise ValueError(f"TEST: missing NorName->Time mapping for labels: {missing_labels[:6]}")

# 4) 统一时间零点并映射到电化学轨迹
time_zero = echem_seg["time/s"].iloc[0]
echem_time_mn = (echem_seg["time/s"] - time_zero).dt.total_seconds().to_numpy(dtype=float) / 3600.0
echem_voltage_mn = echem_seg["Ewe/V"].to_numpy(dtype=float)
echem_current_ua_mn = echem_seg["<I>/mA"].to_numpy(dtype=float) * 1000.0

spec_time_mn = (spec_time_dt - time_zero).dt.total_seconds().to_numpy(dtype=float) / 3600.0
spec_volt_mn = np.interp(spec_time_mn, echem_time_mn, echem_voltage_mn)
spec_curr_ua_mn = np.interp(spec_time_mn, echem_time_mn, echem_current_ua_mn)

special_idx = np.array([0, 4, 9, 12, 14, 17, 24, 30, 34, 38], dtype=int)
special_idx = special_idx[(special_idx >= 0) & (special_idx < n_spec)]

# I 图数据：相对于第一条谱线做差（在清洗阶段完成）
mn_energy = opData_Mn.iloc[:, 0].to_numpy(dtype=float)
mn_mu_mat = opData_Mn.iloc[:, 1:].to_numpy(dtype=float).T
if mn_mu_mat.shape[0] > 0:
    mn_mu_delta = mn_mu_mat - mn_mu_mat[0:1, :]
else:
    mn_mu_delta = mn_mu_mat.copy()

In [ ]:
# 读取 Zn 数据

# 读取 reference + operando 数据
path_file = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")

# Zn
data1 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_1.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
data2 = pd.read_csv(
    Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_2.nor"),
    sep=r"\s+",
    index_col=None,
    header=None,
    comment="#",
)
xas_Zn = pd.concat([data1, data2.iloc[:, 3:]], axis=1, ignore_index=True)  # noqa: N816
charge_index = [0, 1, 2, *list(range(12, xas_Zn.shape[1], 1))]
xas_Zn = xas_Zn.iloc[:, charge_index]  # noqa: N816

pdf_Zn = xas_Zn.iloc[:, 0:3]  # noqa: N816
pdf_Zn.columns = [r"Energy", r"0.5M_ZnSO4(aq.)", r"ZHS"]
opData_Zn = xas_Zn.iloc[:, [0, *list(range(3, xas_Zn.shape[1]))]]  # noqa: N816
opData_Zn.columns = list(range(0, opData_Zn.shape[1], 1))


In [ ]:
# 清洗 Zn 数据

# 获取 delta mu 比较好的归一化范围
opData_Zn = opData_Zn[opData_Zn.iloc[:, 0].between(9660, 9680)].copy()

# 电化学窗口
echem_all1["Time"] = (echem_all1["time/s"] - echem_all1["time/s"].iloc[0]).dt.total_seconds() / 3600
echem_index_low = echem_all1[echem_all1["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all1[echem_all1["cycle number"] == 1]["Ewe/V"].idxmax()

echem_seg = echem_all1.iloc[int(echem_index_low): int(echem_index_high) + 1].copy()
echem_seg = echem_seg.dropna(subset=["time/s", "Ewe/V", "<I>/mA"]).sort_values("time/s").reset_index(drop=True)

# 2) 读取 operando Zn 谱列 NorName
def _read_nor_cols(path_nor):
    cols = {}
    with open(path_nor, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.startswith("# Column."):
                if line.startswith("#------------------------"):
                    break
                continue
            left, right = line.split(":", 1)
            cols[int(left.split(".")[1])] = right.strip().lstrip("-").strip()
    return cols

nor1 = Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_1.nor")
nor2 = Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Zn_Oct2022_2.nor")
cols_1 = _read_nor_cols(nor1)
cols_2 = _read_nor_cols(nor2)
cols1_vals = [cols_1[i] for i in sorted(cols_1)]
cols2_vals = [cols_2[i] for i in sorted(cols_2)]
all_cols = [*cols1_vals, *cols2_vals]
charge_index = [0, 1, 2, *list(range(12, len(all_cols), 1))]
selected_cols = [all_cols[i] for i in charge_index if i < len(all_cols)]
op_labels = [str(x).strip().lstrip('-') for x in selected_cols if re.search(r"P2b_(\d+)", str(x))]

n_spec = opData_Zn.shape[1] - 1
if len(op_labels) != n_spec:
    raise ValueError(f"TEST: op_labels={len(op_labels)} != n_spec={n_spec}")

# 3) NorName -> Time 映射
map_df = pd.read_csv(Path.joinpath(path_file, r"Overview", r"Time_Index_Spectrum.csv"))
map_df = map_df[
    (map_df["Folder"].astype(str).str.contains("cell2_P2b", case=False, na=False))
    & (map_df["Element"].astype(str).str.upper() == "ZN")
    & map_df["NorName"].notna()
].copy()
map_df["Time"] = pd.to_datetime(map_df["Time"], errors="coerce")
map_df["NorName"] = map_df["NorName"].astype(str).str.strip().str.lstrip("-")
map_df = map_df.dropna(subset=["Time"]).sort_values("Time")
name_to_time = map_df.drop_duplicates(subset=["NorName"], keep="first").set_index("NorName")["Time"]

label_s = pd.Series(op_labels, dtype="string").str.strip().str.lstrip("-")
spec_time_dt = label_s.map(name_to_time)
missing_labels = label_s[spec_time_dt.isna()].tolist()
if missing_labels:
    raise ValueError(f"TEST: missing NorName->Time mapping for labels: {missing_labels[:6]}")

# 4) 统一时间零点并映射到电化学轨迹
time_zero = echem_seg["time/s"].iloc[0]
echem_time_zn = (echem_seg["time/s"] - time_zero).dt.total_seconds().to_numpy(dtype=float) / 3600.0
echem_voltage_zn = echem_seg["Ewe/V"].to_numpy(dtype=float)
echem_current_ua_zn = (echem_seg["<I>/mA"].to_numpy(dtype=float) * 1000.0)

spec_time_zn = (spec_time_dt - time_zero).dt.total_seconds().to_numpy(dtype=float) / 3600.0
spec_volt_zn = np.interp(spec_time_zn, echem_time_zn, echem_voltage_zn)
spec_curr_ua_zn = np.interp(spec_time_zn, echem_time_zn, echem_current_ua_zn)

special_idx = np.array([0, 4, 9, 12, 14, 17, 24, 30, 34, 38], dtype=int)
special_idx = special_idx[(special_idx >= 0) & (special_idx < n_spec)]

# J 图数据：相对于第一条谱线做差（在清洗阶段完成）
zn_energy = opData_Zn.iloc[:, 0].to_numpy(dtype=float)
zn_mu_mat = opData_Zn.iloc[:, 1:].to_numpy(dtype=float).T
if zn_mu_mat.shape[0] > 0:
    zn_mu_delta = zn_mu_mat - zn_mu_mat[0:1, :]
else:
    zn_mu_delta = zn_mu_mat.copy()

In [ ]:
# 读取 EJ 数据

echem_all = echem_all1.copy()

ej_path = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\EdgeJump")
edgejump = pd.read_excel(ej_path.joinpath(r"Cell2_Edge Jump.xlsx"), sheet_name=r"EJ_Cheng", header=0, comment="#")
edgejump_2 = pd.read_excel(ej_path.joinpath(r"Cell2_Edge Jump.xlsx"), sheet_name=r"ZSH_MnO2", header=0, comment="#")

In [ ]:
# 清洗 EJ 数据

# 电化学时间对齐
echem_all["Time"] = (echem_all["time/s"] - echem_all["time/s"].iloc[0]).dt.total_seconds() / 3600

echem_index_low = echem_all[echem_all["cycle number"] == 0]["Ewe/V"].idxmin()
echem_index_high = echem_all[echem_all["cycle number"] == 1]["Ewe/V"].idxmax()
echem_seg = echem_all.iloc[int(echem_index_low) : int(echem_index_high) + 1].copy()
echem_seg = echem_seg.dropna(subset=["time/s", "Ewe/V", "<I>/mA"]).sort_values("time/s").reset_index(drop=True)

# NorName 列名读取, 用于谱线到时间戳映射
def _read_nor_cols(path_nor):
    cols = {}
    with open(path_nor, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.startswith("# Column."):
                if line.startswith("#------------------------"):
                    break
                continue
            left, right = line.split(":", 1)
            cols[int(left.split(".")[1])] = right.strip().lstrip("-").strip()
    return cols

path_file = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2")
nor1 = Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_1.nor")
nor2 = Path.joinpath(path_file, r"Overview", r"cell2_opXAS_P2b_Mn_Oct2022_2.nor")
cols_1 = _read_nor_cols(nor1)
cols_2 = _read_nor_cols(nor2)
cols1_vals = [cols_1[i] for i in sorted(cols_1)]
cols2_vals = [cols_2[i] for i in sorted(cols_2)]

# 拼接后使用同一 charge_index 选择 operando 光谱标签
all_cols = [*cols1_vals, *cols2_vals[5:]]
charge_index = [0, 1, 2, 3, 4, *list(range(14, len(all_cols), 1))]
selected_cols = [all_cols[i] for i in charge_index if i < len(all_cols)]
op_labels = [str(x).strip().lstrip("-") for x in selected_cols[5:] if re.search(r"P2b_(\d+)", str(x))]

map_df = pd.read_csv(Path.joinpath(path_file, r"Overview", r"Time_Index_Spectrum.csv"))
map_df = map_df[(map_df["Folder"].astype(str).str.contains("cell2_P2b", case=False, na=False)) & (map_df["Element"].astype(str).str.upper() == "MN") & map_df["NorName"].notna()].copy()
map_df["Time"] = pd.to_datetime(map_df["Time"], errors="coerce")
map_df["NorName"] = map_df["NorName"].astype(str).str.strip().str.lstrip("-")
map_df = map_df.dropna(subset=["Time"]).sort_values("Time")
name_to_time = map_df.drop_duplicates(subset=["NorName"], keep="first").set_index("NorName")["Time"]

label_s = pd.Series(op_labels, dtype="string").str.strip().str.lstrip("-")
spec_time_dt = label_s.map(name_to_time)
missing_labels = label_s[spec_time_dt.isna()].tolist()
if missing_labels:
    raise ValueError(f"missing NorName->Time mapping for labels: {missing_labels[:6]}")

time_zero = echem_seg["time/s"].iloc[0]
echem_time_ej = (echem_seg["time/s"] - time_zero).dt.total_seconds().to_numpy(dtype=float) / 3600.0
edgejump_time_h = (spec_time_dt - time_zero).dt.total_seconds().to_numpy(dtype=float) / 3600.0
echem_voltage_ej = echem_seg["Ewe/V"].to_numpy(dtype=float)
echem_current_ua_ej = echem_seg["<I>/mA"].to_numpy(dtype=float) * 1000.0
spec_volt_ej = np.interp(edgejump_time_h, echem_time_ej, echem_voltage_ej)

# 特殊点索引
special_idx = np.array([0, 4, 9, 12, 14, 17, 24, 30, 34, 38], dtype=int)
special_idx = special_idx[(special_idx >= 0) & (special_idx < edgejump_time_h.shape[0])]


In [ ]:
# 清洗 EJ 数据

edgejump_mn2 = edgejump.iloc[9:, [2, 3]].copy().reset_index(drop=True)
edgejump_mn4 = edgejump.iloc[9:, [4, 5, 8, 9]].copy().reset_index(drop=True)

edgejump_zn2 = edgejump.iloc[9:, [12, 13]].copy().reset_index(drop=True)
edgejump_zsh = edgejump.iloc[9:, [14, 15, 18, 19]].copy().reset_index(drop=True)

edgejump_2.iloc[12, 0] = np.nan

In [ ]:
# 读取 MCR 数据
mcr_path = Path.joinpath(path_data_root, r"XAS\Operando\αMnO2\XAS\CLAESS_2022-10\Results\cell2\FC1st-FD2nd\Oct2022_cell2_P2b\XANES\Mn\MCR")
Mn_MCR_3 = xr.open_dataset(mcr_path.joinpath(r"P2b_Mn_MCR_3_stFixed.NETCDF4"), engine="h5netcdf")

In [ ]:
# 画图
plt.close("all")

fig = plt.figure(figsize=(7.0, 5.0))
layout = [["A", "B", "C", "DEFG"], ["H", "I", "J", "DEFG"]]
axs = fig.subplot_mosaic(
    layout,
    gridspec_kw={"width_ratios": [0.3, 1.0, 1.0, 1.0], "height_ratios": [1.0, 1.0], "wspace": 0.3, "hspace": 0.1},
)

# 图 A
ax = axs["A"]

ax.plot(selected_echem["Voltage/V"], selected_echem["charge_time"], ls="-", lw=1.0, c=colors[0], label=r"Voltage")

index_labels = [15, 25, 37, 46, 50, 61, 70, 75, 87, 99, 109, 115, 126, 137]
range_idx_int = pd.to_numeric(selected_spectrum_time["range_idx"], errors="coerce").astype("Int64")
special_mask = range_idx_int.isin(index_labels)

plot_points = selected_spectrum_time.copy()
plot_points["is_special"] = special_mask.to_numpy()
special_points = plot_points[plot_points["is_special"]].copy()

for _, row in plot_points.iterrows():
    if bool(row["is_special"]):
        ax.scatter(
            row["Voltage/V"],
            row["spec_time_h"],
            c=colors[0],
            edgecolors="face",
            marker="*",
            s=50,
            zorder=5,
        )
    else:
        ax.scatter(
            row["Voltage/V"],
            row["spec_time_h"],
            c="lightgrey",
            edgecolors="face",
            s=30,
            zorder=1,
        )

ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_xlim(0.8, 2.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.6, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.3, offset=0))

ax.set_ylabel(r"Duration Time (h)", fontsize=9)
ax.set_ylim(y_min_plot, y_max_plot)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=6, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=3, offset=0))
ax.tick_params(axis="both", which="both", labelcolor="k", bottom=True, left=True, labelbottom=True, labelleft=True, labelsize=8)

ax2 = ax.twiny()
ax2.plot(selected_echem["<I>/mA"] * 1000, selected_echem["charge_time"], ls="--", lw=1.0, c=colors[3], label=r"Current")
ax2.set_xlabel(r"$\mathrm{Current  \ (\mu A)}$", fontsize=9)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.set_ylim(y_min_plot, y_max_plot)
ax2.tick_params(axis="both", which="both", labelcolor="k", top=True, labeltop=True, labelsize=8)

ax.text(-0.7, 1.0, "A", transform=ax.transAxes, fontsize=10, fontweight="bold", va="center", ha="right")

# 图 B
spec_time_arr = selected_spectrum_time["spec_time_h"].to_numpy(dtype=float)
if selected_spectrum_plot.shape[1] != len(spec_time_arr):
    raise ValueError(f"spectrum/time length mismatch: {selected_spectrum_plot.shape[1]} vs {len(spec_time_arr)}")

d_axis = selected_spectrum_plot.index.to_numpy(dtype=float)
d_grid, t_grid = np.meshgrid(d_axis, spec_time_arr)

special_y_time = (
    selected_spectrum_time[selected_spectrum_time["range_idx"].isin(index_labels)]["spec_time_h"]
    .dropna()
    .astype(float)
    .tolist()
)

# 图 B
ax = axs["B"]
pos = ax.get_position()
ax.set_position([pos.x0, pos.y0-0.08, pos.width*1.35, pos.height*1.35])
ax.set_box_aspect(1.0)

pc_b = ax.pcolormesh(d_grid, t_grid, selected_spectrum_plot.T.to_numpy(dtype=float), cmap="coolwarm", shading="gouraud", vmin=6000, vmax=42000)
ax.set_yticks([])
ax.set_ylim(float(spec_time_arr.min()), float(spec_time_arr.max()))
ax.set_xlim(2.0, 8.0)
ax.set_xlabel(r"d Spacing ($\AA$)", fontsize=9)
ax.tick_params(axis="both", which="both", labelsize=8)

cax_b = ax.inset_axes((1.03, 0.1, 0.06, 0.8))
cb_b = fig.colorbar(pc_b, cax=cax_b)
cb_b.set_ticks([])
ax.text(1.07, 0.96, "High", transform=ax.transAxes, fontsize=8, va="top", ha="center")
ax.text(1.07, 0.02, "Low", transform=ax.transAxes, fontsize=8, va="bottom", ha="center")

ax.text(-0.05, 1.0, "B", transform=ax.transAxes, fontsize=10, fontweight="bold", va="center", ha="right")

# 图 C
base_c = axs["C"]
base_c.set_axis_off()
opxrd_ref_d_spacing = [1.414916, 2.44884, 5.967425, 8.042391, 2.71, 3.23, 5.47, 10.94]
peak_widths = [
    (0.05, 0.05),
    (0.05, 0.05),
    (0.4, 0.4),
    (0.4, 0.4),
    (0.1, 0.1),
    (0.1, 0.1),
    (0.1, 0.1),
    (0.4, 0.4),
]

# ========== 改动开始 ==========
# 固定每个子图的宽度和高度（相对于 figure 的比例）
fixed_width = 0.05      # 每个子图宽度占 figure 宽度的比例
fixed_height = 0.48      # 每个子图高度占 figure 高度的比例（可根据原图效果调整）
wspace = 0.005           # 子图之间的水平间距（比例）

n = len(opxrd_ref_d_spacing)
total_width = n * fixed_width + (n - 1) * wspace
pos_base = base_c.get_position()
# 调整 base_c 的宽度以容纳所有子图（左边界不变）
base_c.set_position([pos_base.x0, pos_base.y0, total_width, pos_base.height])

c_shift = 0.12          # 可选的额外右移量，保留原代码中的偏移
# ========== 改动结束 ==========

for i, d0 in enumerate(opxrd_ref_d_spacing):
    # ========== 改动开始：手动计算子图位置并创建 ==========
    x0 = pos_base.x0 + i * (fixed_width + wspace) + c_shift
    y0 = pos_base.y0    # 与 base_c 同高，也可微调
    axc = fig.add_axes([x0, y0, fixed_width, fixed_height])
    # ========== 改动结束 ==========
    
    temp = selected_spectrum_plot.loc[(selected_spectrum_plot.index >= d0 - peak_widths[i][0]) & 
                                      (selected_spectrum_plot.index <= d0 + peak_widths[i][1])]
    if temp.empty:
        axc.set_axis_off()
        continue
    td, tt = np.meshgrid(temp.index.to_numpy(dtype=float), spec_time_arr)
    axc.pcolormesh(td, tt, temp.T.to_numpy(dtype=float), cmap="coolwarm", shading="gouraud")
    axc.axvline(d0, ls="--", lw=0.8, color="grey", alpha=0.5)
    axc.yaxis.set_ticks([])
    axc.set_ylim(float(spec_time_arr.min()), float(spec_time_arr.max()))
    axc.set_xlim(float(temp.index.min()), float(temp.index.max()))
    axc.xaxis.set_major_locator(ticker.FixedLocator([d0]))
    axc.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
    axc.spines[["top", "right", "left"]].set_visible(False)
    axc.tick_params(axis="both", which="both", labelsize=8)
    for y_sp in special_y_time:
        axc.axhline(y=y_sp, lw=0.6, ls="--", color=colors[1], alpha=0.45)

base_c.text(0.25, 1.0, "C", transform=base_c.transAxes, fontsize=10, fontweight="bold", va="center", ha="right")

def _axis_edges_from_centers(values):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        raise ValueError("time/energy axis is empty")
    if values.size == 1:
        return np.array([values[0] - 0.5, values[0] + 0.5], dtype=float)
    edges = np.empty(values.size + 1, dtype=float)
    edges[1:-1] = 0.5 * (values[:-1] + values[1:])
    edges[0] = values[0] - 0.5 * (values[1] - values[0])
    edges[-1] = values[-1] + 0.5 * (values[-1] - values[-2])
    return edges

fgh_time_min = min(float(np.nanmin(echem_time_ej)), float(np.nanmin(spec_time_mn)), float(np.nanmin(spec_time_zn))) - 0.25
fgh_time_max = max(float(np.nanmax(echem_time_ej)), float(np.nanmax(spec_time_mn)), float(np.nanmax(spec_time_zn))) + 0.25

# 图 H
ax = axs["H"]
pos = ax.get_position()
ax.set_position([pos.x0, pos.y0-0.15, pos.width, pos.height])

ax.plot(echem_voltage_ej, echem_time_ej, c=colors[0], ls="-", lw=1.0, zorder=1)

n_spec_i = len(edgejump_time_h)
all_idx = np.arange(n_spec_i, dtype=int)
special_mask = np.zeros(n_spec_i, dtype=bool)
if special_idx.size > 0:
    special_mask[special_idx[special_idx < n_spec_i]] = True
normal_idx = all_idx[~special_mask]

if normal_idx.size > 0:
    ax.scatter(spec_volt_ej[normal_idx], edgejump_time_h[normal_idx], c="lightgrey", edgecolors="face", s=14, zorder=1)
if special_idx.size > 0:
    sp = special_idx[special_idx < n_spec_i]
    if sp.size > 0:
        ax.scatter(spec_volt_ej[sp], edgejump_time_h[sp], c=colors[0], edgecolors="face", marker="*", s=35, zorder=4)

ax.set_xlabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_xlim(0.8, 2.0)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=0.4, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=0.2, offset=0))
ax.set_ylabel("Duration time (h)", fontsize=9)
ax.set_ylim(fgh_time_min, fgh_time_max)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=3, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=0))
ax.tick_params(axis="both", which="both", labelsize=8)

ax2 = ax.twiny()
ax2.set_position([pos.x0, pos.y0-0.15, pos.width, pos.height])

ax2.plot(echem_current_ua_ej, echem_time_ej, c=colors[1], ls="--", lw=1.0, alpha=0.9)
ax2.set_xlim(-60, 60)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(base=60, offset=0))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(base=30, offset=0))
ax2.set_xlabel(r"Current ($\mu$A)", fontsize=9)
ax2.tick_params(axis="x", which="both", labelsize=8)

fgh_rect_xmin, fgh_rect_xmax = 1.25, 1.90
fgh_rect_ymin, fgh_rect_ymax = 6.5, 12.0
overlay_indices_fgh = special_idx[(special_idx >= 0) & (special_idx < n_spec_i)]
ax.text(-0.7, 1.0, "F", transform=ax.transAxes, fontsize=10, fontweight="bold", va="center", ha="right")

# I
ax = axs["I"]
pos = ax.get_position()
ax.set_position([pos.x0, pos.y0-0.23, pos.width*1.35, pos.height*1.35])
ax.set_box_aspect(1.0)

energy_mn = mn_energy
mu_mat = mn_mu_delta
n_j = min(mu_mat.shape[0], len(spec_time_mn))
if n_j > 0:
    mu_mat = mu_mat[:n_j, :]
    t_j = np.asarray(spec_time_mn[:n_j], dtype=float)
    energy_edges_mn = _axis_edges_from_centers(energy_mn)
    time_edges = _axis_edges_from_centers(t_j)
    vmin_i = -0.4
    vmax_i = 0.4
    pc_j = ax.pcolormesh(energy_edges_mn, time_edges, mu_mat, cmap="coolwarm", shading="flat", vmin=vmin_i, vmax=vmax_i, zorder=1)

    for idx in overlay_indices_fgh:
        if 0 <= idx < mu_mat.shape[0]:
            ax.axhline(y=float(edgejump_time_h[idx]), ls="--", lw=0.8, color="k", alpha=0.6, zorder=5)

    cax = ax.inset_axes((1.05, 0.10, 0.06, 0.8))
    cb = fig.colorbar(pc_j, cax=cax)
    cb.set_label(r"$\Delta\mu$(E)", fontsize=8)
    cb.set_ticks(np.linspace(vmin_i, vmax_i, 5))
    cb.ax.tick_params(labelsize=8)

ax.set_xlabel("Energy (eV)", fontsize=9)
ax.set_xlim(6540, 6590)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=10, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=5, offset=0))
ax.set_ylabel("Time Series (h)", fontsize=9)
if len(spec_time_mn) > 0:
    ax.set_ylim(fgh_time_min, fgh_time_max)
ax.set_yticks([])
ax.tick_params(labelsize=8)
ax.text(-0.08, 1.02, "G", transform=ax.transAxes, fontsize=10, fontweight="bold", va="center", ha="right")

# J
ax = axs["J"]
pos = ax.get_position()
ax.set_position([pos.x0+0.18, pos.y0-0.23, pos.width*1.35, pos.height*1.35])
ax.set_box_aspect(1.0)

energy_zn = zn_energy
mu_zn = zn_mu_delta
n_e = min(mu_zn.shape[0], len(spec_time_zn))
if n_e > 0:
    mu_zn = mu_zn[:n_e, :]
    t_e = np.asarray(spec_time_zn[:n_e], dtype=float)
    energy_edges_zn = _axis_edges_from_centers(energy_zn)
    time_edges = _axis_edges_from_centers(t_e)
    vmin_e = -0.2
    vmax_e = 0.2
    pc_e = ax.pcolormesh(energy_edges_zn, time_edges, mu_zn, cmap="coolwarm", shading="flat", vmin=vmin_e, vmax=vmax_e)

    for idx in overlay_indices_fgh:
        if 0 <= idx < mu_zn.shape[0]:
            ax.axhline(y=float(edgejump_time_h[idx]), ls="--", lw=0.8, color="k", alpha=0.6, zorder=5)

    cax_e = ax.inset_axes((1.05, 0.10, 0.06, 0.8))
    cb_e = fig.colorbar(pc_e, cax=cax_e)
    cb_e.set_label(r"$\Delta\mu$(E)", fontsize=8)
    cb_e.set_ticks(np.linspace(vmin_e, vmax_e, 5))
    cb_e.ax.tick_params(labelsize=8)

ax.set_xlabel("Energy (eV)", fontsize=9)
ax.set_xlim(9660, 9680)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2.5, offset=0))
ax.set_ylabel("Time Series (h)", fontsize=9)
if len(spec_time_zn) > 0:
    ax.set_ylim(fgh_time_min, fgh_time_max)
ax.set_yticks([])
ax.tick_params(labelsize=8)
ax.text(-0.08, 1.02, "H", transform=ax.transAxes, fontsize=10, fontweight="bold", va="center", ha="right")

# 图 DEFG
ax = axs["DEFG"]
ax.set_axis_off()

sg = ax.get_subplotspec().subgridspec(4, 1, hspace=0.28, wspace=0.2)
axd = fig.add_subplot(sg[0, 0])
axe = fig.add_subplot(sg[1, 0])
axf = fig.add_subplot(sg[2, 0])
axg = fig.add_subplot(sg[3, 0])
defg_shiftx = [0.4, 0.4, 0.4, 0.4]
defg_shifty = [-0.065, -0.11, -0.16, -0.2]
for i, axr in enumerate((axd, axe, axf, axg)):
    axr.set_box_aspect(0.5)
    posr = axr.get_position()
    axr.set_position([posr.x0 + defg_shiftx[i], posr.y0 + defg_shifty[i], posr.width*1.5, posr.height*1.5])

# D
axd.plot(echem_time_ej, echem_voltage_ej, c=colors[0], lw=0.9)
axd2 = axd.twinx()
axd2.set_position(axd.get_position())
axd2.plot(echem_time_ej, echem_current_ua_ej, c=colors[1], lw=0.9, ls="--")
n_spec_d = min(len(edgejump_time_h), len(spec_volt_ej))
if n_spec_d > 0:
    spec_time_d = np.asarray(edgejump_time_h[:n_spec_d], dtype=float)
    spec_volt_d = np.asarray(spec_volt_ej[:n_spec_d], dtype=float)
    all_idx_d = np.arange(n_spec_d, dtype=int)
    special_mask_d = np.zeros(n_spec_d, dtype=bool)
    if special_idx.size > 0:
        sp_d = special_idx[special_idx < n_spec_d]
        if sp_d.size > 0:
            special_mask_d[sp_d] = True
    normal_idx_d = all_idx_d[~special_mask_d]
    if normal_idx_d.size > 0:
        axd.scatter(spec_time_d[normal_idx_d], spec_volt_d[normal_idx_d], c="lightgrey", edgecolors="face", s=14, zorder=3)
    if special_idx.size > 0:
        sp_d = special_idx[special_idx < n_spec_d]
        if sp_d.size > 0:
            axd.scatter(spec_time_d[sp_d], spec_volt_d[sp_d], c=colors[0], edgecolors="w", marker="*", s=36, linewidths=0.4, zorder=6)
axd.set_xlabel("Charge and Discharge", fontsize=9)
axd.set_xlim(-0.3, 19)
axd.set_xticks([])
# axd.xaxis.set_major_locator(ticker.MultipleLocator(base=2))
# axd.xaxis.set_minor_locator(ticker.MultipleLocator(base=1))
axd.tick_params(axis="both", labelsize=8, colors=colors[0])
axd2.tick_params(axis="y", labelsize=8, colors=colors[1])
axd.set_ylabel("Voltage (V)", fontsize=9, color=colors[0])
axd2.set_ylabel(r"Current ($\mu$A)", fontsize=9, color=colors[1])
axd.tick_params(axis="x", labelbottom=False)
# axd.text(0.02, 0.05, "C/10", transform=axd.transAxes, fontsize=8, color="0.25")
for spine in axd.spines.values():
    spine.set_linestyle("--")
    spine.set_color("#2b9fe8")
    spine.set_linewidth(0.8)
axd.text(-0.18, 1.0, "D", transform=axd.transAxes, fontsize=10, fontweight="bold", va="center", ha="right")

# E
if edgejump_time_h.shape[0] < edgejump_mn2.shape[0]:
    raise ValueError(f"edgejump_time_h({edgejump_time_h.shape[0]}) < edgejump_mn2({edgejump_mn2.shape[0]})")

timea = edgejump_time_h[: edgejump_mn2.shape[0]]
special_idx_b = special_idx[special_idx < timea.shape[0]]
axe.errorbar(timea, edgejump_mn2.iloc[:, 0], edgejump_mn2.iloc[:, 1], c=colors[0], ls="-", lw=1.0, marker="o", ms=3, label=r"$\mathrm{Liquid \ Mn^{2+}}$")
if special_idx_b.size > 0:
    axe.scatter(timea[special_idx_b], edgejump_mn2.iloc[special_idx_b, 0], c="k", marker="*", s=28, edgecolors="w", linewidths=0.4, zorder=7)

axe2 = axe.twinx()
axe2.set_position(axe.get_position())
axe2.errorbar(timea, edgejump_zn2.iloc[:, 0], edgejump_zn2.iloc[:, 1], c=colors[1], ls="--", lw=1.0, marker="s", ms=3, label=r"$\mathrm{Liquid \ Zn^{2+}}$")
if special_idx_b.size > 0:
    axe2.scatter(timea[special_idx_b], edgejump_zn2.iloc[special_idx_b, 0], c="k", marker="*", s=28, edgecolors="w", linewidths=0.4, zorder=7)

axe.tick_params(axis="both", which="both", labelsize=8, colors=colors[0])
axe2.tick_params(axis="y", which="both", labelsize=8, colors=colors[1])
axe.set_ylabel("Mn EdgeJump", fontsize=9, color=colors[0])
axe2.set_ylabel("Zn EdgeJump", fontsize=9, color=colors[1])
axe.set_xlabel("Charge and Discharge", fontsize=9)
axe.set_xlim(-0.3, 19)
axe.set_xticks([])
# axe.xaxis.set_major_locator(ticker.MultipleLocator(base=2))
# axe.xaxis.set_minor_locator(ticker.MultipleLocator(base=1))
axe.tick_params(axis="x", labelbottom=False)
axe.legend(loc="upper right", bbox_to_anchor=(0.9, 0.4), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
axe2.legend(loc="upper right", bbox_to_anchor=(0.9, 0.25), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
axe.text(-0.18, 1.0, "E", transform=axe.transAxes, fontsize=10, fontweight="bold", va="center", ha="right")

# F
if edgejump_time_h.shape[0] < edgejump_2.shape[0]:
    raise ValueError(f"edgejump_time_h({edgejump_time_h.shape[0]}) < edgejump_2({edgejump_2.shape[0]})")

timef = edgejump_time_h[: edgejump_2.shape[0]]
mn_solid = edgejump_2.iloc[:, 0].to_numpy(dtype=float)
zn_solid = edgejump_2.iloc[:, 1].to_numpy(dtype=float)
special_idx_f = special_idx[special_idx < timef.shape[0]]

axf.plot(timef, mn_solid, c=colors[0], ls="-", lw=1.0, marker="o", ms=3, label=r"Solid Mn")
if special_idx_f.size > 0:
    axf.scatter(timef[special_idx_f], mn_solid[special_idx_f], c="k", marker="*", s=28, edgecolors="w", linewidths=0.4, zorder=7)

axf2 = axf.twinx()
axf2.set_position(axf.get_position())
axf2.plot(timef, zn_solid, c=colors[1], ls="--", lw=1.0, marker="s", ms=3, label=r"Solid Zn")
if special_idx_f.size > 0:
    axf2.scatter(timef[special_idx_f], zn_solid[special_idx_f], c="k", marker="*", s=28, edgecolors="w", linewidths=0.4, zorder=7)

axf.tick_params(axis="both", which="both", labelsize=8, colors=colors[0])
axf2.tick_params(axis="y", which="both", labelsize=8, colors=colors[1])
axf.set_ylabel(r"Mn (solid, $\mu$mol)", fontsize=9, color=colors[0])
axf2.set_ylabel(r"Zn (solid, $\mu$mol)", fontsize=9, color=colors[1])
axf.set_xlabel("Charge and Discharge", fontsize=9)
axf.set_ylim(-0.5, 12)
axf2.set_ylim(-0.5, 15)
axf.set_xticks([])
axf.yaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=0))
axf.yaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=0))
axf2.yaxis.set_major_locator(ticker.MultipleLocator(base=3, offset=0))
axf2.yaxis.set_minor_locator(ticker.MultipleLocator(base=1.5, offset=0))
axf.legend(loc="upper left", bbox_to_anchor=(0.65, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
axf2.legend(loc="upper left", bbox_to_anchor=(0.65, 0.88), ncols=1, frameon=False, labelcolor="linecolor", fontsize=8)
axf.text(-0.18, 1.0, "I", transform=axf.transAxes, fontsize=10, fontweight="bold", va="center", ha="right")

# G
if "Mn_MCR_3" in globals() and "MCR_C" in Mn_MCR_3:
    xvals = np.asarray(Mn_MCR_3["spectrum_number"].values, dtype=float)
    comp = np.asarray(Mn_MCR_3["MCR_C"].values, dtype=float)
    n_comp = min(comp.shape[1], 3)
    labels = [r"$\alpha$-MnO$_2$", r"0.2 M Mn$^{2+}$", "FC#D"]
    colors_f = [colors[0], colors[1], colors[3]]
    for i in range(n_comp):
        axg.plot(xvals, comp[:, i], marker="o", ms=2.8, lw=1.0, c=colors_f[i], label=labels[i])

    special_idx_g = special_idx[special_idx < comp.shape[0]]
    if special_idx_g.size > 0:
        for i in range(n_comp):
            axg.scatter(xvals[special_idx_g], comp[special_idx_g, i], c="k", marker="*", s=26, edgecolors="w", linewidths=0.35, zorder=7)

    axg.set_xlabel("Spectrum Number", fontsize=9)
    axg.set_xlabel("Charge and Discharge", fontsize=9)
    axg.set_xticks([])
    axg.set_ylabel("Mn Molar Fraction (arb. u.)", fontsize=9)
    axg.set_ylim(-0.05, 0.8)
    axg.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0))
    axg.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0))
    axg.tick_params(axis="both", which="both", labelsize=8)
    axg.legend(loc="upper right", bbox_to_anchor=(0.95, 1.02), ncols=3, columnspacing=0.5, frameon=False, labelcolor="linecolor", fontsize=8)

axg.text(-0.18, 1.0, "J", transform=axg.transAxes, fontsize=10, fontweight="bold", va="center", ha="right")

# 保存图片
# plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_02_V1_300.tif"), dpi=300, bbox_inches="tight", pad_inches=0.05, pil_kwargs={"compression": "tiff_lzw"})
# plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_02_V1_600.tif"), dpi=600, bbox_inches="tight", pad_inches=0.05, pil_kwargs={"compression": "tiff_lzw"})
plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_02_V1_600.png"), dpi=600, bbox_inches="tight", pad_inches=0.05)
# plt.savefig(Path.joinpath(path_out, r"PaperDos_FigureMS_02_V1_900.svg"), dpi=900, bbox_inches="tight", pad_inches=0.05)

plt.gcf().set_facecolor("white")
plt.show()


## FigureMS 01 -- Limited electrolyte governs multi-stage features, Zn specie + MnO2 -V1

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar


def add_sizebar(ax, size, bardata, color, barunits=None):
    if isinstance(bardata, float):
        if not isinstance(barunits, str):
            raise ValueError("barunits must be provided if bardata is a float.")
        asb = AnchoredSizeBar(
            ax.transData,
            size / bardata,
            "{} {}".format(size, barunits),
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 8},
        )
        ax.add_artist(asb)
    else:
        if len(bardata.axes_manager.navigation_shape) == 2:
            barsize = bardata.axes_manager.navigation_axes[0].scale
            unit = bardata.axes_manager.navigation_axes[0].units
        elif len(bardata.axes_manager.signal_shape) == 2:
            barsize = bardata.axes_manager.signal_axes[0].scale
            unit = bardata.axes_manager.signal_axes[0].units
        asb = AnchoredSizeBar(
            ax.transData,
            size / barsize,  # type: ignore
            "{} {}".format(size, unit),  # type: ignore
            loc="lower left",
            pad=0.1,
            borderpad=0.5,
            sep=1,
            frameon=False,
            color=color,
            label_top=True,
            fontproperties={"size": 8},
        )
        ax.add_artist(asb)
    return asb


def transparent_single_color_cmap(color0, color1):
    """Return a single color matplotlib cmap with the transparency increasing
    linearly from 0 to 1."""
    if color0 is None:
        return LinearSegmentedColormap.from_list("", [to_rgba(color1, 0), to_rgba(color1, 1)])
    return LinearSegmentedColormap.from_list("", [to_rgba(color0, 1), to_rgba(color1, 1)])


eds_colors = {
    "C": "#FF0000",
    "K": "#FF8000",
    "Mn": "#FFFF00",
    "Zn": "#1f3cef",
    "S": "#00ffff",
    "O": "#00ff00",
}

In [ ]:
# αMnO2 in Swagelok Cells, 1M ZnSO4 + 0.2M MnSO4
# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r'Echem\αMnO2\GCD\1M ZnSO4 + 0.2M MnSO4').glob(r'LC*.xlsx'))
echema = []
# 读取电化学数据
for path in path_filelist[0:1]:
   df = pd.read_excel(path, sheet_name=2, engine='openpyxl', comment='#', header=0, index_col=None).dropna(axis=1, how='all')
   # 转换数据格式
   df[[r'Voltage/V',  r'Current/uA',  r'Capacity/uAh',  r'SpeCap/mAh/g']] = df[[r'Voltage/V',  r'Current/uA',  r'Capacity/uAh',  r'SpeCap/mAh/g']].apply(pd.to_numeric, errors='coerce')
   df['TestTime'] = df['TestTime'].apply(pd.to_timedelta, errors='coerce')
   df['Cycle'] = df['Cycle'].astype(np.int16)
   echema.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echema)):
    echema[i] = echema[i][echema[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echema[i] = echema[i][echema[i].iloc[:, 2]>=0]

# EMD in Swagelok Cells, 1M ZnSO4 + 0.2M MnSO4
# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r'Echem\EMD\GCD\1M ZnSO4 + 0.2M MnSO4\Results\1.60V-2mAh_cm-2-1MZnAC2+02M-pH5.0_1M+0.2M').glob(r'**/LC*.xlsx'))
echemb = []
path_filelist = path_filelist[0:1]
# 读取电化学数据
for path in path_filelist:
    if r"2V" not in path.stem:
        df = pd.read_excel(path, sheet_name=2, engine='openpyxl', comment='#', header=0, index_col=None).dropna(axis=1, how='all')
        # 转换数据格式
        df[[r'Voltage/V',  r'Current/uA',  r'Capacity/uAh',  r'SpeCap/mAh/g']] = df[[r'Voltage/V',  r'Current/uA',  r'Capacity/uAh',  r'SpeCap/mAh/g']].apply(pd.to_numeric, errors='coerce')
        df['TestTime'] = df['TestTime'].apply(pd.to_timedelta, errors='coerce')
        df['Cycle'] = df['Cycle'].astype(np.int16)
        echemb.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemb)):
    echemb[i] = echemb[i][echemb[i].iloc[:, 0].isin([0, 1, 2])]
    echemb[i] = echemb[i][echemb[i].iloc[:, 2]>=0]


In [ ]:
# 清洗数据
echema = pd.concat(
    [df[['Cycle','Voltage/V','Capacity/uAh']] for df in echema],
    ignore_index=True,
).dropna(inplace=False)

echemb = pd.concat(
    [df[['Cycle','Voltage/V','Capacity/uAh']] for df in echemb],
    ignore_index=True,
).dropna(inplace=False)


# 需要一圈半的数据
cycle_index = echema['Cycle'].unique()
voltage_min_index = [echema[echema['Cycle'] == i].idxmin()[r'Voltage/V'] for i in cycle_index]
voltage_max_index = [echema[echema['Cycle'] == i].idxmax()[r'Voltage/V'] for i in cycle_index]

echema.loc[voltage_min_index[0]:voltage_max_index[0], r'Capacity/uAh'] += echema.loc[voltage_min_index[0]-1, r'Capacity/uAh']
echema.loc[voltage_max_index[0]+1:voltage_min_index[1]+1, r'Capacity/uAh'] += echema.loc[voltage_max_index[0], r'Capacity/uAh']

echema = echema.iloc[:voltage_min_index[1], :]

echema[r'Capacity/uAh'] = echema[r'Capacity/uAh']*100 / echema[r'Capacity/uAh'].max()

# 需要一圈半的数据
cycle_index = echemb['Cycle'].unique()
voltage_min_index = [echemb[echemb['Cycle'] == i].idxmin()[r'Voltage/V'] for i in cycle_index]
voltage_max_index = [echemb[echemb['Cycle'] == i].idxmax()[r'Voltage/V'] for i in cycle_index]

echemb.loc[voltage_min_index[0]+1:voltage_max_index[0], r'Capacity/uAh'] += echemb.loc[voltage_min_index[0]-1, r'Capacity/uAh']
echemb.loc[voltage_max_index[0]+1:voltage_min_index[1]+1, r'Capacity/uAh'] += echemb.loc[voltage_max_index[0], r'Capacity/uAh']

echemb = echemb.iloc[:voltage_min_index[1], :]

echemb[r'Capacity/uAh'] = echemb[r'Capacity/uAh']*100 / echemb[r'Capacity/uAh'].max()

In [ ]:
# 1st1.80V - 1M ZnSO4 + 0.2M MnSO4 - ZHS - 有机测试

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\SpecialTest\OrganicEchem\1st1.80V\ZHS\1M ZnSO4 + 0.2M MnSO4\Results\15#-03-2023-2").glob(r"LC*.xlsx"))
echemc = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemc.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemc)):
    if i == 2:
        echemc[i] = echemc[i][echemc[i].iloc[:, 0].isin([0, 1, 3])]
    else:
        echemc[i] = echemc[i][echemc[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemc[i] = echemc[i][echemc[i].iloc[:, 2] >= 0]

# 调整电化学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
indices = [0, 1, 2]  # 指定新顺序
echemc = [echemc[i] for i in indices]


# 2nd0.9V - 1M ZnSO4 + 0.2M MnSO4 - ZHS - 有机测试

# 电化学上的时间
path_filelist = list(Path.joinpath(path_data_root, r"Echem\αMnO2\SpecialTest\OrganicEchem\2nd0.9V\ZHS\1M ZnSO4 + 0.2M MnSO4\Results\10#-03-2023-B").glob(r"LC*.xlsx"))
echemd = []
# 读取电化学数据
for path in path_filelist:
    df = pd.read_excel(path, sheet_name=2, engine="openpyxl", comment="#", header=0, index_col=None).dropna(axis=1, how="all")
    # 转换数据格式
    df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]] = df[[r"Voltage/V", r"Current/uA", r"Capacity/uAh", r"SpeCap/mAh/g"]].apply(pd.to_numeric, errors="coerce")
    df["TestTime"] = df["TestTime"].apply(pd.to_timedelta, errors="coerce")
    df["Cycle"] = df["Cycle"].astype(np.int16)
    echemd.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echemd)):
    echemd[i] = echemd[i][echemd[i].iloc[:, 0].isin([0, 1, 2, 3])]
    echemd[i] = echemd[i][echemd[i].iloc[:, 2] >= 0]

# 调整电化学数据的顺序，满足，首先制备对应的样品，然后再有机电解液，然后再含水的有机电解液中的反应
indices = [0, 1, 2]  # 指定新顺序
echemd = [echemd[i] for i in indices]

In [ ]:
path_file = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\1st1.80V\1M ZnSO4 + 0.2M MnSO4\2023-EMCA\EDS\Results\0021 - 1500 Pristine MnO2 HAADF 58000 x\Sum")
eds_fc = hs.load(path_file.joinpath(r"0021 - 1500 Pristine MnO2 HAADF 58000 x.emd"))
for file in eds_fc:
    if len(file.axes_manager.shape) >= 2:
        if len(file.axes_manager.navigation_shape) == 2:
            if file.axes_manager.navigation_axes[0].units == r"µm":
                file.axes_manager.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
        elif len(file.axes_manager.signal_shape) == 2 and file.axes_manager.signal_axes[0].units == r"µm":
            file.axes_manager.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
    if len(file.axes_manager.shape) == 3:
        if len(file.axes_manager.navigation_shape) == 2:
            for axis in file.axes_manager.navigation_axes:
                axis.offset = 0
        elif len(file.axes_manager.signal_shape) == 2:
            for axis in file.axes_manager.signal_axes:
                axis.offset = 0
path_file_fd = Path.joinpath(path_data_root, r"TEM\ExSitu\αMnO2\Charge\αMnO2 + PVDF + SP\2nd0.9V\1M ZnSO4 + 0.2M MnSO4\2025-EMCA-B2\EDS\Results\0028-HAADF_67000_x_EDS_SI\Sum")
eds_fd = hs.load(path_file_fd.joinpath(r"0028-HAADF_67000_x_EDS_SI.emd"))
for file in eds_fd:
    if len(file.axes_manager.shape) >= 2:
        if len(file.axes_manager.navigation_shape) == 2:
            if file.axes_manager.navigation_axes[0].units == r"µm":
                file.axes_manager.convert_units(axes="navigation", units="nm", same_units=True, factor=1000)
        elif len(file.axes_manager.signal_shape) == 2 and file.axes_manager.signal_axes[0].units == r"µm":
            file.axes_manager.convert_units(axes="signal", units="nm", same_units=True, factor=1000)
    if len(file.axes_manager.shape) == 3:
        if len(file.axes_manager.navigation_shape) == 2:
            for axis in file.axes_manager.navigation_axes:
                axis.offset = 0
        elif len(file.axes_manager.signal_shape) == 2:
            for axis in file.axes_manager.signal_axes:
                axis.offset = 0

In [ ]:
plt.close("all")
# 画图
fig = plt.figure(figsize=(7.0, 5.0), dpi=300)
layout = [['A', 'A', 'B'], ['C', 'D', 'E']]
axs = fig.subplot_mosaic(
    layout,
    gridspec_kw={"width_ratios": [1.0, 1.0, 1.0], "height_ratios": [1.0, 1.0], "wspace": 0.0, "hspace": 0.0},
)
# 图 A
ax = axs["A"]
ax.set_box_aspect(0.3)

ax.plot(echema[r'Capacity/uAh'], echema[r'Voltage/V'], ls='-', lw=0.8, c=colors[0], label=r"$\alpha$-MnO$_2$", zorder=0)
line=ax.plot(echemb[r'Capacity/uAh'], echemb[r'Voltage/V'], ls='--', lw=0.8, c=colors[4], label=r"EMD", alpha=0.3, zorder=0)[0]
# 阶段标签
labelcolor = [colors[3], colors[1]]
ax.text(33, 0.94, r"FD#A", ha="left", va="top", transform=ax.transData, fontsize=8, c=labelcolor[0], alpha=0.5)
ax.plot(31.5, 0.92, marker="*", markersize=6, mfc=labelcolor[0], mec=labelcolor[0], alpha=0.5)
ax.text(50.5, 1.48, r"HC#B", ha="left", va="top", transform=ax.transData, fontsize=8, c=labelcolor[0], alpha=0.5)
ax.plot(50, 1.52, marker="*", markersize=6, mfc=labelcolor[0], mec=labelcolor[0], alpha=0.5)
ax.text(63, 1.59, r"HC#C", ha="left", va="top", transform=ax.transData, fontsize=8, c=labelcolor[0], alpha=0.5)
ax.plot(62.5, 1.63, marker="*", markersize=6, mfc=labelcolor[0], mec=labelcolor[0], alpha=0.5)
ax.text(71.5, 1.84, r"FC#D", ha="left", va="top", transform=ax.transData, fontsize=8, c=labelcolor[0], alpha=0.5)
ax.plot(69, 1.80, marker="*", markersize=6, mfc=labelcolor[0], mec=labelcolor[0], alpha=0.5)
ax.text(80, 1.25, r"HD#E", ha="left", va="top", transform=ax.transData, fontsize=8, c=labelcolor[0], alpha=0.5)
ax.plot(82, 1.30, marker="*", markersize=6, mfc=labelcolor[0], mec=labelcolor[0], alpha=0.5)
ax.text(93, 0.85, r"FD#F", ha="left", va="top", transform=ax.transData, fontsize=8, c=labelcolor[0], alpha=0.5)
ax.plot(100, 0.9, marker="*", markersize=6, mfc=labelcolor[0], mec=labelcolor[0], alpha=0.5)
ax.text(13, 1.32, r"D0", ha="left", va="top", transform=ax.transData, fontsize=8, c=labelcolor[1])
ax.text(38, 1.58, r"C1", ha="left", va="top", transform=ax.transData, fontsize=8, c=labelcolor[1])
ax.text(54, 1.67, r"C2", ha="left", va="top", transform=ax.transData, fontsize=8, c=labelcolor[1])
ax.text(62, 1.80, r"C3", ha="left", va="top", transform=ax.transData, fontsize=8, c=labelcolor[1])
ax.text(71, 1.70, r"D3", ha="left", va="top", transform=ax.transData, fontsize=8, c=labelcolor[1])
ax.text(75, 1.50, r"D2", ha="left", va="top", transform=ax.transData, fontsize=8, c=labelcolor[1])
ax.text(87, 1.43, r"D1", ha="left", va="top", transform=ax.transData, fontsize=8, c=labelcolor[1])
ax.text(97, 1.30, r"D0", ha="left", va="top", transform=ax.transData, fontsize=8, c=labelcolor[1])
# 设置刻度线等格式
ax.set_xlabel(r"Discharge and Charge", fontsize=9)
# ax.set_xlim(0, 105)
# ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
# ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))
ax.set_xticks([])
ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_ylim(0.7, 1.9)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0.1))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1,  offset=0.1))
ax.tick_params(axis='both', direction='out', labelsize=8)
leg = ax.legend(loc='upper left', bbox_to_anchor=(0.03, 1.0), ncols=1, frameon=False, labelcolor='linecolor', fontsize=8)
leg.get_texts()[1].set_alpha(0.5)
ax.text(0.03, 0.2, r'$\mathrm{\textasciitilde \ 60 \ \mu L}$' + '\n' + r'$\mathrm{1 \ M \ ZnSO_4 + 0.2 \ M \ MnSO_4}$', ha='left', va='top', transform=ax.transAxes, fontsize=8, c='k')
ax.text(-0.11, 1.0, r"A", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 B
ax = axs["B"]
pos = ax.get_position()
ax.set_position([pos.x0 + 0.1, pos.y0+0.0, pos.width*0.9, pos.height*0.9])
ax.set_box_aspect(0.8)

labelcolor = [colors[3], colors[1], colors[0]]
linelstyle = ["-", "-", "--"]
labels = [
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2M \ MnSO_4}$", None, None, None],
    [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO}$", None, None, None],
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2M \ MnSO_4}$", None, None, None],
]

for i, data in enumerate(echemc):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls=linelstyle[i], lw=0.8, c=labelcolor[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls=linelstyle[i], lw=0.8, c=labelcolor[i], label=None, zorder=0)
ax.plot(320, 1.80, marker="*", markersize=10, mfc=labelcolor[1], mec=labelcolor[1], alpha=0.5)
# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=9)
ax.set_xlim(-0.1, 450)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=150, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=75, offset=0))
ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_ylim(0.8, 2.3)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15,  offset=0.2))
ax.tick_params(axis="both", direction="out", labelsize=8)
ax.legend(loc="upper left", bbox_to_anchor=(0.02, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=7)
ax.text(0.03, 0.07, r"No Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=8, c="k")
ax.text(-0.25, 1.0, r"B", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 C
ax = axs["C"]
pos = ax.get_position()
ax.set_position([pos.x0-0.1, pos.y0+0.05, pos.width*0.95, pos.height*0.95])
c_rect = ax.get_position().bounds
ax.remove()

c_grid = ImageGrid(
    fig,
    c_rect,
    nrows_ncols=(2, 2),
    axes_pad=0.05,
    share_all=True,
    label_mode="L",
)
img_axes = [
    (c_grid[0], eds_fc[7].data, "gray", r"HAADF"),
    (c_grid[1], eds_fc[9].data, transparent_single_color_cmap(r"k", eds_colors["K"]), r"K $\mathit{K \alpha}$"),
    (c_grid[2], eds_fc[6].data, transparent_single_color_cmap(r"k", eds_colors["Zn"]), r"Zn $\mathit{K \alpha}$"),
    (c_grid[3], eds_fc[4].data, transparent_single_color_cmap(r"k", eds_colors["Mn"]), r"Mn $\mathit{K \alpha}$"),
]
for ax, img, cmap, label in img_axes:
    ax.imshow(img, cmap=cmap, alpha=1.0, interpolation="bilinear", aspect=1)
    ax.set_axis_off()
    ax.text(0.02, 0.98, label, transform=ax.transAxes, fontsize=8, va="top", ha="left", fontfamily="Arial", color="w")
sizebar = add_sizebar(c_grid[0], 100, eds_fc[7], "w")
sizebar.set_bbox_to_anchor(Bbox.from_bounds(-0.05, -0.01, 0, 0), transform=c_grid[0].transAxes)
c_grid[0].text(0.33, 1.05, r"FC#D", transform=c_grid[0].transAxes, fontsize=8, va="center", ha="right", fontfamily="Arial", fontweight="bold")
c_grid[0].text(-0.08, 1.0, r"C", transform=c_grid[0].transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 D
ax = axs["D"]
pos = ax.get_position()
ax.set_position([pos.x0-0.15, pos.y0-0.072, pos.width*1.44, pos.height*1.44])
d_rect = ax.get_position().bounds
ax.remove()

d_grid = ImageGrid(
    fig,
    d_rect,
    nrows_ncols=(2, 2),
    axes_pad=0.05,
    share_all=True,
    label_mode="L",
)

fd_haadf = eds_fd[6].data[10:60, :].astype(float)
fd_haadf_p1, fd_haadf_p98 = np.percentile(fd_haadf, [1, 98])
fd_haadf = np.clip(fd_haadf, fd_haadf_p1, fd_haadf_p98)
fd_haadf = (fd_haadf - fd_haadf_p1) / (fd_haadf_p98 - fd_haadf_p1 + 1e-12)
fd_haadf = np.power(fd_haadf, 3.2)

fd_img_axes = [
    (d_grid[0], fd_haadf, "gray", r"HAADF"),
    (d_grid[1], eds_fd[7].data[10:60, :], transparent_single_color_cmap(r"k", eds_colors["K"]), r"K $\mathit{K \alpha}$"),
    (d_grid[2], eds_fd[8].data[10:60, :], transparent_single_color_cmap(r"k", eds_colors["Zn"]), r"Zn $\mathit{K \alpha}$"),
    (d_grid[3], eds_fd[5].data[10:60, :], transparent_single_color_cmap(r"k", eds_colors["Mn"]), r"Mn $\mathit{K \alpha}$"),
]
for ax, img, cmap, label in fd_img_axes:
    ax.imshow(img, cmap=cmap, alpha=1.0, interpolation="bilinear", aspect=1)
    ax.set_axis_off()
    ax.text(0.02, 0.98, label, transform=ax.transAxes, fontsize=8, va="top", ha="left", fontfamily="Arial", color="w")
sizebar = add_sizebar(d_grid[0], 100, eds_fd[6], "k")
sizebar.set_bbox_to_anchor(Bbox.from_bounds(0.0, -0.02, 0, 0), transform=d_grid[0].transAxes)
d_grid[0].text(0.16, 1.05, r"FD#F", transform=d_grid[0].transAxes, fontsize=8, va="center", ha="right", fontfamily="Arial", fontweight="bold")
# d_grid[0].text(-0.025, 1.0, r"D", transform=d_grid[0].transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 图 E
ax = axs["E"]
pos = ax.get_position()
ax.set_position([pos.x0 + 0.1, pos.y0+0.02, pos.width*0.9, pos.height*0.9])
ax.set_box_aspect(0.8)

labelcolor = [colors[3], colors[1], colors[0]]
linelstyle = ["-", "-", "--"]
labels = [
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2M \ MnSO_4}$", None, None, None],
    [r"$\mathrm{1 \ M \ Zn(OTf)_2 \ in \ DMSO}$", None, None, None],
    [r"$\mathrm{1 \ M \ ZnSO_4 + 0.2M \ MnSO_4}$", None, None, None],
]

for i, data in enumerate(echemd):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx]
        # 找到电压最小值的索引
        idx_min = temp["Voltage/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(temp.loc[: idx_min - 1, "SpeCap/mAh/g"], temp.loc[: idx_min - 1, "Voltage/V"], ls=linelstyle[i], lw=0.8, c=labelcolor[i], label=labels[i][j], zorder=0)
        ax.plot(temp.loc[idx_min + 1 :, "SpeCap/mAh/g"], temp.loc[idx_min + 1 :, "Voltage/V"], ls=linelstyle[i], lw=0.8, c=labelcolor[i], label=None, zorder=0)
ax.plot(328, 0.93, marker="*", markersize=10, mfc=labelcolor[1], mec=labelcolor[1], alpha=0.5)
# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO_2})}$", fontsize=9)
ax.set_xlim(-1, 500)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=100, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=50, offset=0))
ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=9)
ax.set_ylim(0.8, 2.3)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.3, offset=0.2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.15,  offset=0.2))
ax.tick_params(axis="both", direction="out", labelsize=8)
ax.legend(loc="upper left", bbox_to_anchor=(0.02, 1.0), ncols=1, frameon=False, labelcolor="linecolor", fontsize=7)
ax.text(0.03, 0.07, r"No Acid-treated", ha="left", va="top", transform=ax.transAxes, fontsize=8, c="k")
ax.text(-0.25, 1.0, r"D", transform=ax.transAxes, fontsize=10, va="center", ha="right", fontfamily="Arial", fontweight="bold")

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigrueMS_01_V2_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigrueMS_01_V2_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigrueMS_01_V2_600.png"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
)
plt.savefig(
    Path.joinpath(path_out, r"PaperDos_FigrueMS_01_V2_900.svg"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.gcf().set_facecolor("white")
plt.show()